# Is gas surface density a better tracer of the quenching *mode* than the quenching *time*?

**Scope.** A focused, hypothesis-driven re-run of the residual-dust pipeline across **multiple
selection redshifts** (z = 0.0, 0.4, 0.7, 0.9, 1.0, 4.0). For galaxies selected at each epoch
(realistically populated in stars *and* gas) we track histories back to formation, define the
critical points (formation, sSFR peak, AGN activation, SFT, QT, post-quench, gas_min), check the
histories and halo tracking are well-behaved, and split by **AGN–ISM coupling** (no-AGN / weak /
strong).

**Question.** Quenching time (τ_q; fast/slow) is a degenerate tracer of *how* a galaxy quenched.
Is the directly-measured **gas surface density** a better tracer of the coupling strength and of
the dust outcome at gas_min?

**End goal.** Do galaxies that are dustier at gas_min have a characteristic quenching mode
(fast/slow, weak/strong)? Is the post-gas_min sSFR rebound the norm for dusty galaxies, and does it
track τ_q or coupling? Operationalised in the Σ_H2–compactness(/age) plane (Part 6h).

**This notebook is self-contained and anchor-parameterised; it reuses the proven machinery of
`residual_dust_quenching.ipynb` (it does not modify it).** Heavy products are gated behind
`BUILD_*` flags and built on the cluster; with the gates off the notebook only loads + analyses.
All statistics use `scipy.stats` (the cluster `pd39` kernel has no statsmodels).

# Part 0 — Setup & configuration

In [ ]:
import os
import gc
import json
import numpy as np
import h5py
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy import units as u
from astropy import constants as const
from astropy.cosmology import Planck15 as COSMO   # matches the repo's quenching batch

# scipy.stats only — NO statsmodels on the cluster kernel (p-values would silently be NaN)
from scipy.stats import spearmanr, pearsonr, ks_2samp, fisher_exact, mannwhitneyu, gaussian_kde

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen
from simbanator.analysis.quenching import find_quenching_times

# ── simulation / tracking config (identical to residual_dust_quenching.ipynb §0) ──
SIM_NAME    = "cis100"
BOX         = 100               # Mpc/h (sanity only)
START_SNAP  = 150               # z~0 anchor for the z=0 tracks
END_SNAP    = 44                # z~4.8

MASS_FLOOR      = 9.5
MASS_BIN_EDGES  = [9.5, 10.5, 11.0, np.inf]
MASS_BIN_LABELS = ["9.5-10.5", "10.5-11", ">=11"]
NBINS           = len(MASS_BIN_LABELS)
HMASS_BIN_EDGES  = [11.5, 12.0, 13.0, 14.5, np.inf]
HMASS_BIN_LABELS = ["11.5-12", "12-13", "13-14.5", ">=14.5"]
PASSIVE_FACTOR  = 0.2           # passive if sSFR < PASSIVE_FACTOR / t_H  (== QT threshold)
TAU_SPLIT_LOG   = -1.25         # fast/slow cut on log10(tau_q / t_H(z_QT))

# ── selection: "realistically populated" in BOTH stars and gas ──
NSTAR_MIN = 20                  # min star particles at the anchor epoch  (NEW vs §0)
NGAS_MIN  = 10                  # min gas particles  (raised from 5 for profile reliability)

# ── AGN / coupling constants (residual_dust §4b, §8j) ──
EPS_R          = 0.1            # radiative/feedback efficiency for the AGN energy proxy
JET_LOGMBH     = 7.5            # jet mode: log10(M_BH/Msun) > 7.5 ...
JET_FEDD       = 0.2           #           ... AND f_Edd < 0.2
XRAY_FEDD_MAX  = 0.02          # X-ray (full-velocity jet) coupling: f_Edd <= 0.02 (Dave+19)
XRAY_FGAS_MAX  = 0.2           # ... AND f_gas = Mgas/M* < 0.2
c2_erg_per_Msun = (const.c ** 2 * u.Msun).to("erg").value
GYR = 1e9                       # yr per Gyr (event-stack clock unit)

# fast/slow styling (group A = fast, group B = slow) — the engines key off these
COL_A, COL_B, LBL_A, LBL_B = "C3", "C0", "fast", "slow"

# ── the redshift anchors this whole notebook keys off ──
ANCHOR_REDSHIFTS = [0.0]#, 0.4, 0.7, 0.9, 1.0, 4.0]
TRACK_AGE_FRAC   = 0.09         # track back to ~this fraction of the cosmic age at selection
ANCHOR_END_OVERRIDE = {}
CORRUPT_SNAPS = set()

# ── heavy-build gates (set True on the cluster, then reuse cached products) ──
BUILD_MULTI_Z      = False      # per-anchor progenitor FITS + property history HDF5
BUILD_BH           = False      # per-anchor BH accretion / AGN-energy history
BUILD_PROFILE_PLAN = False      # write per-anchor dust_profile_plan_<tag>.hdf5 (then run SLURM)
CHECK_FILE_INTEGRITY  = False   # Part 1·QC: scan original snapshots/catalogs for corruption (heavy I/O)
BUILD_PROFILE_CACHE   = False   # 6s: add new reduced profiles to the cache (heavy walk over reduced files)
REBUILD_PROFILE_CACHE = False   # 6s: force a full cache rebuild (needed after adding temperature)
BUILD_SAT_CENSUS      = False   # 6e·sat: neighbour search at each galaxy's gas_min snapshot (heavy catalog I/O)
NOVEL_ONLY   = False   # restrict each z>0 anchor to galaxies NOT on a z=0-sample progenitor track
NOVEL_REPORT = False   # just report novel/overlap counts per anchor (needs the z=0 progenitor index)

# ── paths ──
# ── ensure the 50 Mpc variant configs exist so SIM_NAME="cis50"/"cis50nox" resolve here AND in the
#    SLURM jobs that call Simulation(sim_name). Idempotent; leaves cis100/cis25 untouched. ──
_SIMBA50 = "/mnt/home/share/simbas/SIMBA_50"
_M50_CFG = {
    "cis50":    dict(data_dir=f"{_SIMBA50}/s50",    catalog_dir=f"{_SIMBA50}/s50/Groups",
                     file_format="m50n512_{snap:03d}.hdf5", snap_z_map="zsnap_map_caesar_box100.txt"),
    "cis50nox": dict(data_dir=f"{_SIMBA50}/s50nox", catalog_dir=f"{_SIMBA50}/s50nox/Groups",
                     file_format="m50n512_{snap:03d}.hdf5", snap_z_map="zsnap_map_caesar_box100.txt"),
}
_cfgp = os.path.join(os.path.expanduser("~"), ".simbanator", "config.json")
os.makedirs(os.path.dirname(_cfgp), exist_ok=True)
_cfg = json.load(open(_cfgp)) if os.path.exists(_cfgp) else {"simulations": {}}
_cfg.setdefault("simulations", {})
if any(_cfg["simulations"].get(_k) != _v for _k, _v in _M50_CFG.items()):
    _cfg["simulations"].update(_M50_CFG)
    json.dump(_cfg, open(_cfgp, "w"), indent=2)
    print("config.json: registered/updated", list(_M50_CFG))

sim     = Simulation(SIM_NAME)
OUT     = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR  = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
PLOTDIR = os.path.join(OUT, "plots", "quench_mode_sigma")
os.makedirs(TABLEDIR, exist_ok=True)
os.makedirs(PLOTDIR, exist_ok=True)

PROG_FILE     = "progenitors_most_mass.fits"
HIST_FILE     = "history_passive_z0.hdf5"
HIST_PATH     = os.path.join(SFHDIR, HIST_FILE)
BH_HIST_PATH  = os.path.join(SFHDIR, "bh_history_passive_z0.hdf5")

def figpath(name):
    """Route a figure into the topical output folders by filename prefix.

    Folders mirror the analysis's conceptual groups (numeric prefix = reading order):
      01_sanity_and_stats           QC / censoring + statistical diagnostics
      02_stacked_global_properties  event-aligned tracks, fast/slow & strong/weak
      03_dusty_vs_nondusty          the dusty/non-dusty separation + its predictors
      04_gas_geometry_catalogs      Sigma geometry from the caesar catalogs
      05_geometry_from_profiles     radial Sigma profiles from the reduced particle files
      06_sfh_checks                 star-formation histories
      07_summary_story              the Part-7 headline figures
    An unrouted name lands in 99_misc (signals a new figure prefix to add below).
    """
    base = os.path.basename(name)
    # ordered, most-specific prefix first; first startswith() match wins
    _ROUTES = [
        ("sigma_dist_reduced", "05_geometry_from_profiles"),
        ("reduced_",           "05_geometry_from_profiles"),
        ("avgprof_",           "05_geometry_from_profiles"),
        ("clock3pt",           "05_geometry_from_profiles"),
        ("stage_",             "05_geometry_from_profiles"),
        ("sigma_dist",         "04_gas_geometry_catalogs"),
        ("sigma_xstrength",    "04_gas_geometry_catalogs"),
        ("sigma_vs_fgas",      "04_gas_geometry_catalogs"),
        ("sigma_geom",         "04_gas_geometry_catalogs"),
        ("sigma_satcensus",    "03_dusty_vs_nondusty"),
        ("coupling_lag",       "01_sanity_and_stats"),
        ("synthesis_leadlag",  "01_sanity_and_stats"),
        ("censoring",          "01_sanity_and_stats"),
        ("coupling_clock",     "02_stacked_global_properties"),
        ("coupling_bymass",    "02_stacked_global_properties"),
        ("scenario_",          "02_stacked_global_properties"),
        ("response_",          "02_stacked_global_properties"),
        ("postqt_",            "02_stacked_global_properties"),
        ("agnclock_",          "02_stacked_global_properties"),
        ("compaction_",        "02_stacked_global_properties"),
        ("dustsplit",          "03_dusty_vs_nondusty"),
        ("weakdust",           "03_dusty_vs_nondusty"),
        ("sfh_",               "06_sfh_checks"),
        ("story_",             "07_summary_story"),
    ]
    sub = next((f for pre, f in _ROUTES if base.startswith(pre)), "99_misc")
    d = os.path.join(PLOTDIR, sub); os.makedirs(d, exist_ok=True)
    return os.path.join(d, base)

def statpath(name):
    """Path inside the 01_sanity_and_stats folder (stat tables + diagnostic figures)."""
    d = os.path.join(PLOTDIR, "01_sanity_and_stats"); os.makedirs(d, exist_ok=True)
    return os.path.join(d, name)
print("simulation :", SIM_NAME)
print("anchors    :", ANCHOR_REDSHIFTS)
print("selection  : passive + logM*>%.1f + nstar>=%d + ngas>=%d"
      % (MASS_FLOOR, NSTAR_MIN, NGAS_MIN))
print("z=0 t_H    : %.3f Gyr -> passive cut sSFR < %.2e /yr"
      % (COSMO.age(0).value, PASSIVE_FACTOR / (COSMO.age(0).value * 1e9)))

In [ ]:
# ── global plot style (verbatim from residual_dust_quenching.ipynb §0b) ───────
import functools
import matplotlib as mpl
from matplotlib.axes import Axes
from matplotlib.figure import Figure

PLOT_FIG_SCALE  = 1.45
PLOT_FONT_BASE  = 15
PLOT_FONT_FLOOR = 14

mpl.rcParams.update({
    "font.size":        PLOT_FONT_BASE,
    "axes.titlesize":   PLOT_FONT_BASE + 2,
    "axes.labelsize":   PLOT_FONT_BASE + 1,
    "xtick.labelsize":  PLOT_FONT_BASE,
    "ytick.labelsize":  PLOT_FONT_BASE,
    "legend.fontsize":  PLOT_FONT_BASE,
    "figure.titlesize": PLOT_FONT_BASE + 4,
    "axes.titleweight": "bold",
    "figure.dpi":       110,
    "lines.linewidth":  2.0,
})

def _bump(fs):
    if isinstance(fs, (int, float)) and not isinstance(fs, bool) and fs < PLOT_FONT_FLOOR:
        return PLOT_FONT_FLOOR
    return fs

def _patch_fontsize(cls, names):
    for nm in names:
        orig = getattr(cls, nm, None)
        if orig is None or getattr(orig, "_fontfloor", False):
            continue
        @functools.wraps(orig)
        def wrapper(*a, __orig=orig, **kw):
            if "fontsize" in kw: kw["fontsize"] = _bump(kw["fontsize"])
            if "size" in kw:     kw["size"]     = _bump(kw["size"])
            return __orig(*a, **kw)
        wrapper._fontfloor = True
        setattr(cls, nm, wrapper)

_patch_fontsize(Axes, ["set_title", "set_xlabel", "set_ylabel", "legend", "text", "annotate",
                       "set_xticklabels", "set_yticklabels"])
_patch_fontsize(Figure, ["suptitle", "legend"])
if not getattr(plt.figure, "_figscale", False):
    _orig_figure = plt.figure
    @functools.wraps(_orig_figure)
    def _figure(*a, **kw):
        base = kw.get("figsize") or mpl.rcParams["figure.figsize"]
        kw["figsize"] = (base[0] * PLOT_FIG_SCALE, base[1] * PLOT_FIG_SCALE)
        return _orig_figure(*a, **kw)
    _figure._figscale = True
    plt.figure = _figure
print(f"plot style: fonts>={PLOT_FONT_FLOOR}pt, figures x{PLOT_FIG_SCALE}")

# Part 1 — Multi-redshift anchors & per-epoch samples

Each anchor builds its **own** progenitor tracks + property history with that snapshot as row 0
(`BUILD_MULTI_Z`, cluster). The z=0 anchor reuses the existing `residual_dust_quenching.ipynb`
products. The sample at every epoch is one shared definition: **passive + massive + realistically
populated in stars and gas**.

In [ ]:
# property list to track (same families as residual_dust §1b; robust drop-and-retry on build)
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust", "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass",
        "radii.stellar_r20", "radii.stellar_r80", "radii.gas_r20", "radii.gas_r80",
        "pos", "ngas", "nstar", "ages.mass_weighted", "metallicities.mass_weighted",
        "metallicities.stellar",
        "rotation.gas_kappa_rot", "rotation.stellar_kappa_rot",
        "rotation.gas_BoverT", "rotation.stellar_BoverT",
    ],
    "halo_data": ["masses.total"],
}

def passive_sample_mask(P_, t_cosmic_yr_):
    """Passive + massive + realistically-populated-in-stars-and-gas, evaluated at row 0
    (the anchor epoch). Shared by every anchor. Returns (mask, dict of the individual cuts)."""
    mstar0 = P_["masses.stellar"][0]
    sfr0   = P_["sfr"][0]
    ngas0  = P_["ngas"][0]
    nstar0 = P_["nstar"][0] if "nstar" in P_ else np.full_like(mstar0, np.inf)
    ssfr0  = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)
    cuts = {
        "passive":    ssfr0 < (PASSIVE_FACTOR / t_cosmic_yr_[0]),
        "massive":    np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR,
        "enoughgas":  ngas0  >= NGAS_MIN,
        "enoughstar": nstar0 >= NSTAR_MIN,
    }
    m = cuts["passive"] & cuts["massive"] & cuts["enoughgas"] & cuts["enoughstar"]
    return m, cuts

def _ztag(z):
    return ("z%g" % z).replace(".", "p")


In [ ]:
# ── anchor table: snapshot, track end, and every per-anchor product path ──
_sall, _zall = [], []
for _s in range(0, START_SNAP + 1):
    try:
        _zv = float(sim.get_z_from_snap(_s))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)
_aall = COSMO.age(_zall).value

ANCHORS = {}
for _zt in ANCHOR_REDSHIFTS:
    if _zt == 0.0:
        ANCHORS[_zt] = dict(z_target=0.0, tag=_ztag(0.0), snap=START_SNAP,
                            z=float(sim.get_z_from_snap(START_SNAP)), end_snap=END_SNAP,
                            prog_file=PROG_FILE, hist_path=HIST_PATH, bh_path=BH_HIST_PATH)
    else:
        _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
        _age_end = TRACK_AGE_FRAC * float(_aall[_sall == _snap][0])
        _end = int(ANCHOR_END_OVERRIDE.get(_zt, int(_sall[np.searchsorted(_aall, _age_end)])))
        _tag = _ztag(_zt)
        ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap,
                            z=float(sim.get_z_from_snap(_snap)), end_snap=_end,
                            prog_file=f"progenitors_anchor_{_tag}.fits",
                            hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"),
                            bh_path=os.path.join(SFHDIR, f"bh_history_anchor_{_tag}.hdf5"))
    # each anchor gets its OWN plan dir so the SLURM job's profiles_part_*.hdf5 partials
    # (named by task id) never collide across anchors sharing SFHDIR.
    ANCHORS[_zt]["plan_path"] = os.path.join(SFHDIR, f"prof_{ANCHORS[_zt]['tag']}",
                                             f"dust_profile_plan_{ANCHORS[_zt]['tag']}.hdf5")

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'end':>5s} {'hist':>6s} {'BH':>4s}")
for _zt, A in ANCHORS.items():
    print(f"{_zt:6.1f} {A['snap']:5d} {A['z']:7.3f} {A['end_snap']:5d} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(A['bh_path']) else '--':>4s}")

# Part 1·QC — Original SIMBA file integrity (before anything else)

The reduced-Σ gaps trace back to **unreadable original SIMBA files** (truncated / half-staged
downloads), not to the extraction. This cell scans every snapshot the anchors touch and reports
which **snapshot** and **CAESAR catalog** HDF5 files are missing or corrupt (the same open-and-catch
test the pipeline uses). It then writes a reviewable `redownload_corrupt.sh`, and — gated behind
`REDOWNLOAD_CORRUPT` with the Simba base URLs set — re-fetches the bad files into a writable `DOWNLOAD_DIR` (the config `data_dir` may be a shared, read-only share — writing there gives *Permission denied*),
verifies each download opens, and replaces in place. A progress bar tracks the scan, and `DOWNLOAD_WORKERS` parallelises the fetch, and `DOWNLOAD_SCOPE` chooses whether to re-fetch only the **corrupt** (on-disk-but-broken) files or **both** corrupt and **absent** (never-downloaded) ones. Re-download is the only fix that recovers the
lost points; it can't run from the notebook unattended, so it is gated.

In [ ]:
# ── file-integrity scan (with progress) + scope-selectable, parallel re-download ──
def _hdf5_status(path, all_of=()):
    """'ok' | 'missing' | 'no-perm' | 'empty' | 'no-keys' | 'no:<grp>' | 'corrupt:<Err>'."""
    if not os.path.exists(path):
        return "missing"
    if not os.access(path, os.R_OK):
        return "no-perm"                                # exists but unreadable (e.g. shared dir) — NOT corrupt
    if os.path.getsize(path) == 0:
        return "empty"
    try:
        with h5py.File(path, "r") as f:
            keys = set(f.keys())
            if not keys:
                return "no-keys"
            for g in all_of:
                if g not in f:
                    return f"no:{g}"
        return "ok"
    except Exception as e:                              # truncated / corrupt HDF5 raises OSError here
        return f"corrupt:{type(e).__name__}"

try:                                                    # progress bar; falls back to plain \r if no tqdm
    from tqdm.auto import tqdm
    def _pbar(seq, desc, total=None):
        return tqdm(seq, desc=desc, total=total, leave=False)
except Exception:
    def _pbar(seq, desc, total=None):
        seq = list(seq); n = total or len(seq) or 1
        for k, x in enumerate(seq, 1):
            if k == 1 or k % max(1, n // 20) == 0 or k == n:
                print(f"\r  {desc}: {k}/{n} ({100 * k // n}%)", end="", flush=True)
            yield x
        print()

if not CHECK_FILE_INTEGRITY:
    print("[QC] file-integrity scan skipped (set CHECK_FILE_INTEGRITY=True to run).")
else:
    _snaps_qc = sorted({s for A in ANCHORS.values() for s in range(int(A["end_snap"]), int(A["snap"]) + 1)})
    print(f"[{SIM_NAME}] data_dir={sim.data_dir}\n  catalog_dir={sim.catalog_dir}")
    print(f"  checking {len(_snaps_qc)} snapshots [{_snaps_qc[0]}..{_snaps_qc[-1]}] (snapshot + catalog HDF5) ...")
    bad = []                                                # (snap, kind, path, status)
    for s in _pbar(_snaps_qc, "scanning"):
        sp, cp = sim.get_snapshot_file(s), sim.get_caesar_file(s)
        ss = _hdf5_status(sp, all_of=("Header", "PartType0"))    # snapshots always carry these
        cs = _hdf5_status(cp)                                    # CAESAR groups vary -> open+keys is enough
        if ss != "ok":
            bad.append((s, "snap", sp, ss))
        if cs != "ok":
            bad.append((s, "catalog", cp, cs))

    _absent  = [b for b in bad if b[3] == "missing"]        # not on disk
    _noperm  = [b for b in bad if b[3] == "no-perm"]        # exists but you can't read it (shared dir)
    _corrupt = [b for b in bad if b[3] not in ("missing", "no-perm")]   # on disk, unreadable/truncated
    if not bad:
        print("  all snapshot & catalog files OK.")
    else:
        print(f"\n  {len(_corrupt)} corrupt + {len(_absent)} absent + {len(_noperm)} no-permission file(s):")
        for s, kind, path, status in bad:
            print(f"    snap {s:3d}  {kind:7s}  [{status:14s}]  {path}")

    # ── re-download from the online Simba repository (parallel) ──
    REDOWNLOAD_CORRUPT = False     # True -> fetch + verify + place the files
    DOWNLOAD_SCOPE     = "both"    # "corrupt" = on-disk-but-broken ; "both" = corrupt + absent (no-perm always re-fetched if DOWNLOAD_DIR set)
    DOWNLOAD_WORKERS   = 4         # parallel wget processes
    # The config data_dir above may be a SHARED, READ-ONLY path (wget there -> 'Permission denied').
    # DOWNLOAD_DIR is a directory YOU own: fetched files land here (basename preserved), never the share.
    # Afterwards point ~/.simbanator/config.json data_dir (and catalog_dir) here so the analysis uses them.
    DOWNLOAD_DIR   = ""   # "" = write in place (data_dir) under the proper name. Corrupt originals were
                         # renamed to *_corr.hdf5, so those names are free to (re)create — no overwrite of
                         # another user's file. Set to a dir you own only if data_dir itself is unwritable.
    SIMBA_SNAP_URL = "http://simba.roe.ac.uk/simdata/m100n1024/s50/snapshots"   # snap_m100n1024_NNN.hdf5
    SIMBA_CAT_URL  = "http://simba.roe.ac.uk/simdata/m100n1024/s50/catalogs"    # m100n1024_NNN.hdf5 (verify)

    def _remote_url(kind, path):
        base = SIMBA_SNAP_URL if kind == "snap" else SIMBA_CAT_URL
        return f"{base.rstrip('/')}/{os.path.basename(path)}" if base else None

    def _dest(path):                                        # where the fresh copy is written
        return os.path.join(DOWNLOAD_DIR, os.path.basename(path)) if DOWNLOAD_DIR else path

    targets = []
    for b in bad:
        st = b[3]
        if st == "no-perm":
            if DOWNLOAD_DIR:
                targets.append(b)                           # can only re-fetch elsewhere; the share is read-only
        elif st == "missing":
            if DOWNLOAD_SCOPE == "both":
                targets.append(b)
        else:
            targets.append(b)                               # corrupt
    if _noperm and not DOWNLOAD_DIR:
        print(f"\n  NOTE: {len(_noperm)} file(s) are unreadable in {sim.data_dir} (shared/read-only) and the "
              f"originals cannot be overwritten -> set DOWNLOAD_DIR to a path you own to fetch fresh copies.")

    if targets:
        DL = os.path.join(os.getcwd(), "redownload_corrupt.sh")
        with open(DL, "w") as sh:
            sh.write(f"#!/bin/bash\n# generated by quench_mode_vs_sigma_gas.ipynb: re-fetch SIMBA files "
                     f"(scope={DOWNLOAD_SCOPE}); up to $WORKERS in parallel\n")
            sh.write(f"WORKERS=${{WORKERS:-{int(DOWNLOAD_WORKERS)}}}\n")
            if DOWNLOAD_DIR:
                sh.write(f'mkdir -p "{DOWNLOAD_DIR}"\n')
            sh.write('gate() { while [ "$(jobs -rp | wc -l)" -ge "$WORKERS" ]; do sleep 0.5; done; }\n\n')
            for s, kind, path, status in targets:
                url = _remote_url(kind, path) or "<SET_BASE_URL>"; d = _dest(path)
                sh.write(f'gate; wget -c --progress=dot:giga -o "{d}.log" -O "{d}" "{url}" &   # snap {s} ({status})\n')
            sh.write("\nwait\necho 'redownload complete'\n")
        print(f"\n  wrote {DL}: {len(targets)} file(s) -> {DOWNLOAD_DIR or '(in place)'} "
              f"[scope={DOWNLOAD_SCOPE}, {DOWNLOAD_WORKERS} parallel]")
        print(f"  run: bash {os.path.basename(DL)}   (watch a file with  tail -f <dest>.log )")

        if REDOWNLOAD_CORRUPT:
            import subprocess
            from concurrent.futures import ThreadPoolExecutor

            def _fetch_one(t):
                s, kind, path, status = t
                url = _remote_url(kind, path)
                if not url:
                    return t, False, "no-url"
                dest = _dest(path); dd = os.path.dirname(dest) or "."
                try:
                    os.makedirs(dd, exist_ok=True)
                except Exception as e:
                    return t, False, f"mkdir:{type(e).__name__}"
                if not os.access(dd, os.W_OK):
                    return t, False, "dest-not-writable"
                tmp = dest + ".part"
                rc = subprocess.run(["wget", "-q", "-c", "-O", tmp, url]).returncode
                chk = _hdf5_status(tmp, all_of=("Header", "PartType0") if kind == "snap" else ())
                if rc == 0 and chk == "ok":
                    os.replace(tmp, dest); return t, True, dest
                return t, False, f"rc={rc},{chk}"

            nw = max(1, int(DOWNLOAD_WORKERS))
            print(f"  downloading {len(targets)} file(s) -> {DOWNLOAD_DIR or '(in place)'} with {nw} workers ...")
            fails = []
            with ThreadPoolExecutor(max_workers=nw) as ex:
                futs = [ex.submit(_fetch_one, t) for t in targets]
                for fut in _pbar(futs, "downloading", total=len(futs)):
                    t, ok, info = fut.result()
                    if not ok:
                        fails.append((t, info))
            if fails:
                print(f"  {len(fails)} download(s) FAILED:")
                for (s, kind, path, status), info in fails:
                    print(f"    snap {s:3d} {kind:7s} [{info}]  {os.path.basename(path)}")
            else:
                print(f"  all {len(targets)} downloads OK -> {DOWNLOAD_DIR or '(in place)'}")
                if DOWNLOAD_DIR and DOWNLOAD_DIR not in (sim.data_dir, sim.catalog_dir):
                    print(f"  point ~/.simbanator/config.json data_dir/catalog_dir at {DOWNLOAD_DIR} to use them.")
    elif bad:
        print(f"\n  no download targets under scope='{DOWNLOAD_SCOPE}'"
              f"{' (set DOWNLOAD_DIR for the no-perm files)' if _noperm and not DOWNLOAD_DIR else ''}.")

In [ ]:
# ── 1z·build (GATED, cluster): per-anchor progenitor table + property history ──
# Verbatim port of residual_dust §1z·build; z=0 reuses the §1 products.
if BUILD_MULTI_Z:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["hist_path"]):          # reuse if present (e.g. residual_dust z=0); else build it
            print(f"[{A['tag']}] cached -> {os.path.basename(A['hist_path'])}"); continue
        end = int(A["end_snap"])
        while end < A["snap"] and (end in CORRUPT_SNAPS or not os.path.exists(sim.get_caesar_file(end))):
            end += 1
        A["end_snap"] = end
        print(f"[{A['tag']}] anchor snap {A['snap']} (z={A['z']:.2f}) <- {end}: progenitor table ...")
        cs_a = sim.load_catalog(snap=A["snap"])
        caesar_read_progen([g.GroupID for g in cs_a.galaxies], A["prog_file"],
                           range(end, A["snap"] + 1), sim, output_dir=None)
        hist = HDF5BuildHistory(sim, cs_a, progfilename=A["prog_file"])
        with fits.open(hist.progen_file) as hdul:
            valid_ids = np.asarray(hdul[1].data["GroupID"])
            _tHa = COSMO.age(float(A["z"])).value * 1e9
            _gid = np.array([g.GroupID for g in cs_a.galaxies])
            _ms  = np.array([float(g.masses["stellar"]) for g in cs_a.galaxies])
            _sf  = np.array([float(g.sfr) for g in cs_a.galaxies])
            with np.errstate(all="ignore"):
                _ss = np.where(_ms > 0, _sf / _ms, np.nan)
                _ok = (np.log10(np.where(_ms > 0, _ms, np.nan)) >= MASS_FLOOR) & (_ss < PASSIVE_FACTOR / _tHa)
            _keep = {int(g) for g in _gid[_ok]}
            valid_ids = np.asarray([i for i in valid_ids if int(i) in _keep], dtype=valid_ids.dtype)
            print(f"  [pre-select] {len(valid_ids)}/{len(_gid)} massive+quenched at z={A['z']:.2f}")
        hist.get_history_indx(valid_ids, A["snap"], end)
        props_try = {k: list(v) for k, v in PROPS.items()}
        while True:
            try:
                hist.get_property_history(props_try, verbose=0); break
            except KeyError as e:
                msg = str(e); dropped = False
                for fam, plist in props_try.items():
                    for pr in list(plist):
                        if pr in msg or pr.split("/")[-1] in msg:
                            plist.remove(pr); print("  [drop]", pr); dropped = True
                if not dropped:
                    raise
        hist.save_history_to_hdf5(os.path.basename(A["hist_path"]))
        del cs_a, hist; gc.collect()
        print(f"[{A['tag']}] history -> {A['hist_path']}")
else:
    print("BUILD_MULTI_Z=False -> expecting per-anchor histories under", SFHDIR)

In [ ]:
# ── 1z·load — generic anchor loader (row 0 of every array = the anchor epoch) ──
def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H

def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows.
    Needed only to BUILD the BH history and the profile plan (catalogue load)."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP; gc.collect()
    return M

# Part 2 — Machinery (anchor-parameterised functions)

Each function takes one anchor's arrays and returns its products, so the same proven logic runs at
every redshift. Bodies are ported from `residual_dust_quenching.ipynb` (§3 critical points,
§2·diag QC, §4b BH, §8j coupling, §5 profiles, §8e/§8g clock engine, §7 plotters).

In [ ]:
# ── §3 → build_records: critical points + fast/slow + mass bins (verbatim §3 logic) ──
def build_records(P, t_cosmic_yr, redshift, galaxy_ids, cols,
                  MASS_BIN_EDGES, HMASS_BIN_EDGES, PASSIVE_FACTOR, TAU_SPLIT_LOG, COSMO,
                  find_quenching_times):
    """Per-galaxy critical evolutionary points & fast/slow classification for ONE anchor.
    `cols` = column indices (into n_gal) of the selected sample. Adds a `formation` stage
    (earliest snapshot with M*>0) to the residual_dust STAGES list."""
    t_cosmic_yr = np.asarray(t_cosmic_yr, float); redshift = np.asarray(redshift, float)
    galaxy_ids = np.asarray(galaxy_ids); cols = np.asarray(cols, int)
    sample_gids = galaxy_ids[cols]
    STAGES = ["formation", "sf_peak", "ssfr_min", "sft", "qt", "post_quench",
              "dust_peak", "dust_min", "gas_min", "z0"]
    SMOOTH_W = 3

    def _nearest_row(t_target):
        if not np.isfinite(t_target):
            return -1
        return int(np.argmin(np.abs(t_cosmic_yr - t_target)))

    def _smooth(y, w=SMOOTH_W):
        n = len(y); h = w // 2; out = np.full(n, np.nan)
        for i in range(n):
            seg = y[max(0, i - h):min(n, i + h + 1)]; seg = seg[np.isfinite(seg)]
            if seg.size:
                out[i] = np.median(seg)
        return out

    def _ordered_positive(q, valid):
        d = np.where(valid, q, np.nan).astype(float)
        pos = np.isfinite(d) & (d > 0)
        if pos.sum() < 2:
            return None, None
        order = np.where(pos)[0]; order = order[np.argsort(t_cosmic_yr[order])]
        return order, _smooth(d[order])

    def _peak_time(q, valid):
        order, ds = _ordered_positive(q, valid)
        if order is None or not np.isfinite(ds).any():
            return np.nan
        return t_cosmic_yr[order[np.nanargmax(ds)]]

    def _trough_before_floor(q, valid, floor_frac=1e-6):
        order, ds = _ordered_positive(q, valid)
        if order is None or not np.isfinite(ds).any():
            return np.nan
        floor = floor_frac * np.nanmax(ds)
        below = np.isfinite(ds) & (ds <= floor)
        cut = int(np.argmax(below)) if below.any() else len(ds)
        cut = max(cut, 2); seg = ds[:cut]
        if not np.isfinite(seg).any():
            return np.nan
        return t_cosmic_yr[order[np.nanargmin(seg)]]

    def _formation_row(col):
        mstar = P["masses.stellar"][:, col]
        has = np.where(np.isfinite(mstar) & (mstar > 0))[0]
        return int(has.max()) if has.size else -1

    records = []
    for col, gid in zip(cols, sample_gids):
        mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
        dust = P["masses.dust"][:, col]; gas = P["masses.gas"][:, col]
        ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
        valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
        if valid.sum() < 5:
            continue
        t = t_cosmic_yr[valid]; s = ssfr[valid]
        o = np.argsort(t); t, s = t[o], s[o]
        tu, ui = np.unique(t, return_index=True); su = s[ui]
        if len(tu) < 5:
            continue
        qts, sfts, _, dbg = find_quenching_times(
            tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True)
        if len(qts) == 0:
            continue
        k = int(np.argmax(qts)); t_qt, t_sft = qts[k], sfts[k]
        t_postq = dbg["persistence_end_times"][k]
        kf = int(np.argmin(qts)); t_qt_first, t_sft_first = qts[kf], sfts[kf]
        t_sfpeak = _peak_time(ssfr, valid); t_ssfrmin = _trough_before_floor(ssfr, valid)
        t_dpeak = _peak_time(dust, valid); t_dmin = _trough_before_floor(dust, valid)
        t_gasmin = _trough_before_floor(gas, valid); t_z0 = t_cosmic_yr[0]
        row_form = _formation_row(col); t_form = t_cosmic_yr[row_form] if row_form >= 0 else np.nan
        t_times = {"formation": t_form, "sf_peak": t_sfpeak, "ssfr_min": t_ssfrmin,
                   "sft": t_sft, "qt": t_qt, "post_quench": t_postq, "dust_peak": t_dpeak,
                   "dust_min": t_dmin, "gas_min": t_gasmin, "z0": t_z0}
        rows = {st: _nearest_row(t_times[st]) for st in STAGES}
        rows["formation"] = row_form
        tau_q = t_qt - t_sft
        z_qt = float(np.interp(t_qt, t_cosmic_yr[::-1], redshift[::-1]))
        tH_qt = COSMO.age(z_qt).value * 1e9
        records.append(dict(
            gid=int(gid), col=int(col), t_formation=t_form,
            t_sf_peak=t_sfpeak, t_ssfr_min=t_ssfrmin, t_sft=t_sft, t_qt=t_qt,
            t_post_quench=t_postq, t_dust_peak=t_dpeak, t_dust_min=t_dmin,
            t_gas_min=t_gasmin, t_z0=t_z0,
            tau_q=tau_q, tau_q_over_tH=tau_q / tH_qt, z_qt=z_qt,
            t_qt_first=t_qt_first, t_sft_first=t_sft_first,
            **{f"row_{st}": rows[st] for st in STAGES}))

    n = len(records)
    cols_rec = np.array([r["col"] for r in records], int) if n else np.array([], int)
    gids_rec = np.array([r["gid"] for r in records]) if n else np.array([])
    tau_q = np.array([r["tau_q"] for r in records]) if n else np.array([])
    tau_n = np.array([r["tau_q_over_tH"] for r in records]) if n else np.array([])
    is_fast = np.log10(tau_n) < TAU_SPLIT_LOG if n else np.array([], bool)
    is_slow = (np.isfinite(tau_n) & ~is_fast) if n else np.array([], bool)
    if n:
        ms0 = P["masses.stellar"][0, cols_rec]; mh0 = P["masses.total"][0, cols_rec]
        logm_rec = np.log10(np.where(ms0 > 0, ms0, np.nan))
        logmh_rec = np.log10(np.where(mh0 > 0, mh0, np.nan))
    else:
        logm_rec = np.array([]); logmh_rec = np.array([])
    mbin = np.full(n, -1, int)
    for b in range(len(MASS_BIN_EDGES) - 1):
        mbin[(logm_rec >= MASS_BIN_EDGES[b]) & (logm_rec < MASS_BIN_EDGES[b + 1])] = b
    hmbin = np.full(n, -1, int)
    for b in range(len(HMASS_BIN_EDGES) - 1):
        hmbin[(logmh_rec >= HMASS_BIN_EDGES[b]) & (logmh_rec < HMASS_BIN_EDGES[b + 1])] = b
    return dict(records=records, STAGES=STAGES, tau_q=tau_q, tau_q_over_tH=tau_n,
                is_fast=is_fast, is_slow=is_slow, mbin=mbin, hmbin=hmbin,
                logm_rec=logm_rec, logmh_rec=logmh_rec, gids=gids_rec, cols=cols_rec)

In [ ]:
# ── §2·diag QC: well-behaved histories & halo tracking (per-galaxy boolean mask) ──
# Ported from residual_dust §2·diag2 (spatial continuity), §2·diag4 (halo-mass continuity),
# S2 (halo association). Thresholds copied verbatim. `redshift` passed directly (the anchor
# history carries it) instead of inverting COSMO.age.
def qc_anchor(P, t_cosmic_yr, redshift, snaps_arr, galaxy_ids, cols, BOX, tag):
    JUMP_FRAC, SPEED_HI = 0.05, 0.05      # §2·diag2: step>5% box; box-frac/Gyr implausible
    RATE_HI = 0.5                          # §2·diag4: dex/Gyr implausible
    BAD_FRAC, MIN_STEPS = 0.2, 3           # >20% over-rate steps & need >=3 steps to judge
    SHMR_HI = 0.2                          # S2: M*/Mhalo>0.2 suspicious
    cols = np.asarray(cols, int); ncol = cols.size
    t = np.asarray(t_cosmic_yr, float); dt = np.abs(np.diff(t)) / 1e9
    z = np.asarray(redshift, float)

    def _to_comoving(posc):
        fin = np.isfinite(posc)
        if not fin.any():
            return posc, 1.0
        def _ext(row):
            a = posc[row][np.isfinite(posc[row])]
            return float(np.nanmax(a) - np.nanmin(a)) if a.size else np.nan
        bs, be = _ext(0), _ext(-1)
        as_, ae = 1.0 / (1.0 + z[0]), 1.0 / (1.0 + z[-1])
        phys = (np.isfinite(bs) and np.isfinite(be)
                and abs(be / bs - ae / as_) < abs(be / bs - 1.0))
        if phys:
            out = posc * (1.0 + z)[:, None, None]
            L = bs / as_ if np.isfinite(bs) else np.nanpercentile(out[np.isfinite(out)], 99.99)
        else:
            out = posc
            L = bs if np.isfinite(bs) else np.nanpercentile(out[np.isfinite(out)], 99.99)
        return out, float(L)

    spatial_ok = np.ones(ncol, bool); n_broken = 0
    if "pos" in P:
        _pos = np.asarray(P["pos"], float)
        if _pos.ndim == 3 and _pos.shape[-1] == 3:
            posc, L = _to_comoving(_pos[:, cols, :])
            d = np.diff(posc, axis=0); d -= L * np.round(d / L)
            disp = np.sqrt(np.sum(d ** 2, axis=2)) / L
            with np.errstate(all="ignore"):
                speed = disp / np.where(dt > 0, dt, np.nan)[:, None]
                galbad = np.nansum(speed > SPEED_HI, axis=0) / np.maximum(np.sum(np.isfinite(speed), axis=0), 1)
            tracked = np.sum(np.isfinite(disp), axis=0) >= MIN_STEPS
            broken = tracked & (galbad > BAD_FRAC); n_broken = int(broken.sum()); spatial_ok = ~broken

    halo_ok = np.ones(ncol, bool); n_tele = 0
    if "masses.total" in P:
        _mh = np.asarray(P["masses.total"], float)[:, cols]
        with np.errstate(all="ignore"):
            lmh = np.log10(np.where(_mh > 0, _mh, np.nan))
            rate = np.abs(np.diff(lmh, axis=0)) / np.where(dt > 0, dt, np.nan)[:, None]
            halobad = np.nansum(rate > RATE_HI, axis=0) / np.maximum(np.sum(np.isfinite(rate), axis=0), 1)
        trk = np.sum(np.isfinite(np.diff(lmh, axis=0)), axis=0) >= MIN_STEPS
        susp = trk & (halobad > BAD_FRAC); n_tele = int(susp.sum()); halo_ok = ~susp

    _ms0 = np.asarray(P["masses.stellar"], float)[0, cols]
    _mh0 = np.asarray(P["masses.total"], float)[0, cols]
    with np.errstate(all="ignore"):
        shmr = np.where(_mh0 > 0, _ms0 / _mh0, np.nan)
    bad_ratio = np.isfinite(shmr) & (shmr > SHMR_HI)
    bad_order = np.isfinite(_ms0) & np.isfinite(_mh0) & (_mh0 < _ms0)
    no_halo = ~np.isfinite(_mh0) | (_mh0 <= 0)
    assoc_ok = ~(bad_ratio | bad_order | no_halo)
    median_shmr = float(np.nanmedian(shmr))

    well_behaved = spatial_ok & halo_ok & assoc_ok
    print(f"{tag}: well-behaved {int(well_behaved.sum())}/{ncol} | broken-link {n_broken} "
          f"| halo-jump {n_tele} | bad-assoc {int((bad_ratio | bad_order).sum())} "
          f"| no-halo {int(no_halo.sum())} | median M*/Mh {median_shmr:.4f}")
    return dict(well_behaved=well_behaved, n_broken_link=n_broken, n_halo_teleport=n_tele,
                n_bad_assoc=int((bad_ratio | bad_order).sum()), n_no_halo=int(no_halo.sum()),
                median_shmr=median_shmr)

In [ ]:
# ── §4b/§7j BH: per-anchor build + load + AGN metrics + AGN-activation time ──
BH_CANDIDATES = {"bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
                 "bh_mdot": ["bhmdot", "bh_mdot"],
                 "bh_fedd": ["bh_fedd", "bhfedd", "fedd"]}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

def build_bh_for_anchor(A, galaxy_ids, snaps_arr, n_gal):
    """Verbatim port of residual_dust §1 BH build, per anchor -> A['bh_path']."""
    pidx = build_prog_index(A, galaxy_ids, snaps_arr)
    n_snap = len(snaps_arr)
    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri]); cv = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    if p is not None:
                        BH[k][ri, cv] = f[p][:][vi]
        except (OSError, KeyError) as e:
            print(f"  [skip] snap {snap}: {type(e).__name__}"); CORRUPT_SNAPS.add(snap)
    with h5py.File(A["bh_path"], "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print(f"[{A['tag']}] BH history -> {A['bh_path']}")
    return BH

def load_bh(bh_hist_path):
    with h5py.File(bh_hist_path, "r") as f:
        return {k: f[k][:] for k in f.keys()}

def bh_metrics(BH, records, t_cosmic_yr, EPS_R, c2_erg_per_Msun, JET_LOGMBH, JET_FEDD):
    def _sft_qt_rows(r):
        rs, rq = r["row_sft"], r["row_qt"]
        if rs < 0 or rq < 0:
            return np.array([], int)
        lo, hi = sorted((rs, rq)); return np.arange(lo, hi + 1)
    n = len(records); dMbh, Eagn, jetfrac, mdot_qt = (np.full(n, np.nan) for _ in range(4))
    for i, r in enumerate(records):
        col = r["col"]
        mdot_qt[i] = BH["bh_mdot"][r["row_qt"], col] if r["row_qt"] >= 0 else np.nan
        idx = _sft_qt_rows(r)
        if len(idx) < 2:
            continue
        t = t_cosmic_yr[idx]; o = np.argsort(t); idx = idx[o]; t = t[o]
        md = BH["bh_mdot"][idx, col]; mb = BH["bh_mass"][idx, col]; fe = BH["bh_fedd"][idx, col]
        good = np.isfinite(md) & np.isfinite(t)
        if good.sum() >= 2:
            Eagn[i] = EPS_R * np.trapz(md[good], t[good]) * c2_erg_per_Msun
        if np.isfinite(mb).sum() >= 2:
            mbv = mb[np.isfinite(mb)]; dMbh[i] = mbv[-1] - mbv[0]
        jet = np.isfinite(mb) & np.isfinite(fe) & (mb > 10 ** JET_LOGMBH) & (fe < JET_FEDD)
        vs = np.isfinite(mb) & np.isfinite(fe)
        jetfrac[i] = jet.sum() / vs.sum() if vs.sum() > 0 else np.nan
    return dict(Eagn=Eagn, dMbh=dMbh, jetfrac=jetfrac, mdot_qt=mdot_qt)

def agn_activation_time(BH, records, t_cosmic_yr, IGN_FRAC=0.5, IGN_FLOOR=1e-3, GYR=1e9):
    """t_AGN = first upward crossing of IGN_FRAC*(pre-quench peak BHAR) at/before QT (§8c)."""
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    def _track(arr2d, col):
        return arr2d[_ord, col].astype(float)
    t_agn = np.full(len(records), np.nan)
    for i, r in enumerate(records):
        col, tqt = r["col"], r["t_qt"]
        before = t_inc <= tqt + 0.05 * GYR
        md = _track(BH["bh_mdot"], col)
        m = before & np.isfinite(md) & (md > IGN_FLOOR)
        if m.sum() >= 2:
            thr = max(IGN_FRAC * np.nanmax(md[m]), IGN_FLOOR)
            cr = np.where(before & np.isfinite(md) & (md >= thr))[0]
            if len(cr):
                t_agn[i] = t_inc[cr[0]]
    return t_agn

In [ ]:
# ── §8j AGN–ISM coupling (verbatim): xstr_post strength split = SPLIT_XS ──
# Coupling strength is the mean of xcoup_hist over snapshots AFTER t_AGN (xstr_post), exactly as
# §8j. SPLIT_XS = top/bottom terciles of xstr_post ("strong"/"weak"); SPLIT_X = strict binary
# (xfrac_post>0). no_agn = ignited-but-never-coupled (xstr_post==0). An optional mass window
# (in_win) reproduces §8j's intermediate-mass restriction; default = whole sample.
def build_coupling(BH, P, records, t_cosmic_yr, t_agn,
                   JET_LOGMBH, JET_FEDD, XRAY_FEDD_MAX, XRAY_FGAS_MAX, in_win=None):
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    def _track(arr2d, col):
        return arr2d[_ord, col].astype(float)
    with np.errstate(all="ignore"):
        fgas_hist = np.where(P["masses.stellar"] > 0, P["masses.gas"] / P["masses.stellar"], np.nan)
        _bh_ok = np.isfinite(BH["bh_mass"]) & np.isfinite(BH["bh_fedd"])
        _mbh_ok = BH["bh_mass"] > 10 ** JET_LOGMBH
        wjet_hist = np.where(_bh_ok, np.where(_mbh_ok,
                             np.clip(np.log10(JET_FEDD / np.clip(BH["bh_fedd"], 1e-12, None)), 0.0, 1.0),
                             0.0), np.nan)
        xray_hist = np.where(_bh_ok & np.isfinite(fgas_hist),
                             (_mbh_ok & (BH["bh_fedd"] < XRAY_FEDD_MAX) & (fgas_hist < XRAY_FGAS_MAX)).astype(float),
                             np.nan)
        xcoup_hist = np.where(np.isfinite(wjet_hist) & np.isfinite(fgas_hist),
                              wjet_hist * (fgas_hist < XRAY_FGAS_MAX).astype(float), np.nan)
    n = len(records)
    xfrac_post = np.full(n, np.nan); xstr_post = np.full(n, np.nan)   # post-t_AGN strict frac / strength
    for i, r in enumerate(records):
        if not np.isfinite(t_agn[i]):
            continue
        post = t_inc >= t_agn[i]
        xs = _track(xray_hist, r["col"]); ok = post & np.isfinite(xs)
        if ok.sum() >= 2:
            xfrac_post[i] = np.nanmean(xs[ok])
        cs = _track(xcoup_hist, r["col"]); ok = post & np.isfinite(cs)
        if ok.sum() >= 2:
            xstr_post[i] = np.nanmean(cs[ok])
    base = np.ones(n, bool) if in_win is None else np.asarray(in_win, bool)
    bx = base & np.isfinite(xstr_post)
    lo_q = hi_q = np.nan; strong = np.zeros(n, bool); weak = np.zeros(n, bool)
    if bx.sum() >= 3:
        lo_q, hi_q = np.nanquantile(xstr_post[bx], [1.0 / 3.0, 2.0 / 3.0])
        strong = bx & (xstr_post >= hi_q); weak = bx & (xstr_post <= lo_q)   # §8j SPLIT_XS terciles
    no_agn = base & np.isfinite(xstr_post) & (xstr_post == 0)                 # ignited, never coupled (subset of weak)
    inter  = bx & ~strong & ~weak                                            # middle tercile of post-AGN coupling
    no_fb  = base & ~np.isfinite(xstr_post)                                   # never ignited / no measurable coupling
    SPLIT_X  = [(base & np.isfinite(xfrac_post) & (xfrac_post > 0),  "C3", "ISM-coupled"),
                (base & np.isfinite(xfrac_post) & (xfrac_post == 0), "C0", "never coupled")]
    # complete 4-way partition: strong + weak + inter + no_fb == base (used everywhere coupling splits)
    SPLIT_XS = [(strong, "C3",  f"strong (>={hi_q:.2f})"),
                (weak,   "C0",  f"weak (<={lo_q:.2f})"),
                (inter,  "C2",  "intermediate"),
                (no_fb,  "0.5", "no AGN")]
    return dict(fgas_hist=fgas_hist, wjet_hist=wjet_hist, xray_hist=xray_hist, xcoup_hist=xcoup_hist,
                xfrac_post=xfrac_post, xstr_post=xstr_post,
                no_agn=no_agn, weak=weak, strong=strong, inter=inter, no_fb=no_fb, tercile=(lo_q, hi_q),
                SPLIT_X=SPLIT_X, SPLIT_XS=SPLIT_XS)

In [ ]:
# ── §5 particle profiles: per-anchor plan, loader, and Sigma scalars ──
def build_profile_plan(records, STAGES, snaps_arr, _PROG_INDEX, plan_path, sim_name,
                       rmax_kpc=50.0, nbins_r=25, store_sigma=1):
    """Write the per-anchor SLURM extraction plan (residual_dust §5a)."""
    _ri, _si, _gx, _snap = [], [], [], []
    for ri, r in enumerate(records):
        for si, st in enumerate(STAGES):
            row = r[f"row_{st}"]
            if row < 0:
                continue
            gx = _PROG_INDEX[row, r["col"]]
            if not np.isfinite(gx):
                continue
            _ri.append(ri); _si.append(si); _gx.append(int(gx)); _snap.append(int(snaps_arr[row]))
    os.makedirs(os.path.dirname(plan_path), exist_ok=True)
    with h5py.File(plan_path, "w") as out:
        out.attrs["nrec"] = len(records); out.attrs["nst"] = len(STAGES)
        out.attrs["sim_name"] = sim_name; out.attrs["rmax_kpc"] = rmax_kpc
        out.attrs["nbins_r"] = nbins_r; out.attrs["store_sigma"] = int(store_sigma)
        out.attrs["stages"] = ",".join(STAGES)
        out.create_dataset("entry_ri", data=np.array(_ri, np.int32))
        out.create_dataset("entry_si", data=np.array(_si, np.int32))
        out.create_dataset("entry_gx", data=np.array(_gx, np.int64))
        out.create_dataset("entry_snap", data=np.array(_snap, np.int32))
        out.create_dataset("gid", data=np.array([r["gid"] for r in records], np.int64))
    print(f"wrote plan: {len(_ri)} (galaxy,stage) entries -> {plan_path}")
    return plan_path

def _align_records_by_gid(arrays, cache_gid, cur_gid, axis=0, name=""):
    cache_gid = np.asarray(cache_gid).astype(np.int64); cur_gid = np.asarray(cur_gid).astype(np.int64)
    if len(cache_gid) == len(cur_gid) and np.array_equal(cache_gid, cur_gid):
        return arrays, len(cur_gid)
    pos = {int(g): j for j, g in enumerate(cache_gid)}
    sel = np.array([pos.get(int(g), -1) for g in cur_gid]); found = sel >= 0
    out = {}
    for k, a in arrays.items():
        a = np.asarray(a)
        if a.ndim <= axis or a.shape[axis] != len(cache_gid):
            out[k] = a; continue
        nshape = list(a.shape); nshape[axis] = len(cur_gid)
        b = np.full(nshape, np.nan, float); idx = np.where(found)[0]
        sl = [slice(None)] * b.ndim; sl[axis] = idx
        b[tuple(sl)] = np.take(a, sel[found], axis=axis); out[k] = b
    print(f"[warn] {name}: cache {len(cache_gid)} vs sample {len(cur_gid)} "
          f"({int(found.sum())} matched, {int((~found).sum())} -> NaN).")
    return out, int(found.sum())


In [ ]:
# ── reduced-aperture particle machinery (loaders + flexible-binning Σ); used by 6g·matched, 6k, 6l ──
# Defined here in Part 2 so the corrected 100 kpc ISM+CGM Σ is available to the event-aligned
# analysis (6g·matched) as well as the distribution renders (6l). All functions resolve RESULTS /
# _diag_at / build_prog_index at call time, so they are safe to define before those exist.
SIGMA_INFO = [("Sigma_H2_eff", r"$\Sigma_{\rm H_2}$"), ("Sigma_HI_eff", r"$\Sigma_{\rm HI}$"),
              ("Sigma_gas_eff", r"$\Sigma_{\rm gas}$"), ("Sigma_dust_eff", r"$\Sigma_{\rm dust}$")]

REDUCED_PREFIX = sim.file_format.split("_{")[0]       # e.g. m100n1024 / m50n512 — must match the SLURM job env
REDUCED_DIR    = os.path.join(OUT, "reduced_particles")
_COMP_DS = {"gas": "m_gas", "dust": "m_dust", "HI": "m_HI", "H2": "m_H2"}
_COMP_KEY = {"H2": "Sigma_H2_eff", "HI": "Sigma_HI_eff", "gas": "Sigma_gas_eff", "dust": "Sigma_dust_eff"}
_PIDX_CACHE = {}

def _pidx(z):
    if z not in _PIDX_CACHE:
        B = RESULTS[z]; _PIDX_CACHE[z] = build_prog_index(B["A"], B["galaxy_ids"], B["snaps_arr"])
    return _PIDX_CACHE[z]

def reduced_path(snap, gx):
    return os.path.join(REDUCED_DIR, f"snap_{int(snap):03d}",
                        f"{REDUCED_PREFIX}_snap{int(snap):03d}_gal{int(gx):06d}.h5")

def load_reduced(snap, gx):
    p = reduced_path(snap, gx)
    if not os.path.exists(p):
        return None
    out = {}
    with h5py.File(p, "r") as f:
        out.update({k: f.attrs[k] for k in f.attrs})
        for grp in ("gas", "star"):
            if grp in f:
                out[grp] = {k: f[grp][k][:] for k in f[grp].keys()}
    return out

def _reduced_R(pos, evecs, geom):
    if geom == "3d":
        return np.sqrt(np.sum(np.asarray(pos, float) ** 2, axis=1))
    proj = np.asarray(pos, float) @ np.asarray(evecs, float).T   # onto stellar principal axes
    return np.sqrt(proj[:, 0] ** 2 + proj[:, 1] ** 2)            # face-on cylindrical R

def _reduced_sel(red, comp, geom, region):
    g = red.get("gas")
    if g is None or _COMP_DS[comp] not in g:
        return None, None, None
    R = _reduced_R(g["pos"], red["evecs"], geom)
    m = np.asarray(g[_COMP_DS[comp]], float)
    sfr = np.asarray(g.get("sfr", np.zeros(len(m))), float)
    reg = np.ones(len(m), bool)
    if region == "ism":
        reg = sfr > 0
    elif region == "cgm":
        reg = sfr == 0
    return R, m, reg

def reduced_sigma_scalar(red, comp="H2", r_aper_kpc=10.0, geom="cyl", region="all"):
    '''Mean Σ of `comp` inside a circular/spherical aperture [Msun/kpc^2]; NaN if no particles.'''
    if red is None:
        return np.nan
    R, m, reg = _reduced_sel(red, comp, geom, region)
    if R is None:
        return np.nan
    sel = reg & (R <= r_aper_kpc)
    if not np.any(sel):
        return 0.0 if np.any(reg) else np.nan       # 0 = genuinely gas-empty aperture; NaN = no data
    return float(np.nansum(m[sel]) / (np.pi * r_aper_kpc ** 2))

def reduced_sigma_profile(red, comp="gas", edges=None, geom="cyl", region="all"):
    '''Σ(R) of `comp` on arbitrary bin edges [kpc]; returns (rmid, Sigma).'''
    if edges is None:
        edges = np.linspace(0.0, float(red.get("rmax_kpc", 100.0)), 26)
    R, m, reg = _reduced_sel(red, comp, geom, region)
    rmid = 0.5 * (edges[:-1] + edges[1:])
    if R is None:
        return rmid, np.full(len(rmid), np.nan)
    s, _ = np.histogram(R[reg], bins=edges, weights=np.nan_to_num(m[reg]))
    return rmid, s / (np.pi * (edges[1:] ** 2 - edges[:-1] ** 2))

T_HOT, T_COLD = 1.0e5, 1.0e4                            # phase cuts [K] (match build_profiles_job)
_PHASES = {"cold": (0.0, T_COLD), "warm": (T_COLD, T_HOT), "hot": (T_HOT, np.inf)}

def reduced_phase_profile(red, phase="cold", edges=None, geom="cyl"):
    """Σ(R) [Msun/kpc^2] of GAS in a temperature phase (cold/warm/hot); NaN if the file has no temp."""
    if edges is None:
        edges = np.linspace(0.0, float(red.get("rmax_kpc", 100.0)), 26) if red else np.linspace(0, 100, 26)
    rmid = 0.5 * (edges[:-1] + edges[1:])
    g = red.get("gas") if red else None
    if g is None or "temp" not in g or _COMP_DS["gas"] not in g:
        return rmid, np.full(len(rmid), np.nan)
    R = _reduced_R(g["pos"], red["evecs"], geom)
    m = np.asarray(g[_COMP_DS["gas"]], float); T = np.asarray(g["temp"], float)
    lo, hi = _PHASES[phase]; sel = np.isfinite(T) & (T >= lo) & (T < hi)
    s, _ = np.histogram(R[sel], bins=edges, weights=np.nan_to_num(m[sel]))
    return rmid, s / (np.pi * (edges[1:] ** 2 - edges[:-1] ** 2))

def reduced_temp_profile(red, edges=None, geom="cyl", region="all"):
    """Mass-weighted gas temperature T(R) [K]; NaN where empty or no temp."""
    if edges is None:
        edges = np.linspace(0.0, float(red.get("rmax_kpc", 100.0)), 26) if red else np.linspace(0, 100, 26)
    rmid = 0.5 * (edges[:-1] + edges[1:])
    g = red.get("gas") if red else None
    if g is None or "temp" not in g:
        return rmid, np.full(len(rmid), np.nan)
    R, m, reg = _reduced_sel(red, "gas", geom, region)
    if R is None:
        return rmid, np.full(len(rmid), np.nan)
    T = np.asarray(g["temp"], float)
    num, _ = np.histogram(R[reg], bins=edges, weights=np.nan_to_num(m[reg] * T[reg]))
    den, _ = np.histogram(R[reg], bins=edges, weights=np.nan_to_num(m[reg]))
    with np.errstate(all="ignore"):
        return rmid, np.where(den > 0, num / den, np.nan)

def reduced_star_sigma_profile(red, edges=None, geom="cyl"):
    """Σ_*(R) [Msun/kpc^2] from the reduced STAR particles (red['star']); NaN if none."""
    if edges is None:
        edges = np.linspace(0.0, float(red.get("rmax_kpc", 100.0)), 26) if red else np.linspace(0, 100, 26)
    rmid = 0.5 * (edges[:-1] + edges[1:])
    s = red.get("star") if red else None
    if not s or "pos" not in s or "m_star" not in s:
        return rmid, np.full(len(rmid), np.nan)
    R = _reduced_R(s["pos"], red["evecs"], geom)
    m = np.asarray(s["m_star"], float)
    h, _ = np.histogram(R, bins=edges, weights=np.nan_to_num(m))
    return rmid, h / (np.pi * (edges[1:] ** 2 - edges[:-1] ** 2))

def _reduced_r50_star(red, geom="cyl"):
    """Stellar half-mass radius [kpc] from the reduced star particles (NaN if none)."""
    s = red.get("star")
    if not s or "pos" not in s or "m_star" not in s:
        return np.nan
    R = _reduced_R(s["pos"], red["evecs"], geom); m = np.asarray(s["m_star"], float)
    o = np.argsort(R); cm = np.cumsum(m[o])
    if cm[-1] <= 0:
        return np.nan
    return float(R[o][min(int(np.searchsorted(cm, 0.5 * cm[-1])), len(R) - 1)])

def sigma_table_reduced(stage="post_quench", r_aper_kpc=10.0, geom="cyl", region="all",
                        aper="fixed", aper_factor=1.0):
    '''Pooled Σ table recomputed from reduced files at a chosen aperture/geometry/region.
    Rows are pooled over RESULTS with each anchor's well_behaved mask — SAME order/length as POOL,
    so columns align 1:1 with POOL (fgas, dusty, xstr_post, is_fast/slow).'''
    cols = {key: [] for key, _ in SIGMA_INFO}
    for k in ("age", "logM", "is_fast", "is_slow", "strong", "weak"):
        cols[k] = []
    n_have = n_miss = 0
    for z in RESULTS:
        B = RESULTS[z]; wb = B["well_behaved"]; recs = B["records"]; snaps = B["snaps_arr"]; PIDX = _pidx(z)
        sig = {key: np.full(len(recs), np.nan) for key, _ in SIGMA_INFO}
        for i, r in enumerate(recs):
            row = r.get(f"row_{stage}", -1); col = r["col"]
            if row < 0:
                continue
            gx = PIDX[row, col]
            if not np.isfinite(gx):
                continue
            red = load_reduced(int(snaps[row]), int(gx))
            if red is None:
                n_miss += 1; continue
            if aper == "R50star":                       # size-relative aperture (per-galaxy R50*)
                r50 = _reduced_r50_star(red, geom)
                ap = r50 * aper_factor if (np.isfinite(r50) and r50 > 0) else np.nan
            else:
                ap = r_aper_kpc                          # fixed physical aperture [kpc]
            if not np.isfinite(ap):
                n_miss += 1; continue
            n_have += 1
            for comp, key in _COMP_KEY.items():
                sig[key][i] = reduced_sigma_scalar(red, comp, ap, geom, region)
        for key, _ in SIGMA_INFO:
            cols[key].append(sig[key][wb])
        with np.errstate(all="ignore"):
            msv = _diag_at(B, "Mstar", stage)
            cols["age"].append(_diag_at(B, "age", stage)[wb]); cols["logM"].append(np.log10(np.where(msv > 0, msv, np.nan))[wb])
        cols["is_fast"].append(B["is_fast"][wb]); cols["is_slow"].append(B["is_slow"][wb])
        cols["strong"].append(B["CO"]["strong"][wb]); cols["weak"].append(B["CO"]["weak"][wb])
    T = {k: (np.concatenate(v) if v else np.array([])) for k, v in cols.items()}
    apdesc = (f"{r_aper_kpc:g} kpc" if aper == "fixed" else f"{aper_factor:g}xR50*")
    print(f"reduced Σ table @ {stage} (aper={apdesc}, {geom}, {region}): "
          f"{n_have} galaxies loaded, {n_miss} skipped")
    return T, n_have

### Part 2·Σ-cache — log-binned Σ profiles (gas / H2 / HI / dust / star) for the whole analysis

Precompute the **log-spaced** radial surface-density profiles of every reduced 100 kpc file once and
store them in a **config-tagged** file `tables/reduced_profiles_cache_<binconfig>.hdf5` (keyed by
`(snap, gx)`, shared bin edges). These reduced-file profiles are the **single source** for every
downstream Σ analysis (the `RP` scalar layer below replaces the old dust-profile products). The build
is incremental (skips already-cached galaxies).

**Adding `star`/`temp` needs a one-time `REBUILD_PROFILE_CACHE=True`** — the filename tag does not
encode the component list, so an incremental build will not backfill new components into existing
keys. Re-extract the reduced files with `temp` to also cache cold/warm/hot Σ and the mass-weighted
T(R).

In [ ]:
# ── Part 2·cache — log-binned Σ profiles (gas/H2/HI/dust/star) per reduced galaxy, saved to HDF5 ──
# The reduced 100 kpc files are the SINGLE profile source. Adding "star"/"temp" needs a one-time
# REBUILD_PROFILE_CACHE=True (incremental build never backfills new comps into existing keys).
PROF_COMPS = ["gas", "H2", "HI", "dust", "star"]
PROF_RVIEW, PROF_NBINS, PROF_RMIN = 50.0, 20, 0.3        # view radius [kpc], n log bins, inner edge [kpc]
PROF_GEOM, PROF_REGION = "cyl", "all"
PROF_EDGES = np.concatenate([[0.0], np.logspace(np.log10(PROF_RMIN), np.log10(PROF_RVIEW), PROF_NBINS)])
PROF_RMID  = 0.5 * (PROF_EDGES[:-1] + PROF_EDGES[1:])
_CACHE_NBAD = 0

def _cache_path():
    """Cache filename encodes the bin config, so a different config writes a NEW file (never
    overwrites another config's cache)."""
    tag = f"n{PROF_NBINS}_r{PROF_RMIN:g}-{PROF_RVIEW:g}_{PROF_GEOM}_{PROF_REGION}"
    return os.path.join(TABLEDIR, f"reduced_profiles_cache_{tag}.hdf5")

PROF_CACHE_FILE = _cache_path()

def _parse_reduced_name(base):                          # "<prefix>_snapNNN_galGGGGGG.h5" -> (snap, gx)
    try:
        return int(base.split("_snap")[1].split("_")[0]), int(base.split("_gal")[1].split(".")[0])
    except (IndexError, ValueError):
        return None

def build_profile_cache(rebuild=False):
    """Walk the reduced files; cache each galaxy's log-binned Σ(gas/H2/HI/dust) into the config-tagged
    file. Incremental (skips already-cached); rebuild=True recomputes this config's file from scratch."""
    if not os.path.isdir(REDUCED_DIR):
        print(f"[cache] no reduced dir {REDUCED_DIR}"); return
    path = _cache_path()
    files = [os.path.join(d, fn) for d, _s, fs in os.walk(REDUCED_DIR) for fn in fs if fn.endswith(".h5")]
    have = set()
    if os.path.exists(path) and not rebuild:
        with h5py.File(path, "r") as cf:
            have = set(cf.keys())
    todo = []
    for p in files:
        sg = _parse_reduced_name(os.path.basename(p))
        if sg is not None and f"s{sg[0]:03d}_g{sg[1]:06d}" not in have:
            todo.append(p)
    if not rebuild and not todo:                        # nothing new -> do not touch the file
        print(f"[cache] up to date ({len(have) - ('edges' in have)} cached) -> {os.path.basename(path)}"); return
    n_new = 0
    work = files if rebuild else todo
    with h5py.File(path, "w" if rebuild else "a") as cf:
        cf.attrs.update(dict(r_view=PROF_RVIEW, nbins=PROF_NBINS, r_min=PROF_RMIN,
                             geom=PROF_GEOM, region=PROF_REGION, comps=",".join(PROF_COMPS)))
        if "edges" not in cf:
            cf.create_dataset("edges", data=PROF_EDGES)
        for p in _pbar(work, "caching profiles"):
            sg = _parse_reduced_name(os.path.basename(p))
            if sg is None:
                continue
            snap, gx = sg; key = f"s{snap:03d}_g{gx:06d}"
            if key in cf and not rebuild:
                continue
            red = load_reduced(snap, gx)
            if red is None or "gas" not in red:
                continue
            grp = cf.require_group(key)
            for comp in PROF_COMPS:
                if comp == "star":
                    _, sig = reduced_star_sigma_profile(red, edges=PROF_EDGES, geom=PROF_GEOM)
                else:
                    _, sig = reduced_sigma_profile(red, comp=comp, edges=PROF_EDGES, geom=PROF_GEOM, region=PROF_REGION)
                if comp in grp:
                    del grp[comp]
                grp.create_dataset(comp, data=sig.astype("f4"))
            if "temp" in red.get("gas", {}):            # phase Σ + mass-weighted T(R), if temperature stored
                for ph in ("cold", "warm", "hot"):
                    _, sg = reduced_phase_profile(red, ph, edges=PROF_EDGES, geom=PROF_GEOM)
                    if f"phase_{ph}" in grp:
                        del grp[f"phase_{ph}"]
                    grp.create_dataset(f"phase_{ph}", data=sg.astype("f4"))
                _, Tr = reduced_temp_profile(red, edges=PROF_EDGES, geom=PROF_GEOM, region=PROF_REGION)
                if "Tprof" in grp:
                    del grp["Tprof"]
                grp.create_dataset("Tprof", data=Tr.astype("f4"))
            n_new += 1
    print(f"[cache] {n_new} new profiles -> {os.path.basename(path)} ({len(files)} reduced files seen)")

def load_profile_cache():
    """Return {(snap,gx): {comp: Σ(R)}} from THIS config's cache file (length-checked for safety)."""
    global _CACHE_NBAD
    path = _cache_path(); out = {}; _CACHE_NBAD = 0
    if not os.path.exists(path):
        print(f"[cache] none yet for this config ({os.path.basename(path)}) — run build_profile_cache()."); return out
    with h5py.File(path, "r") as cf:
        for key in cf.keys():
            if key == "edges":
                continue
            d = {c: cf[key][c][:] for c in cf[key].keys()}
            if any(np.ndim(v) != 1 or len(v) != PROF_NBINS for v in d.values()):
                _CACHE_NBAD += 1; continue
            out[(int(key[1:4]), int(key[6:]))] = d
    if _CACHE_NBAD:
        print(f"[cache] {_CACHE_NBAD} stale entries ignored in {os.path.basename(path)} (rebuild=True to clean).")
    return out

if BUILD_PROFILE_CACHE or REBUILD_PROFILE_CACHE:
    build_profile_cache(rebuild=REBUILD_PROFILE_CACHE)
PROFILE_CACHE = load_profile_cache()
if not PROFILE_CACHE:
    print("[cache] empty/not built — set BUILD_PROFILE_CACHE=True (Part 0) to build it from the reduced files.")
print(f"[cache] {len(PROFILE_CACHE)} galaxies in memory from {os.path.basename(PROF_CACHE_FILE)}; "
      f"bins {PROF_NBINS} log [{PROF_RMIN},{PROF_RVIEW}] kpc.")

In [ ]:
# ── Part 2·RP — reduced-profile scalars: the single profile source (replaces the old DP/SIG) ──
# Enclosed-fraction radii + mean-Σ from a cached log-binned Σ(R), then a per-(record,stage) scalar
# table so every consumer reads the 100 kpc reduced files (via PROFILE_CACHE). Empty annulus = Σ0
# (kept); galaxy absent from the cache = NaN — same semantics as the retired dust-profile product.
RP_COMPS   = ("gas", "H2", "HI", "dust", "star")
_RP_SIGKEY = {"gas": "Sigma_gas_eff", "H2": "Sigma_H2_eff", "HI": "Sigma_HI_eff", "dust": "Sigma_dust_eff"}

def _enclosed_radii(prof_lin, fracs=(0.2, 0.5, 0.8, 0.9)):
    """Radii enclosing `fracs` of the component mass from a cached log-binned Σ(R)."""
    if prof_lin is None:
        return (np.nan,) * len(fracs)
    area = np.pi * (PROF_EDGES[1:] ** 2 - PROF_EDGES[:-1] ** 2)
    M = np.nancumsum(np.nan_to_num(np.asarray(prof_lin, float) * area)); Mtot = float(M[-1])
    if not np.isfinite(Mtot) or Mtot <= 0:
        return (np.nan,) * len(fracs)
    return tuple(float(np.interp(f * Mtot, M, PROF_EDGES[1:])) for f in fracs)

def _mean_sigma_inside(prof_lin, R):
    """Mean Σ over bins with PROF_RMID <= R (old 'Sigma_eff' semantics). NaN if R invalid/empty."""
    if prof_lin is None or not np.isfinite(R) or R <= 0:
        return np.nan
    inb = PROF_RMID <= R
    if not inb.any():
        return np.nan
    v = np.asarray(prof_lin, float)[inb]
    return float(np.nanmean(v)) if np.isfinite(v).any() else np.nan

def reduced_profile_scalars(records, STAGES, PIDX, snaps_arr, cache=None):
    """Per-(record,stage) scalars from the cached Σ(R) of the reduced 100 kpc files. Drop-in for the
    retired load_profiles+sigma_scalars. Returns a dict of (nrec,nst) arrays + STAGES/RMID/OK.
    All-NaN (OK=False) when PIDX is None or the cache is empty — so runs without reduced files still
    complete, exactly like the old DP_OK=False path."""
    cache = PROFILE_CACHE if cache is None else cache
    nrec, nst = len(records), len(STAGES)
    RP = {f"{q}_{c}": np.full((nrec, nst), np.nan) for c in RP_COMPS for q in ("R20", "R50", "R80", "R90")}
    RP.update({k: np.full((nrec, nst), np.nan) for k in _RP_SIGKEY.values()})
    RP["RATIO"] = np.full((nrec, nst), np.nan)                    # R50_dust / R50_star (old DP_RATIO)
    RP["C2080"] = {c: np.full((nrec, nst), np.nan) for c in RP_COMPS}
    if PIDX is not None and cache:
        for i, r in enumerate(records):
            col = r["col"]
            for si, st in enumerate(STAGES):
                row = r.get(f"row_{st}", -1)
                if not (isinstance(row, (int, np.integer)) and row >= 0):
                    continue
                g = PIDX[row, col]
                if not np.isfinite(g):
                    continue
                d = cache.get((int(snaps_arr[row]), int(g)))
                if not d:
                    continue
                for c in RP_COMPS:
                    r20, r50, r80, r90 = _enclosed_radii(d.get(c))
                    RP[f"R20_{c}"][i, si] = r20; RP[f"R50_{c}"][i, si] = r50
                    RP[f"R80_{c}"][i, si] = r80; RP[f"R90_{c}"][i, si] = r90
                    if np.isfinite(r80) and r80 > 0:
                        RP["C2080"][c][i, si] = r20 / r80
                rstar = RP["R50_star"][i, si]
                for c, key in _RP_SIGKEY.items():
                    RP[key][i, si] = _mean_sigma_inside(d.get(c), rstar)
                rdust = RP["R50_dust"][i, si]
                if np.isfinite(rstar) and rstar > 0:
                    RP["RATIO"][i, si] = rdust / rstar
    RP["STAGES"] = list(STAGES); RP["RMID"] = PROF_RMID
    RP["OK"] = bool(np.isfinite(RP["Sigma_H2_eff"]).any())
    return RP

In [ ]:
# ── §4 DIAGS + §7 plotters + §8g clock engine (operate on bound globals; see Part 3) ──
# ── canonical catalog-property definitions (single source of truth) ──────────────
# name -> (build(P) -> (n_snap × n_gal) array | None, logy, label). Every catalog-derived
# scalar the notebook stacks / bins / samples is defined ONCE here; the four samplers
# (build_diags, build_registry, _build_dms_tracks, _sfh_track) all read from this table,
# so adding a quantity is a one-line edit that propagates everywhere.
def _pratio(P, num, den):
    A, B = P.get(num), P.get(den)
    if A is None or B is None:
        return None
    with np.errstate(all="ignore"):
        return np.where(B > 0, A / np.where(B > 0, B, np.nan), np.nan)

def _oh_arr(P):
    """Gas-phase 12+log(O/H) from mass-weighted metallicity (Z_sun=0.0134, (O/H)_sun=8.69)."""
    Zg = P.get("metallicities.mass_weighted")
    if Zg is None:
        return None
    with np.errstate(all="ignore"):
        return np.where(Zg > 0, 8.69 + np.log10(np.where(Zg > 0, Zg, np.nan) / 0.0134), np.nan)

def _devis_dtm(P):
    """Dust-to-metal ratio, De Vis+19: M_dust / (f_z·M_H2 + M_dust), f_z = 27.36·10^(O/H−12)."""
    md, mh2, oh = P.get("masses.dust"), P.get("masses.H2"), _oh_arr(P)
    if md is None or mh2 is None or oh is None:
        return None
    with np.errstate(all="ignore"):
        mz = 27.36 * 10.0 ** (oh - 12.0) * mh2 + md
        return np.where(mz > 0, md / mz, np.nan)

CATALOG_PROPS = {
    "SFR":         (lambda P: P.get("sfr"),                                                 True,  "SFR"),
    "sSFR":        (lambda P: _pratio(P, "sfr", "masses.stellar"),                          True,  "sSFR"),
    "M_halo":      (lambda P: P.get("masses.total"),                                        True,  r"$M_{\rm halo}$"),
    "M_star":      (lambda P: P.get("masses.stellar"),                                      True,  r"$M_*$"),
    "M_gas":       (lambda P: P.get("masses.gas"),                                          True,  r"$M_{\rm gas}$"),
    "M_dust":      (lambda P: P.get("masses.dust"),                                         True,  r"$M_{\rm dust}$"),
    "M_H2":        (lambda P: P.get("masses.H2"),                                           True,  r"$M_{\rm H_2}$"),
    "M_HI":        (lambda P: P.get("masses.HI"),                                           True,  r"$M_{\rm HI}$"),
    "dust/M*":     (lambda P: _pratio(P, "masses.dust", "masses.stellar"),                  True,  r"$M_{\rm dust}/M_*$"),
    "H2/M*":       (lambda P: _pratio(P, "masses.H2", "masses.stellar"),                    True,  r"$M_{\rm H_2}/M_*$"),
    "HI/M*":       (lambda P: _pratio(P, "masses.HI", "masses.stellar"),                    True,  r"$M_{\rm HI}/M_*$"),
    "gas/M*":      (lambda P: _pratio(P, "masses.gas", "masses.stellar"),                   True,  r"$M_{\rm gas}/M_\star$"),
    "dust/gas":    (lambda P: _pratio(P, "masses.dust", "masses.gas"),                      True,  r"$M_{\rm dust}/M_{\rm gas}$"),
    "Rgas/Rstar":  (lambda P: _pratio(P, "radii.gas_half_mass", "radii.stellar_half_mass"), False, r"$R_{\rm gas}/R_*$"),
    "R20g/R80g":   (lambda P: _pratio(P, "radii.gas_r20", "radii.gas_r80"),                 True,  r"$R_{20}/R_{80}$ gas"),
    "R20s/R80s":   (lambda P: _pratio(P, "radii.stellar_r20", "radii.stellar_r80"),         True,  r"$R_{20}/R_{80}$ star"),
    "Age_s":       (lambda P: P.get("ages.mass_weighted"),                                  True,  r"Age$_\star$"),
    "kappa_gas":   (lambda P: P.get("rotation.gas_kappa_rot"),                              False, r"$\kappa_{\rm rot}^{\rm gas}$"),
    "kappa_star":  (lambda P: P.get("rotation.stellar_kappa_rot"),                          False, r"$\kappa_{\rm rot}^{\star}$"),
    "BoverT_gas":  (lambda P: P.get("rotation.gas_BoverT"),                                 False, r"$(B/T)_{\rm gas}$"),
    "BoverT_star": (lambda P: P.get("rotation.stellar_BoverT"),                             False, r"$(B/T)_{\star}$"),
    "oh":          (lambda P: _oh_arr(P),                                                   False, r"12+log(O/H)"),
    "dtm":         (lambda P: _devis_dtm(P),                                                False, r"$M_{\rm dust}/M_Z$ (De Vis+19)"),
}
# the catalog scalars registered as full-history tracks in build_registry (BH / coupling /
# profile / reduced-Σ quantities are handled separately, further down):
_REG_CATALOG_PROPS = ["sSFR", "SFR", "M_halo", "M_star", "M_gas", "M_dust", "M_H2", "M_HI",
                      "dust/M*", "H2/M*", "HI/M*", "dust/gas", "Rgas/Rstar", "R20g/R80g",
                      "R20s/R80s", "Age_s", "kappa_gas", "kappa_star", "BoverT_gas",
                      "BoverT_star", "dtm"]

def build_diags(records, P, cols, STAGES):
    """diags[stage][quantity] -> per-record array (NaN if missing). Samples CATALOG_PROPS at
    each record's stage row; local keys kept for back-compat with the _diag_at / POOL consumers."""
    _DIAG_MAP = {"Mdust": "M_dust", "MH2": "M_H2", "MHI": "M_HI", "Mstar": "M_star",
                 "Mgas": "M_gas", "age": "Age_s", "sSFR": "sSFR", "dust/M*": "dust/M*",
                 "H2/M*": "H2/M*", "HI/M*": "HI/M*", "dust/gas": "dust/gas",
                 "Rgas/Rstar": "Rgas/Rstar", "C20/80*": "R20s/R80s", "C20/80gas": "R20g/R80g"}
    arrs = {k: CATALOG_PROPS[cn][0](P) for k, cn in _DIAG_MAP.items()}
    def sample(arr, st):
        out = np.full(len(records), np.nan)
        if arr is None:
            return out
        rows = np.array([r[f"row_{st}"] for r in records])
        for i, (row, col) in enumerate(zip(rows, cols)):
            if row >= 0:
                out[i] = arr[row, col]
        return out
    return {st: {k: sample(arrs[k], st) for k in _DIAG_MAP} for st in STAGES}

# These plot engines read module globals (records, STAGES, P, t_cosmic_yr, is_fast, is_slow,
# mbin, NBINS, MASS_BIN_LABELS, COL_A/COL_B/LBL_A/LBL_B, DIAGS, DP*) — bound per anchor by
# activate_anchor() in Part 3.
def violin_stage_g(diags, qty, binarr, binlabels, binname="logM*", logy=True, ylabel=None, fname=None):
    nbn = len(binlabels)
    fig, axs = plt.subplots(1, nbn, figsize=(5.0 * nbn, 4.5), sharey=True); axs = np.atleast_1d(axs)
    for bi in range(nbn):
        ax = axs[bi]; binmask = binarr == bi
        for j, st in enumerate(STAGES):
            for off, msk, c in [(-0.18, is_fast & binmask, COL_A), (0.18, is_slow & binmask, COL_B)]:
                v = diags[st][qty][msk]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    continue
                v = np.log10(v) if logy else v
                parts = ax.violinplot([v], positions=[j + off], widths=0.32, showmedians=True)
                for b in parts["bodies"]:
                    b.set_facecolor(c); b.set_alpha(0.6)
                for key in ("cmedians", "cbars", "cmins", "cmaxes"):
                    if key in parts:
                        parts[key].set_color(c)
        ax.set_xticks(range(len(STAGES))); ax.set_xticklabels(STAGES, rotation=35, ha="right")
        ax.set_title(f"{binname} {binlabels[bi]} (n={binmask.sum()})")
    axs[0].set_ylabel(ylabel or (f"log10 {qty}" if logy else qty))
    fig.suptitle(f"{qty} across stages  ({LBL_A} vs {LBL_B})", y=1.02); plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

# ── §8c/§8e event-stack engine (the §8 workhorse): property tracks aligned on triggers ──
# build_registry(B) constructs the per-anchor PROP_REGISTRY + TRIGGERS (fn closures are
# self-contained — they capture that anchor's arrays/records/_ord). _estack/event_stack read
# the globals bound by activate_anchor (t_inc, PROP_REGISTRY, TRIGGERS, is_fast/is_slow, GYR...).
def build_registry(B):
    P = B["P"]; BH = B["BH"]; records = B["records"]; t_ = B["t_cosmic_yr"]; n_snap = B["n_snap"]
    CO = B["CO"]; rp = B["RP"]
    _ord = np.argsort(t_); t_inc = t_[_ord]
    with np.errstate(all="ignore"):
        ssfr_hist = np.where(P["masses.stellar"] > 0, P["sfr"] / P["masses.stellar"], np.nan)
    REG = {}
    def _track(a, col):
        return a[_ord, col].astype(float)
    def _ratio(a, b):
        A, Bd = P.get(a), P.get(b)
        if A is None or Bd is None:
            return None
        with np.errstate(all="ignore"):
            return np.where(Bd > 0, A / np.where(Bd > 0, Bd, np.nan), np.nan)
    def reg(name, arr, logy, label, by="gal"):
        if arr is None:
            return
        fn = (lambda i, a=arr: _track(a, records[i]["col"])) if by == "gal" else (lambda i, a=arr: _track(a, i))
        REG[name] = dict(fn=fn, logy=logy, label=label)
    def reg_stage(name, A, SL, logy, label):
        if A is None or SL is None:
            return
        def fn(i, A=A, SL=SL):
            full = np.full(n_snap, np.nan)
            for si, st in enumerate(SL):
                if si >= A.shape[1]:
                    break
                row = records[i].get(f"row_{st}", -1)
                if isinstance(row, (int, np.integer)) and row >= 0:
                    full[row] = A[i, si]
            return full[_ord]
        REG[name] = dict(fn=fn, logy=logy, label=label, sparse=True)
    if BH is not None:
        reg("BHAR", BH["bh_mdot"], True, r"$\dot M_{\rm BH}$"); reg("f_Edd", BH["bh_fedd"], True, r"$f_{\rm Edd}$")
        reg("M_BH", BH["bh_mass"], True, r"$M_{\rm BH}$")
        reg("jet_strength", CO.get("wjet_hist"), False, r"$w_{\rm jet}$")
        reg("xcoup", CO.get("xcoup_hist"), False, r"$x_{\rm coup}$")
        reg("f_gas", CO.get("fgas_hist"), False, r"$f_{\rm gas}$")
    else:
        reg("f_gas", _ratio("masses.gas", "masses.stellar"), False, r"$f_{\rm gas}$")
    for _pn in _REG_CATALOG_PROPS:                   # catalog scalars, from the CATALOG_PROPS table
        _pf, _ply, _plb = CATALOG_PROPS[_pn]
        reg(_pn, _pf(P), _ply, _plb)

    # dense (every-snapshot) catalog Σ within R20/R50/R80 — coverage-matched to the M_* tracks.
    # gas is exact (gas R_x enclose f_x of the gas mass); H2 uses the same enclosed fraction (H2~gas).
    _mh2, _mgas = P.get("masses.H2"), P.get("masses.gas")
    for _f, _rk, _tag in ((0.2, "radii.gas_r20", "20"), (0.5, "radii.gas_half_mass", "50"), (0.8, "radii.gas_r80", "80")):
        _Rr = P.get(_rk)
        if _Rr is None:
            continue
        with np.errstate(all="ignore"):
            _den = np.pi * np.where(_Rr > 0, _Rr, np.nan) ** 2
            if _mh2 is not None:
                reg(f"Sigma_H2_R{_tag}", _f * _mh2 / _den, True, fr"$\Sigma_{{\rm H_2}}(<R_{{{_tag}}})$")
            if _mgas is not None:
                reg(f"Sigma_gas_R{_tag}", _f * _mgas / _den, True, fr"$\Sigma_{{\rm gas}}(<R_{{{_tag}}})$")
    if rp["OK"]:
        SL = rp["STAGES"]
        reg_stage("Rdust/Rstar", rp["RATIO"], SL, False, r"$R_{50}^{\rm dust}/R_{50}^{\star}$")
        reg_stage("Sigma_H2", rp["Sigma_H2_eff"], SL, True, r"$\Sigma_{\rm H_2}$")
        reg_stage("Sigma_HI", rp["Sigma_HI_eff"], SL, True, r"$\Sigma_{\rm HI}$")
        reg_stage("Sigma_gas", rp["Sigma_gas_eff"], SL, True, r"$\Sigma_{\rm gas}$")
        reg_stage("Sigma_dust", rp["Sigma_dust_eff"], SL, True, r"$\Sigma_{\rm dust}$")
        # profile R20/R80 concentration per component (OUR radii; only at critical-point stages).
        # H2/HI compactness is available ONLY here (the catalog has gas & stellar radii only).
        for _c, _lab in (("gas", "gas"), ("H2", r"H$_2$"), ("HI", "HI"), ("star", "star"), ("dust", "dust")):
            if _c in rp["C2080"]:
                reg_stage(f"C2080_{_c}_prof", rp["C2080"][_c], SL, True, fr"$R_{{20}}/R_{{80}}$ {_lab} (profile)")
    def _ta(key):
        return np.array([r.get(key, np.nan) for r in records], float)
    TRIG = {"t_AGN": np.asarray(B["t_agn"], float), "t_QT": _ta("t_qt"), "t_SFT": _ta("t_sft"),
            "t_SFpeak": _ta("t_sf_peak"), "t_PostQuench": _ta("t_post_quench"),
            "post_quench": _ta("t_post_quench"), "t_GasMin": _ta("t_gas_min"),
            "gas_min": _ta("t_gas_min"), "t_formation": _ta("t_formation")}
    return dict(_ord=_ord, t_inc=t_inc, ssfr_hist=ssfr_hist, PROP_REGISTRY=REG, TRIGGERS=TRIG)

def _estack(ax, zero_t, arr_of, ylab, logy=True, xwin=2.0, nb=12, band="iqr", ref0=False, split=None,
            show_detect=False):
    """Median (+IQR band) of arr_of(i) around per-record zero_t, per group (default fast/slow).
    ref0=True subtracts each galaxy's own value at the trigger (composition-immune). Verbatim §8c.

    show_detect=True adds, on a right axis, the % of the WHOLE group that is validly used in each
    bin (finite, and >0 for log quantities, since the log-median drops zeros) — drawn as open
    markers of a distinct shape per group. A bin whose marker sits low is a median over a small,
    selected subset, so a strong-vs-weak gap that holds only where the valid-% is low, or where the
    two groups' valid-% differ a lot, is survivorship rather than a real amplitude difference. The
    right axis is created only when there is something to show."""
    xe = np.linspace(-xwin, xwin, nb + 1); xc = 0.5 * (xe[:-1] + xe[1:])
    groups = split if split is not None else [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]
    det_markers = ["s", "^", "D", "v", "P", "X"]          # distinct from the round value markers
    axd = None
    for gi, (msk, c, lab) in enumerate(groups):
        ii = np.where(msk & np.isfinite(zero_t))[0]
        prof = np.full((len(ii), len(xc)), np.nan)
        for k, i in enumerate(ii):
            tt = (t_inc - zero_t[i]) / GYR; raw = arr_of(i)
            with np.errstate(divide="ignore", invalid="ignore"):
                vv = np.where(raw > 0, np.log10(raw), np.nan) if logy else raw
            for b in range(len(xc)):
                sel = (tt >= xe[b]) & (tt < xe[b + 1]) & np.isfinite(vv)
                if sel.any():
                    prof[k, b] = np.nanmedian(vv[sel])
        if not len(ii):
            continue
        if ref0:
            b0 = int(np.clip(np.searchsorted(xe, 0.0) - 1, 0, len(xc) - 1))
            for k in range(prof.shape[0]):
                row = prof[k]; fin = np.where(np.isfinite(row))[0]
                if not len(fin):
                    continue
                bref = b0 if np.isfinite(row[b0]) else fin[np.argmin(np.abs(fin - b0))]
                prof[k] = row - row[bref]
        with np.errstate(all="ignore"):
            med = np.nanmedian(prof, axis=0); nbin = np.sum(np.isfinite(prof), axis=0)
            if band == "iqr":
                lo = np.nanpercentile(prof, 25, axis=0); hi = np.nanpercentile(prof, 75, axis=0)
            else:
                lo = hi = None
        med = np.where(nbin >= 3, med, np.nan)
        ax.plot(xc, med, "-o", color=c, ms=7, label=f"{lab} (n={len(ii)})", zorder=3)
        if lo is not None:
            ax.fill_between(xc, np.where(nbin >= 3, lo, np.nan), np.where(nbin >= 3, hi, np.nan),
                            color=c, alpha=0.15, lw=0)
        if show_detect:
            frac_pct = 100.0 * nbin / len(ii)             # % of the whole group validly used per bin
            if np.any(frac_pct > 0):
                if axd is None:
                    axd = ax.twinx(); axd.set_ylim(0, 105)
                    axd.set_ylabel("valid % of group", fontsize=7); axd.tick_params(labelsize=6)
                axd.plot(xc, frac_pct, ls="none", marker=det_markers[gi % len(det_markers)],
                         mfc="none", mec=c, ms=6, mew=1.3, alpha=0.9, zorder=2)
    ax.axvline(0, ls="--", c="k"); ax.set_ylabel(ylab); ax.legend(fontsize=8)

def event_stack(props, triggers=("t_AGN",), ref0=False, band="iqr", xwin=2.0, nb=12, fname=None, split=None,
                show_detect=False):
    """Event-aligned stacks for any registered property × trigger, comparing groups (default
    fast/slow). props/triggers: name or list. ref0=True -> per-galaxy ΔX. Verbatim §8e."""
    props = [props] if isinstance(props, str) else list(props)
    triggers = [triggers] if isinstance(triggers, str) else list(triggers)
    props = [p for p in props if p in PROP_REGISTRY] or list(PROP_REGISTRY)[:1]
    triggers = [t for t in triggers if t in TRIGGERS] or ["t_QT"]
    nr, nc = len(props), len(triggers)
    fig, axs = plt.subplots(nr, nc, figsize=(5.6 * nc, 3.5 * nr), squeeze=False, sharex="col", sharey="row")
    for ri, pn in enumerate(props):
        spec = PROP_REGISTRY[pn]
        ylab = (r"$\Delta$" if ref0 else "med ") + ("log " if spec["logy"] else "") + spec["label"]
        for ci, tn in enumerate(triggers):
            ax = axs[ri][ci]
            _estack(ax, TRIGGERS[tn], spec["fn"], ylab, logy=spec["logy"], xwin=xwin, nb=nb, band=band,
                    ref0=ref0, split=split, show_detect=show_detect)
            if ref0:
                ax.axhline(0, ls=":", c="grey")
            if ri == nr - 1:
                ax.set_xlabel(f"t − {tn}  [Gyr]")
            if ri == 0:
                ax.set_title(tn)
    _grp = "  (" + " vs ".join(g[2] for g in split) + ")" if split else f"  ({SPLIT_DESC})"
    fig.suptitle(f"event-aligned {'Δ-' if ref0 else ''}stacks{_grp}, 25–75% bands", y=1.0)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
# ── §8g·shape full-history SFH/reservoir stacks on a movable critical-point clock ──
_SFH_CANON = {"sfr": "SFR", "ssfr": "sSFR", "gas/M*": "gas/M*", "dust/M*": "dust/M*",
              "H2/M*": "H2/M*", "HI/M*": "HI/M*", "dust/gas": "dust/gas"}
_SFH_YLAB = {"sfr": r"$\log_{10}$ SFR", "ssfr": r"$\log_{10}$ sSFR [1/yr]",
             "gas/M*": r"$\log_{10}\,M_{\rm gas}/M_\star$", "dust/M*": r"$\log_{10}\,M_{\rm dust}/M_\star$",
             "H2/M*": r"$\log_{10}\,M_{\rm H_2}/M_\star$", "HI/M*": r"$\log_{10}\,M_{\rm HI}/M_\star$",
             "dust/gas": r"$\log_{10}\,M_{\rm dust}/M_{\rm gas}$"}

def _sfh_track(q_col):
    """log10 of a per-snapshot quantity column (finite & >0), sorted by cosmic time."""
    q = np.asarray(q_col, float)
    good = np.isfinite(q) & (q > 0) & np.isfinite(t_cosmic_yr)
    if good.sum() < 3:
        return None, None
    ts, ys = t_cosmic_yr[good], np.log10(q[good]); o = np.argsort(ts)
    return ts[o], ys[o]

def sfh_shape(anchors=("sf_peak", "qt", "gas_min"), kind="ssfr", split=None,
              xwin=(-12.0, 10.0), nx=61, min_n=5, facet_mass=False, fname=None):
    """Median (+IQR) log-quantity stacked on each critical-point clock, per population."""
    _q = CATALOG_PROPS[_SFH_CANON[kind]][0](P)       # per-snapshot × galaxy array for this kind
    if _q is None:
        print(f"  {kind} unavailable at this anchor; skipped."); return
    groups = split if split is not None else [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]
    xg = np.linspace(xwin[0], xwin[1], nx); rows = list(range(NBINS)) if facet_mass else [None]
    fig, axes = plt.subplots(len(rows), len(anchors), squeeze=False,
                             figsize=(4.6 * len(anchors), 3.6 * len(rows)), sharex=True, sharey="row")
    for ri, bi in enumerate(rows):
        binmask = np.ones(len(records), bool) if bi is None else (mbin == bi)
        for ci, st in enumerate(anchors):
            ax = axes[ri][ci]; tkey = f"t_{st}"
            for msk, c, lab in groups:
                acc = []
                for i in np.where(msk & binmask)[0]:
                    t0 = records[i].get(tkey, np.nan)
                    if not np.isfinite(t0):
                        continue
                    ts, ys = _sfh_track(_q[:, records[i]["col"]])
                    if ts is None:
                        continue
                    acc.append(np.interp(xg, (ts - t0) / 1e9, ys, left=np.nan, right=np.nan))
                if not acc:
                    continue
                M = np.asarray(acc); nbin = np.sum(np.isfinite(M), axis=0)
                with np.errstate(all="ignore"):
                    med = np.nanmedian(M, axis=0); lo = np.nanpercentile(M, 25, axis=0); hi = np.nanpercentile(M, 75, axis=0)
                keep = nbin >= min_n
                ax.plot(xg, np.where(keep, med, np.nan), "-", color=c, lw=2, label=f"{lab} (n={len(acc)})")
                ax.fill_between(xg, np.where(keep, lo, np.nan), np.where(keep, hi, np.nan), color=c, alpha=0.15, lw=0)
            ax.axvline(0, ls=":", c="k", lw=1)
            if ri == 0:
                ax.set_title(f"aligned on {st}")
            if ri == len(rows) - 1:
                ax.set_xlabel(rf"$t - t_{{\rm {st}}}$ [Gyr]")
            if ci == 0:
                ax.set_ylabel(((f"logM* {MASS_BIN_LABELS[bi]}\n" if bi is not None else "") + "med " + _SFH_YLAB[kind]))
            ax.legend(fontsize=8)
    fig.suptitle(f"critical-point-clock stacks — {kind};  split: " + " vs ".join(g[2] for g in groups), y=1.01)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

def dust_split(thresh=-5.0, stage="gas_min", colors=("C2", "0.5")):
    """Split on log10(M_dust/M*) at `stage` (default gas_min): dusty vs non-dusty."""
    dfr = DIAGS[stage]["dust/M*"]; defi = np.isfinite(dfr)
    dusty = defi & (dfr > 10.0 ** thresh); nond = defi & ~(dfr > 10.0 ** thresh)
    return [(dusty, colors[0], f"dusty (log Md/M*>{thresh:g} @{stage})"), (nond, colors[1], "non-dusty")]

# Part 3 — Driver: build every anchor's products

Pure load + in-memory analysis (records, QC, AGN metrics, coupling, DIAGS, profiles). Heavy
products are loaded from cache; missing ones degrade to NaN with a warning. `RESULTS[z]` bundles
everything; `activate_anchor(z)` binds the engine globals to that anchor.

In [ ]:
RESULTS = {}
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        print(f"[{A['tag']}] history missing -> set BUILD_MULTI_Z=True (cluster). Skipping."); continue
    H = load_anchor_history(A)
    P_ = H["P"]; t_ = H["t_cosmic_yr"]; z_ = H["redshift"]; sn_ = H["snaps_arr"]; gid_ = H["galaxy_ids"]
    n_snap, n_gal = P_["sfr"].shape
    smask, _ = passive_sample_mask(P_, t_)
    scols = np.where(smask)[0]
    REC = build_records(P_, t_, z_, gid_, scols, MASS_BIN_EDGES, HMASS_BIN_EDGES,
                        PASSIVE_FACTOR, TAU_SPLIT_LOG, COSMO, find_quenching_times)
    records = REC["records"]; STAGES = REC["STAGES"]; cols = REC["cols"]
    if len(records) == 0:
        print(f"[{A['tag']}] no usable quench events; skipping."); continue
    QC = qc_anchor(P_, t_, z_, sn_, gid_, cols, BOX, f"[{A['tag']}]")
    # AGN history (optional build)
    if BUILD_BH and not os.path.exists(A["bh_path"]):
        build_bh_for_anchor(A, gid_, sn_, n_gal)
    BH = load_bh(A["bh_path"]) if os.path.exists(A["bh_path"]) else None
    if BH is not None:
        t_agn = agn_activation_time(BH, records, t_)
        MET = bh_metrics(BH, records, t_, EPS_R, c2_erg_per_Msun, JET_LOGMBH, JET_FEDD)
        CO = build_coupling(BH, P_, records, t_, t_agn, JET_LOGMBH, JET_FEDD, XRAY_FEDD_MAX, XRAY_FGAS_MAX)
    else:
        print(f"[{A['tag']}] BH history missing -> coupling NaN (set BUILD_BH=True).")
        nan = np.full(len(records), np.nan); z0 = np.zeros(len(records), bool)
        t_agn = nan.copy(); MET = dict(Eagn=nan, dMbh=nan, jetfrac=nan, mdot_qt=nan)
        CO = dict(fgas_hist=None, wjet_hist=None, xray_hist=None, xcoup_hist=None,
                  xfrac_post=nan.copy(), xstr_post=nan.copy(), tercile=(np.nan, np.nan),
                  no_agn=z0.copy(), weak=z0.copy(), strong=z0.copy(), inter=z0.copy(), no_fb=z0.copy(),
                  SPLIT_X=[(z0.copy(), "C3", "ISM-coupled"), (z0.copy(), "C0", "never coupled")],
                  SPLIT_XS=[(z0.copy(), "C3", "strong"), (z0.copy(), "C0", "weak"),
                            (z0.copy(), "C2", "intermediate"), (z0.copy(), "0.5", "no AGN")])
    # mirror the tau_q fast/slow split into CO so all class masks live in one place (used by 6q/6r)
    CO["fast"] = np.asarray(REC["is_fast"], bool); CO["slow"] = np.asarray(REC["is_slow"], bool)
    # add AGN activation as a critical point (stage "agn"), inserted after ssfr_min
    if "agn" not in STAGES:
        STAGES.insert(STAGES.index("ssfr_min") + 1, "agn")
    for i, r in enumerate(records):
        r["t_agn"] = t_agn[i]
        r["row_agn"] = int(np.argmin(np.abs(t_ - t_agn[i]))) if np.isfinite(t_agn[i]) else -1
    DIAGS = build_diags(records, P_, cols, STAGES)
    try:                                             # RP profiles come from the reduced 100 kpc files
        _PIDX_z = build_prog_index(A, gid_, sn_); _PIDX_CACHE[_zt] = _PIDX_z
    except Exception as _e:
        print(f"  [{A['tag']}] catalog/PIDX unavailable ({type(_e).__name__}); RP profiles NaN."); _PIDX_z = None
    RP = reduced_profile_scalars(records, STAGES, _PIDX_z, sn_)

    RESULTS[_zt] = dict(
        z=_zt, A=A, n_snap=n_snap, n_gal=n_gal,
        P=P_, t_cosmic_yr=t_, redshift=z_, snaps_arr=sn_, galaxy_ids=gid_,
        sample_cols=scols, records=records, STAGES=STAGES, cols=cols,
        tau_q=REC["tau_q"], tau_q_over_tH=REC["tau_q_over_tH"],
        is_fast=REC["is_fast"], is_slow=REC["is_slow"], mbin=REC["mbin"], hmbin=REC["hmbin"],
        logm_rec=REC["logm_rec"], logmh_rec=REC["logmh_rec"],
        well_behaved=QC["well_behaved"], QC=QC, BH=BH, t_agn=t_agn, MET=MET, CO=CO,
        DIAGS=DIAGS, RP=RP)
    RESULTS[_zt]["REG"] = build_registry(RESULTS[_zt])   # per-anchor PROP_REGISTRY + TRIGGERS
    del H; gc.collect()

print("\nanchors built:", {z: len(RESULTS[z]["records"]) for z in RESULTS})

# Part 4a — Novel-sample restriction (exclude z=0 overlap)

For cross-anchor comparison, optionally restrict each z>0 anchor to galaxies that are **not** progenitors of the z=0-selected sample (set `NOVEL_REPORT` to see the counts, `NOVEL_ONLY` to apply by folding the overlap out of `well_behaved`).

In [ ]:
# ── Part 4·novel — keep only galaxies NOT already on a z=0-sample progenitor track (cross-anchor bias) ──
# A z>0 anchor galaxy that is a progenitor of a z=0-selected galaxy was already analysed at z=0;
# including it would double-count and bias the cross-anchor comparison. Drop those, keep the "novel"
# ones. We test each anchor galaxy's (anchor_snap, catalog_index) against the z=0 sample's progenitor
# tracks. Anchors earlier than the z=0 track start (e.g. z=4 below END_SNAP) are all-novel by default.
NOVEL_Z0 = 0.0
if (NOVEL_ONLY or NOVEL_REPORT) and NOVEL_Z0 in RESULTS and len(RESULTS) > 1:
    P0 = _pidx(NOVEL_Z0); snaps0 = RESULTS[NOVEL_Z0]["snaps_arr"]
    _visited = {int(s): set(int(x) for x in P0[ri, :][np.isfinite(P0[ri, :])])
                for ri, s in enumerate(snaps0)}                      # snapshot -> z=0-sample catalog indices
    print(f"{'z':>6} {'sel':>6} {'novel':>6} {'overlap':>7} {'wb':>6} {'wb&novel':>9}")
    for z, B in RESULTS.items():
        if z == NOVEL_Z0:
            B["novel"] = np.ones(len(B["records"]), bool)
        else:
            vis = _visited.get(int(B["A"]["snap"]), set())
            B["novel"] = np.array([int(r["gid"]) not in vis for r in B["records"]], bool)
        nov = B["novel"]; wb = B["well_behaved"]; n = len(nov)
        print(f"{z:6.2f} {n:6d} {int(nov.sum()):6d} {n - int(nov.sum()):7d} {int(wb.sum()):6d} {int((wb & nov).sum()):9d}")
    if NOVEL_ONLY:
        for z, B in RESULTS.items():
            if z != NOVEL_Z0:
                B["well_behaved"] = B["well_behaved"] & B["novel"]
        print("NOVEL_ONLY -> well_behaved now excludes z=0-overlap galaxies at every z>0 anchor "
              "(re-run the RESULTS cell to reset).")
    else:
        print("NOVEL_REPORT only — counts above; set NOVEL_ONLY=True to restrict the analysis.")
else:
    print("[novel] off (set NOVEL_REPORT/NOVEL_ONLY) or needs z=0 + >1 anchor in RESULTS.")

In [ ]:
def activate_anchor(z):
    """Bind the module globals the reused engines read to anchor `z`'s arrays.
    QC is folded into is_fast/is_slow so every plot respects the well-behaved mask."""
    B = RESULTS[z]; g = globals()
    wb = B["well_behaved"]
    ns = dict(
        P=B["P"], t_cosmic_yr=B["t_cosmic_yr"], redshift=B["redshift"], snaps_arr=B["snaps_arr"],
        galaxy_ids=B["galaxy_ids"], records=B["records"], STAGES=B["STAGES"], cols=B["cols"],
        mbin=B["mbin"], hmbin=B["hmbin"], is_fast=B["is_fast"] & wb, is_slow=B["is_slow"] & wb,
        is_fast_raw=B["is_fast"], is_slow_raw=B["is_slow"], well_behaved=wb,
        DIAGS=B["DIAGS"], BH=B["BH"], t_agn=B["t_agn"], CO=B["CO"], MET=B["MET"])
    rp = B["RP"]
    ns.update(RP=rp, RP_OK=rp["OK"], RP_STAGES=rp["STAGES"], PROF_RMID=PROF_RMID)
    reg = B["REG"]                                   # event-stack engine globals
    ns.update(_ord=reg["_ord"], t_inc=reg["t_inc"], ssfr_hist=reg["ssfr_hist"],
              PROP_REGISTRY=reg["PROP_REGISTRY"], TRIGGERS=reg["TRIGGERS"],
              SPLIT_X=B["CO"]["SPLIT_X"], SPLIT_XS=B["CO"]["SPLIT_XS"], SPLIT_DESC="fast vs slow")
    g.update(ns)
    return B

print("activate_anchor ready. anchors:", list(RESULTS))

In [ ]:
# ── BUILD_PROFILE_PLAN (GATED): write per-anchor SLURM plans, then run the REDUCED extraction ──
# The 100 kpc reduced particle files (gas/HI/H2/dust/star + temperature) are the SINGLE profile
# source. Each plan lives in its own dir (output/cis100/caesar_sfh/prof_<tag>/). Build the reduced
# files per anchor (submit_reduced_particles.sh honours the DUST_PLAN env), then build PROFILE_CACHE:
#   for tag in z0 z0p4 z0p7 z0p9 z1 z4; do
#     DUST_PLAN=output/cis100/caesar_sfh/prof_${tag}/dust_profile_plan_${tag}.hdf5 \
#       sbatch --array=0-15 submit_reduced_particles.sh
#   done
#   # then in the notebook (Part 0): BUILD_PROFILE_CACHE=True (first time), or
#   #   REBUILD_PROFILE_CACHE=True after adding star/temp -> rebuilds PROFILE_CACHE from the reduced files.
if BUILD_PROFILE_PLAN:
    for _zt, B in RESULTS.items():
        A = B["A"]
        if os.path.exists(A["plan_path"]):        # idempotent: z=0 already has prof_z0; delete to rebuild
            print(f"  [{A['tag']}] plan exists -> {os.path.basename(A['plan_path'])} (skip)"); continue
        PIDX = build_prog_index(A, B["galaxy_ids"], B["snaps_arr"])
        build_profile_plan(B["records"], B["STAGES"], B["snaps_arr"], PIDX,
                           A["plan_path"], SIM_NAME)
        print(f"  [{A['tag']}] plan -> {A['plan_path']}  (galaxy dedup happens in the SLURM jobs)")
else:
    print("BUILD_PROFILE_PLAN=False -> expecting reduced Σ-profiles (PROFILE_CACHE / RP).")
    for _zt, B in RESULTS.items():
        print(f"  [{B['A']['tag']}] profiles:", "ok" if B["RP"]["OK"] else "MISSING")

# Part 4b — QC summary across anchors

Before any science: how clean are the samples? Broken progenitor links, halo-mass teleports and
bad halo associations are tabulated per anchor; all downstream analysis uses only `well_behaved`
galaxies (folded into the fast/slow masks by `activate_anchor`).

In [ ]:
import csv
hdr = ["z", "N_rec", "well", "broken", "haloJmp", "badAssoc", "noHalo", "medSHMR"]
rows = []
print(f"{'z':>5s} {'N_rec':>6s} {'well':>5s} {'broken':>7s} {'haloJmp':>8s} "
      f"{'badAssoc':>9s} {'noHalo':>7s} {'medSHMR':>8s}")
for z in RESULTS:
    B = RESULTS[z]; Q = B["QC"]; n = len(B["records"]); well = int(Q['well_behaved'].sum())
    rows.append([z, n, well, Q['n_broken_link'], Q['n_halo_teleport'],
                 Q['n_bad_assoc'], Q['n_no_halo'], round(float(Q['median_shmr']), 4)])
    print(f"{z:5.1f} {n:6d} {well:5d} {Q['n_broken_link']:7d} "
          f"{Q['n_halo_teleport']:8d} {Q['n_bad_assoc']:9d} {Q['n_no_halo']:7d} {Q['median_shmr']:8.4f}")
with open(statpath("qc_integrity_summary.csv"), "w", newline="") as _f:
    csv.writer(_f).writerows([hdr] + rows)
print("\nNOTE: spatial-continuity uses P['pos'] with auto comoving/physical detection from the "
      "tracked extent; if a column over-flags, inspect that anchor's positions before trusting it.")
print(f"[saved] {statpath('qc_integrity_summary.csv')}")

# Part 5 — Sample composition

Fast/slow and AGN-coupling subsample counts per anchor and mass bin — the populations every
comparison below is built on.

In [ ]:
import csv
hdr = ["z", "mass_bin", "N", "fast", "slow", "strong", "weak", "inter", "noAGN"]
rows = []
print(f"{'z':>5s} {'mass bin':>10s} {'N':>4s} {'fast':>5s} {'slow':>5s} "
      f"{'strong':>7s} {'weak':>5s} {'inter':>6s} {'noAGN':>6s}")
for z in RESULTS:
    B = activate_anchor(z); CO = B["CO"]; wb = B["well_behaved"]
    for bi, lab in enumerate(MASS_BIN_LABELS):
        m = (mbin == bi) & wb
        N = int(m.sum()); nf = int((m & is_fast).sum()); ns = int((m & is_slow).sum())
        nst = int((m & CO['strong']).sum()); nw = int((m & CO['weak']).sum())
        ni = int((m & CO['inter']).sum()); nnf = int((m & CO['no_fb']).sum())
        rows.append([z, lab, N, nf, ns, nst, nw, ni, nnf])
        print(f"{z:5.1f} {lab:>10s} {N:4d} {nf:5d} {ns:5d} {nst:7d} {nw:5d} {ni:6d} {nnf:6d}")
with open(statpath("sample_composition.csv"), "w", newline="") as _f:
    csv.writer(_f).writerows([hdr] + rows)
print("fast+slow = N; strong+weak+inter+noAGN = N (anchors with BH; else those galaxies are unclassified).")
print(f"[saved] {statpath('sample_composition.csv')}")

### Part 5·coverage — which figure families each anchor can produce

Plots are not made uniformly across anchors because they depend on **per-anchor data products**:
fast/slow stacks need only the history; **coupling (strong/weak) figures need a BH history**
(`bh_history_anchor_<tag>.hdf5`); the radial-Σ profile figures need the reduced particle files
+ the 6s cache. This cell tabulates, from file existence, which families each anchor can produce
and what to build to fill the gaps — saved to `01_sanity_and_stats/figure_coverage.csv`.

In [ ]:
import csv
_red_any = (os.path.isdir(REDUCED_DIR)
            and any(f.endswith(".h5") for _d, _s, _fs in os.walk(REDUCED_DIR) for f in _fs))
_cache_any = any(fn.startswith("reduced_profiles_cache_")
                 for fn in (os.listdir(TABLEDIR) if os.path.isdir(TABLEDIR) else []))
hdr = ["z", "tag", "in_RESULTS", "has_BH", "has_profiles",
       "reduced_files_present", "profile_cache_present", "fastslow_figs", "coupling_figs", "profile_figs"]
rows = []
print(f"{'z':>5s} {'inRES':>5s} {'BH':>3s} {'prof':>4s} {'redu':>4s}   {'fast/slow':>9s}  {'coupling':>13s}  {'profile-Σ':>12s}")
for _zt, A in ANCHORS.items():
    inres = _zt in RESULTS
    hasbh = inres and (RESULTS[_zt]["BH"] is not None)
    hasprof = bool(RESULTS.get(_zt, {}).get("RP", {}).get("OK"))
    fastslow = "yes" if inres else "NO(history)"
    coupling = "yes" if hasbh else "NO(build BH)"
    profile = "yes" if (_red_any and _cache_any) else ("part(run 6s)" if _red_any else "NO(reduced)")
    rows.append([_zt, A["tag"], int(inres), int(hasbh), int(hasprof),
                 int(_red_any), int(_cache_any), fastslow, coupling, profile])
    print(f"{_zt:5.1f} {('ok' if inres else '--'):>5s} {('ok' if hasbh else '--'):>3s} "
          f"{('ok' if hasprof else '--'):>4s} {('ok' if _red_any else '--'):>4s}   "
          f"{fastslow:>9s}  {coupling:>13s}  {profile:>12s}")
with open(statpath("figure_coverage.csv"), "w", newline="") as _f:
    csv.writer(_f).writerows([hdr] + rows)
print("\nfast/slow stacks (6c, 6e)      -> every anchor in RESULTS.")
print("coupling stacks (6a·clk,6b,6d,6f,6i,6m,6v,7·2) -> need per-anchor BH:")
print("    set BUILD_BH=True and re-run Part 3 on the cluster to cover every anchor.")
print("profile-Σ figs (6l,6n,6q,6r,6t,6w,6x)  -> need reduced files (submit_reduced_particles.sh) + 6s cache.")
print(f"[saved] {statpath('figure_coverage.csv')}")

In [ ]:
# ── pooled record-level scalars across anchors (well-behaved). gas_min is used ONLY to flag
#    dusty vs non-dusty; everything else is event-aligned (Part 6). ──
def _sig_at(B, key, stage):
    RP = B["RP"]; DPS = RP["STAGES"]
    if not RP["OK"] or key not in RP or stage not in DPS:
        return np.full(len(B["records"]), np.nan)
    return RP[key][:, DPS.index(stage)]

def _diag_at(B, key, stage):
    return B["DIAGS"][stage][key] if stage in B["DIAGS"] else np.full(len(B["records"]), np.nan)

def _wb_split(split):
    """Fold the active anchor's well_behaved mask into a [(mask,color,label),...] split spec."""
    return [(m & well_behaved, c, l) for (m, c, l) in split]

POOL = {k: [] for k in ("z", "is_fast", "is_slow", "mbin", "tau_q_over_tH", "xstr_post",
                        "no_agn", "weak", "strong", "inter", "no_fb", "lag_qt_agn",
                        "SigH2_qt", "SigH2_postq", "dust_postq", "dustfrac_gasmin",
                        "Mgas_qt", "Mstar_qt", "Mgas_postq", "Mstar_postq")}
for z in RESULTS:
    B = RESULTS[z]; wb = B["well_behaved"]; n = len(B["records"])
    lag = (np.array([r["t_qt"] for r in B["records"]]) - np.asarray(B["t_agn"], float)) / GYR
    cmap = {
        "z": np.full(n, z), "is_fast": B["is_fast"], "is_slow": B["is_slow"], "mbin": B["mbin"],
        "tau_q_over_tH": B["tau_q_over_tH"], "xstr_post": B["CO"]["xstr_post"],
        "no_agn": B["CO"]["no_agn"], "weak": B["CO"]["weak"], "strong": B["CO"]["strong"],
        "inter": B["CO"]["inter"], "no_fb": B["CO"]["no_fb"],
        "lag_qt_agn": lag,
        "SigH2_qt": _sig_at(B, "Sigma_H2_eff", "qt"),
        "SigH2_postq": _sig_at(B, "Sigma_H2_eff", "post_quench"),
        "dust_postq": _diag_at(B, "dust/M*", "post_quench"),
        "dustfrac_gasmin": _diag_at(B, "dust/M*", "gas_min"),
        "Mgas_qt": _diag_at(B, "Mgas", "qt"), "Mstar_qt": _diag_at(B, "Mstar", "qt"),
        "Mgas_postq": _diag_at(B, "Mgas", "post_quench"), "Mstar_postq": _diag_at(B, "Mstar", "post_quench")}
    for k, v in cmap.items():
        POOL[k].append(np.asarray(v)[wb])
POOL = {k: (np.concatenate(v) if v else np.array([])) for k, v in POOL.items()}
# gas fraction (the BUDGET axis) at the same event stages as Sigma_H2 (the GEOMETRY axis), for 6g
with np.errstate(all="ignore"):
    POOL["fgas_qt"]    = np.where(POOL["Mstar_qt"]    > 0, POOL["Mgas_qt"]    / POOL["Mstar_qt"],    np.nan)
    POOL["fgas_postq"] = np.where(POOL["Mstar_postq"] > 0, POOL["Mgas_postq"] / POOL["Mstar_postq"], np.nan)
DUST_THR = -4.0   # log10(M_dust/M*) at gas_min separating dusty/non-dusty (~1e-4; tune per sample)
POOL["dusty"] = np.isfinite(POOL["dustfrac_gasmin"]) & (POOL["dustfrac_gasmin"] > 10.0 ** DUST_THR)
print("pooled well-behaved records:", len(POOL["z"]),
      f"| dusty-at-gas_min (log Md/M*>{DUST_THR}): {int(POOL['dusty'].sum())}")

# Part 6 — Event-aligned analysis across redshift (§8-style)

The core comparison: property *tracks* aligned on the critical points (`t_SFpeak`, `t_AGN`, `t_SFT`,
`t_QT`, `post_quench`) over a ±window, comparing galaxies at different times **after quenching**.
The organizing axis is **AGN–ISM coupling** (`SPLIT_XS` = terciles of post-`t_AGN` coupling strength,
the §8j definition). `gas_min` enters only as the dusty/non-dusty divider (6e). Each cell loops over
the anchors; coupling/dusty splits are folded through `well_behaved`.

### 6a — Coupling & dusty/non-dusty splits (counts per anchor)

In [ ]:
import csv
hdr = ["z", "N_wb", "strong", "weak", "inter", "noAGN", "dusty_at_gmin"]
rows = []
print(f"{'z':>5s} {'N_wb':>5s} {'strong':>7s} {'weak':>5s} {'inter':>6s} {'noAGN':>6s} "
      f"{'dusty@gmin':>11s}")
for z in RESULTS:
    B = activate_anchor(z); CO = B["CO"]; wb = well_behaved
    nstrong = int((CO["strong"] & wb).sum()); nweak = int((CO["weak"] & wb).sum())
    ninter = int((CO["inter"] & wb).sum()); nnofb = int((CO["no_fb"] & wb).sum())
    df = _diag_at(B, "dust/M*", "gas_min")
    ndusty = int((np.isfinite(df) & (df > 10.0 ** DUST_THR) & wb).sum())
    nwb = int(wb.sum())
    rows.append([z, nwb, nstrong, nweak, ninter, nnofb, ndusty])
    print(f"{z:5.1f} {nwb:5d} {nstrong:7d} {nweak:5d} {ninter:6d} {nnofb:6d} {ndusty:11d}")
with open(statpath("coupling_dusty_counts.csv"), "w", newline="") as _f:
    csv.writer(_f).writerows([hdr] + rows)
print("\ncoupling terciles per anchor: tercile(lo,hi)=",
      {z: tuple(np.round(RESULTS[z]['CO']['tercile'], 3)) for z in RESULTS})
print("strong+weak+inter+noAGN = N_wb (anchors with BH).")
print(f"[saved] {statpath('coupling_dusty_counts.csv')}")

### 6a·clock — Coupling strength around each critical point (fast vs slow)

The continuous AGN–ISM coupling `x_coup` (jet strength gated by gas-poorness) stacked on each
critical-point clock, **fast vs slow**. A direct read of *when* the coupling switches on relative to
the SF peak, AGN ignition, SFT, QT, post-quench and gas_min — and whether fast quenchers couple
earlier or harder. Needs the BH history (coupling).

In [ ]:
# surface-density tracks added to every Part-6 stack (registered only when the profile product exists)
SIGMA_PROPS = ["Sigma_H2", "Sigma_HI", "Sigma_gas", "Sigma_dust"]
COUP_TRIG = ["t_SFpeak", "t_AGN", "t_SFT", "t_QT", "post_quench", "gas_min"]
for z in RESULTS:
    B = activate_anchor(z)   # default split = is_fast / is_slow (well_behaved-folded)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling unavailable (set BUILD_BH=True)."); continue
    print(f"\n=== z = {z} : x_coup + ISM surface densities around the critical points (fast vs slow) ===")
    event_stack([p for p in ["xcoup"] + SIGMA_PROPS if p in PROP_REGISTRY], triggers=COUP_TRIG, xwin=2.0, nb=10,
                fname=f"coupling_clock_fastslow_z{_ztag(z)}.png")

### 6b — The scenario stack, by coupling strength (the canonical §8j call)

`dust/M*`, `H2/M*`, gas & stellar compactness (`R20g/R80g`, `R20s/R80s`) and `Rdust/Rstar` aligned on
each critical point, **strong vs weak coupling**. The strong-coupling prediction: earlier/steeper
compaction and a dust rise after `t_QT`. Needs BH history (coupling); profile product adds `Rdust/Rstar`.

In [ ]:
SCEN = ["dust/M*", "H2/M*", "R20g/R80g", "R20s/R80s", "Rdust/Rstar"] + SIGMA_PROPS
SCEN_TRIG = ["t_SFpeak", "t_AGN", "t_SFT", "t_QT", "post_quench"]
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling split unavailable (set BUILD_BH=True)."); continue
    print(f"\n=== z = {z} : scenario stack by coupling strength ===")
    event_stack([p for p in SCEN if p in PROP_REGISTRY], triggers=SCEN_TRIG,
                split=_wb_split(SPLIT_XS), xwin=2.0, nb=10, fname=f"scenario_xstrength_z{_ztag(z)}.png")

### 6c — The same stack, organized by fast/slow (compare the two axes)

Identical properties/triggers split by quench *time* instead of coupling. If coupling (6b) separates
the tracks more cleanly than fast/slow does here, quench time is the weaker organizing axis.

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)   # default split = is_fast/is_slow (already well_behaved-folded)
    print(f"\n=== z = {z} : scenario stack by fast/slow ===")
    event_stack([p for p in SCEN if p in PROP_REGISTRY], triggers=SCEN_TRIG,
                xwin=2.0, nb=10, fname=f"scenario_fastslow_z{_ztag(z)}.png")

### 6d — SF & AGN response around ignition and quenching

sSFR, BHAR, molecular gas and dust around `t_AGN`, `t_QT`, `post_quench`, by coupling. Shows the
post-quench sSFR behaviour (the rebound) in its proper event-aligned form.

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling figure skipped (set BUILD_BH=True)."); continue
    print(f"\n=== z = {z} : SF/AGN response ===")
    event_stack([p for p in ["sSFR", "BHAR", "M_H2", "dust/M*"] + SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=["t_AGN", "t_QT", "post_quench"], split=_wb_split(SPLIT_XS),
                xwin=3.0, nb=12, fname=f"response_xstrength_z{_ztag(z)}.png")

### 6e — Dusty vs non-dusty at gas_min → quenching mode (the end goal)

gas_min's one job: split dusty/non-dusty. Do the dusty-at-gas_min galaxies show the strong-coupling
signature (compact, dust rising after QT)? Event stacks split by dustiness, plus a Fisher test of
dusty-at-gas_min against coupling and fast/slow (pooled).

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)
    ds = _wb_split(dust_split(thresh=DUST_THR, stage="gas_min"))
    print(f"\n=== z = {z} : stacks split by dust-at-gas_min ===")
    event_stack([p for p in ["dust/M*", "H2/M*", "R20g/R80g", "sSFR"] + SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=["t_QT", "post_quench"], split=ds, xwin=3.0, nb=12,
                fname=f"dustsplit_z{_ztag(z)}.png")

def _fisher(a, b):
    a = a.astype(bool); b = b.astype(bool)
    tab = np.array([[np.sum(a & b), np.sum(a & ~b)], [np.sum(~a & b), np.sum(~a & ~b)]])
    if tab.min() == 0 and tab.sum() < 20:
        return tab, np.nan, np.nan
    odds, p = fisher_exact(tab); return tab, odds, p
print("\nFisher: is dusty-at-gas_min associated with...  (pooled, well-behaved)")
for lab, other in [("strong-coupled", POOL["strong"]), ("weak-coupled", POOL["weak"]),
                   ("no-AGN", POOL["no_agn"]), ("fast", POOL["is_fast"]), ("slow", POOL["is_slow"])]:
    tab, odds, p = _fisher(POOL["dusty"], other)
    print(f"  vs {lab:>15s}: odds={odds:.2f}  p={p:.2g}   "
          f"[dusty&{lab}={tab[0,0]}, dusty&not={tab[0,1]}]")

### 6e·gasmin — Dusty vs non-dusty AT gas_min, against coupling and timescale

`gas_min` is the **timing-independent** anchor (unlike `t_QT` / `post_quench`, which move with the
quenching speed), so it is the fair point for the headline question: **does strong AGN–ISM coupling
leave galaxies MORE or LESS dusty?** Pooled across anchors, the dusty fraction
(`log M_dust/M* > DUST_THR` ≈ 1e-4) is compared across fast / slow / strong / weak, with a Fisher
test on the strong-vs-weak and fast-vs-slow contingencies. Needs BH for the coupling classes.

In [ ]:
# dusty fraction @ gas_min for each class (pooled, well-behaved); answers more-or-less-dusty directly.
defi = np.isfinite(POOL["dustfrac_gasmin"])
classes = [("fast", POOL["is_fast"], "C3"), ("slow", POOL["is_slow"], "C0"),
           ("strong", POOL["strong"], "C3"), ("weak", POOL["weak"], "C0"),
           ("intermediate", POOL["inter"], "C2"), ("no-AGN", POOL["no_fb"], "0.5")]
labs, fr, er, Ns, Ds = [], [], [], [], []
for lab, m, c in classes:
    sel = m & defi; N = int(sel.sum()); d = int((POOL["dusty"] & sel).sum())
    p = (d / N) if N else np.nan
    labs.append(lab); fr.append(p); Ds.append(d); Ns.append(N)
    er.append(np.sqrt(p * (1 - p) / N) if N else np.nan)
fig, ax = plt.subplots(figsize=(8.4, 4.4))
ax.bar(range(len(labs)), fr, yerr=er, color=[c for _, _, c in classes], alpha=0.85, capsize=4)
ax.axvline(1.5, ls=":", c="k", lw=1)   # timescale (left) | coupling (right)
for i, (p, d, N) in enumerate(zip(fr, Ds, Ns)):
    ax.text(i, (p if np.isfinite(p) else 0) + 0.012, f"{d}/{N}", ha="center", fontsize=8)
ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=20, ha="right")
ax.set_ylabel(fr"dusty fraction @ gas_min  ($\log M_d/M_* > {DUST_THR:g}$)")
ax.set_title("Is strong coupling more or less dusty at gas_min?  (pooled)")
plt.tight_layout(); plt.savefig(figpath("dustsplit_gasmin_fraction.png"), dpi=130, bbox_inches="tight"); plt.show()

# completeness check — do the partitions account for every dusty galaxy?
S, W = POOL["strong"].astype(bool), POOL["weak"].astype(bool)
Im, NF = POOL["inter"].astype(bool), POOL["no_fb"].astype(bool)
nd = int(POOL["dusty"].sum())
d_ts = int((POOL["dusty"] & (POOL["is_fast"].astype(bool) | POOL["is_slow"].astype(bool))).sum())
d_co = int((POOL["dusty"] & (S | W | Im | NF)).sum())
d_un = int((POOL["dusty"] & ~(S | W | Im | NF)).sum())
print(f"dusty@gas_min total = {nd};  fast+slow = {d_ts};  strong+weak+inter+noAGN = {d_co};  "
      f"unclassified (anchor without BH coupling) = {d_un}")

def _fisher2(dusty, ga, gb):
    dA = int((dusty & ga).sum()); nA = int(ga.sum())
    dB = int((dusty & gb).sum()); nB = int(gb.sum())
    tab = np.array([[dA, nA - dA], [dB, nB - dB]])
    odds, pp = ((np.nan, np.nan) if (tab.min() == 0 and tab.sum() < 20) else fisher_exact(tab))
    return dA, nA, dB, nB, odds, pp

print(f"dusty@gas_min: log10(M_dust/M*) > {DUST_THR:g}  (pooled well-behaved, n={int(defi.sum())})")
for axis, ga, gb, na, nb_ in [("coupling", POOL["strong"] & defi, POOL["weak"] & defi, "strong", "weak"),
                              ("timescale", POOL["is_fast"] & defi, POOL["is_slow"] & defi, "fast", "slow")]:
    dA, nA, dB, nB, odds, pp = _fisher2(POOL["dusty"], ga, gb)
    fA = (dA / nA) if nA else np.nan; fB = (dB / nB) if nB else np.nan
    verdict = "MORE" if fA > fB else ("LESS" if fA < fB else "EQUALLY")
    print(f"  {axis:>9s}: {na:>6s} {fA*100:5.1f}% dusty ({dA}/{nA})  vs  {nb_:>4s} {fB*100:5.1f}% ({dB}/{nB})"
          f"   -> {na} is {verdict} dusty  (odds={odds:.2f}, p={pp:.2g})")

### 6e·geom — H₂ geometry & central dust at gas_min vs coupling strength

The mechanism behind 6e·gasmin, made explicit and **at `gas_min`** (the timing-independent
anchor). Claim: **strong AGN–ISM coupling carves a central H₂ hole and pushes H₂ outward while
stripping the central dust; weak coupling keeps a compact central H₂ so dust survives in the
centre.** From each galaxy's reduced-profile slab at `gas_min` (`DP`, §5) we build per-galaxy
geometry scalars and test them against the continuous coupling strength `xstr_post` and the dust
budget `M_dust/M*`:

- `hole_H2` = 1 − Σ̄_H₂(<2.5 kpc)/max Σ_H₂  (≈1 → central hole, ≤0 → centrally peaked)
- `Rpeak_H2`, `Rmw_H2`, `R80_H2` — where the H₂ sits / how far it extends
- `Sig_in_dust`, `dust_core_frac` — central dust surface density / its share of the dust peak
- `C2080_dust` = R20/R80 of dust (smaller = more centrally concentrated)

Predicted with rising coupling: hole/extent **up**, central dust **down**, dust less concentrated.
Σ=0 annuli are kept (a real hole); galaxies absent at R are NaN (dropped).

In [ ]:
# ── 6e·geom·a — per-galaxy H2 geometry + central dust at gas_min, from the reduced Σ cache ──
def geom_scalars_at_gasmin(B, r_in_kpc=2.5):
    """Per-record (nrec,) geometry/central-dust scalars from the cached log-binned Σ(R) at gas_min.
    Σ=0 annuli are kept (central hole); a galaxy absent from the cache stays NaN."""
    keys = ("hole_H2", "Rpeak_H2", "Rmw_H2", "R80_H2",
            "Sig_in_dust", "dust_core_frac", "C2080_dust", "R50dust_R50star")
    out = {k: np.full(len(B["records"]), np.nan) for k in keys}
    if not B["RP"]["OK"] or "gas_min" not in B["STAGES"] or not PROFILE_CACHE:
        return out
    PIDX = _pidx(B["z"]); snaps = B["snaps_arr"]; rmid = PROF_RMID
    inb = rmid <= r_in_kpc
    dA = np.gradient(rmid) * 2.0 * np.pi * rmid
    for i, r in enumerate(B["records"]):
        row = r.get("row_gas_min", -1)
        if row is None or row < 0:
            continue
        g = PIDX[row, r["col"]]
        if not np.isfinite(g):
            continue
        d = PROFILE_CACHE.get((int(snaps[row]), int(g)))
        if not d:
            continue
        h2 = d.get("H2"); du = d.get("dust"); stp = d.get("star")
        with np.errstate(all="ignore"):
            if h2 is not None and np.isfinite(h2).any() and np.nanmax(h2) > 0:
                pk = np.nanmax(h2)
                sin = np.nanmean(h2[inb]) if np.isfinite(h2[inb]).any() else 0.0
                out["hole_H2"][i]  = 1.0 - sin / pk
                out["Rpeak_H2"][i] = rmid[np.nanargmax(h2)]
                w = np.where(np.isfinite(h2), np.maximum(h2, 0.0) * dA, 0.0)
                if w.sum() > 0:
                    out["Rmw_H2"][i] = (w * rmid).sum() / w.sum()
                out["R80_H2"][i] = _enclosed_radii(h2)[2]
            if du is not None and np.isfinite(du).any() and np.nanmax(du) > 0:
                out["Sig_in_dust"][i]    = np.nanmean(du[inb]) if np.isfinite(du[inb]).any() else 0.0
                out["dust_core_frac"][i] = out["Sig_in_dust"][i] / np.nanmax(du)
                r20d, _, r80d, _ = _enclosed_radii(du)
                if np.isfinite(r80d) and r80d > 0:
                    out["C2080_dust"][i] = r20d / r80d
            r50d = _enclosed_radii(du)[1] if du is not None else np.nan
            r50s = _enclosed_radii(stp)[1] if stp is not None else np.nan
            if np.isfinite(r50s) and r50s > 0:
                out["R50dust_R50star"][i] = r50d / r50s
    return out

# pool exactly like POOL (same RESULTS order, same well_behaved mask) -> row-aligned to POOL
_GEOM_KEYS = list(geom_scalars_at_gasmin(RESULTS[next(iter(RESULTS))]).keys())
GEOM = {k: [] for k in _GEOM_KEYS}
for z in RESULTS:
    B = RESULTS[z]; wb = B["well_behaved"]; g = geom_scalars_at_gasmin(B)
    for k in _GEOM_KEYS:
        GEOM[k].append(np.asarray(g[k])[wb])
GEOM = {k: (np.concatenate(v) if v else np.array([])) for k, v in GEOM.items()}
_expected = int(sum(int(np.asarray(RESULTS[z]["well_behaved"]).sum()) for z in RESULTS))
_ngeom = int(np.isfinite(GEOM["hole_H2"]).sum())
print(f"[6e·geom] pooled geometry scalars: {len(GEOM['hole_H2'])} records "
      f"({_ngeom} with a finite gas_min H2 profile); POOL has {len(POOL['z'])}.")
assert len(GEOM["hole_H2"]) == _expected, \
    f"GEOM ({len(GEOM['hole_H2'])}) != well-behaved records ({_expected}) — internal bug."
if len(POOL["z"]) != _expected:
    print(f"[6e·geom][warn] POOL has {len(POOL['z'])} rows but RESULTS has {_expected} well-behaved "
          "records -> POOL is STALE. Re-run the POOL cell (Part 5) and Part 6 top-to-bottom.")

In [ ]:
# ── 6e·geom·b — coupling drives the geometry, and the geometry tracks the dust ──
def _spr(x, y):
    """Spearman with a joint finite mask -> (rho, p, n)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 5:
        return np.nan, np.nan, int(m.sum())
    r, p = spearmanr(x[m], y[m]); return r, p, int(m.sum())

_GEOM_PLOT = [("hole_H2", r"$1-\Sigma_{\rm H_2}^{\rm in}/\Sigma_{\rm H_2}^{\rm peak}$ (hole)", "up"),
              ("Rpeak_H2", r"$R_{\rm peak}^{\rm H_2}$ [kpc]", "up"),
              ("R80_H2",   r"$R_{80}^{\rm H_2}$ [kpc]", "up"),
              ("Rmw_H2",   r"$\langle R\rangle_{\rm H_2}$ [kpc]", "up"),
              ("Sig_in_dust", r"$\Sigma_{\rm dust}^{\rm in}$", "down"),
              ("dust_core_frac", r"dust core frac", "down"),
              ("C2080_dust", r"$R_{20}/R_{80}$ dust", "up")]
xs = POOL["xstr_post"]; dusty = POOL["dusty"].astype(bool)
ncol = 4; nrow = int(np.ceil(len(_GEOM_PLOT) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 3.6 * nrow)); axs = np.atleast_1d(axs).ravel()
for ax, (k, lab, pred) in zip(axs, _GEOM_PLOT):
    y = GEOM[k]; m = np.isfinite(xs) & np.isfinite(y)
    ax.scatter(xs[m & ~dusty], y[m & ~dusty], s=8, c="0.6", alpha=0.5, label="non-dusty")
    ax.scatter(xs[m & dusty],  y[m & dusty],  s=10, c="C3", alpha=0.7, label="dusty")
    r, p, n = _spr(xs, y)
    ok = ("✓" if (r > 0) == (pred == "up") else "✗") if np.isfinite(r) else "?"
    ax.set_title(f"{lab}\nρ={r:+.2f} p={p:.1g} n={n}  (pred {pred} {ok})", fontsize=8)
    ax.set_xlabel(r"$x_{\rm coup}$ (xstr_post)")
for ax in axs[len(_GEOM_PLOT):]:
    ax.axis("off")
axs[0].legend(fontsize=7)
fig.suptitle("6e·geom — H2 geometry & central dust vs coupling strength @ gas_min (pooled)", y=1.0)
plt.tight_layout(); plt.savefig(figpath("sigma_geom_vs_xstrength.png"), dpi=130, bbox_inches="tight"); plt.show()

# the keystone: does the H2 hole / extent co-vary with central-dust removal and the dust budget?
print("\n[6e·geom] geometry ↔ dust correlations (pooled, well-behaved):")
for lab, a, b in [("hole_H2 vs M_dust/M* (gas_min)", GEOM["hole_H2"], POOL["dustfrac_gasmin"]),
                  ("Sig_in_dust vs M_dust/M*",       GEOM["Sig_in_dust"], POOL["dustfrac_gasmin"]),
                  ("dust_core_frac vs M_dust/M*",     GEOM["dust_core_frac"], POOL["dustfrac_gasmin"]),
                  ("R80_H2 vs Sig_in_dust",           GEOM["R80_H2"], GEOM["Sig_in_dust"]),
                  ("hole_H2 vs Sig_in_dust",          GEOM["hole_H2"], GEOM["Sig_in_dust"])]:
    r, p, n = _spr(a, b); print(f"  {lab:34s}: ρ={r:+.2f}  p={p:.2g}  (n={n})")

# strong vs weak scalar contrast (Mann-Whitney), medians + verdict
print("\n[6e·geom] strong vs weak (Mann-Whitney, pooled):")
S = POOL["strong"].astype(bool); W = POOL["weak"].astype(bool)
for k, lab, pred in _GEOM_PLOT:
    a = GEOM[k][S]; a = a[np.isfinite(a)]; b = GEOM[k][W]; b = b[np.isfinite(b)]
    if len(a) < 3 or len(b) < 3:
        print(f"  {k:16s}: n too small (strong {len(a)}, weak {len(b)})"); continue
    pp = mannwhitneyu(a, b).pvalue
    verdict = "higher" if np.median(a) > np.median(b) else "lower"
    exp = "higher" if pred == "up" else "lower"
    print(f"  {k:16s}: strong med {np.median(a):+.3g} vs weak {np.median(b):+.3g}  "
          f"-> strong {verdict} (exp {exp})  p={pp:.2g}")

In [ ]:
# ── 6e·geom·c — median Σ(R) at gas_min, strong vs weak (H2 then dust); from the reduced Σ cache ──
def _med_band_floor_local(A, band=(16, 84)):
    """Median + percentile band of log10 Σ INCLUDING empty annuli (Σ=0 -> floor), plus the
    detected fraction (Σ>0 among galaxies present at each R). Mirrors 6t's _med_band_floor."""
    posv = A[A > 0]
    floor = float(np.floor(np.log10(posv.min())) - 0.5) if posv.size else 0.0
    out = np.full(A.shape, np.nan, float)
    out[A > 0] = np.log10(A[A > 0]); out[A == 0] = floor
    with np.errstate(all="ignore"):
        med = np.nanmedian(out, axis=0)
        lo = np.nanpercentile(out, band[0], axis=0); hi = np.nanpercentile(out, band[1], axis=0)
    present = np.isfinite(A).sum(0); ndet = (A > 0).sum(0)
    frac = np.where(present > 0, ndet / np.maximum(present, 1), np.nan)
    return med, lo, hi, frac, floor

def median_profile_gasmin(z, comp="H2", fname=None):
    B = RESULTS.get(z)
    if B is None or not B["RP"]["OK"] or "gas_min" not in B["STAGES"] or not PROFILE_CACHE:
        print(f"[z={z}] no reduced gas_min profiles"); return
    PIDX = _pidx(z); snaps = B["snaps_arr"]; rmid = PROF_RMID
    wb = B["well_behaved"]; CO = B["CO"]
    groups = [("strong", np.asarray(CO["strong"], bool) & wb, "C3"),
              ("weak",   np.asarray(CO["weak"],   bool) & wb, "C0")]
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
    for ax, (lab, mask, c) in zip(axs, groups):
        stack = []
        for i in np.where(mask)[0]:
            row = B["records"][i].get("row_gas_min", -1)
            if row is None or row < 0:
                continue
            g = PIDX[row, B["records"][i]["col"]]
            if not np.isfinite(g):
                continue
            p = PROFILE_CACHE.get((int(snaps[row]), int(g)), {}).get(comp)
            if p is not None:
                stack.append(p)
        A = np.vstack(stack) if stack else np.empty((0, len(rmid)))
        if A.shape[0] < 3:
            ax.set_title(f"{lab} (n={A.shape[0]} — too few)"); continue
        med, lo, hi, frac, floor = _med_band_floor_local(A)
        ax.plot(rmid, med, "-", color=c, lw=2, label=f"{lab} (n={A.shape[0]})")
        ax.fill_between(rmid, lo, hi, color=c, alpha=0.15, lw=0)
        ax.axhline(floor, ls=":", c="0.6", lw=0.8)
        axd = ax.twinx(); axd.plot(rmid, frac, ":", color=c, alpha=0.6, lw=1.2)
        axd.set_ylim(0, 1.05); axd.set_ylabel("frac Σ>0", fontsize=7); axd.tick_params(labelsize=6)
        ax.set_xscale("log"); ax.set_xlim(PROF_RMIN * 0.6, PROF_RVIEW)
        ax.set_xlabel("R [kpc]"); ax.set_title(lab); ax.legend(fontsize=8, loc="upper right")
    axs[0].set_ylabel(fr"$\log_{{10}}\,\Sigma_{{\rm {comp}}}\ [M_\odot\,{{\rm kpc}}^{{-2}}]$")
    fig.suptitle(f"z={z}: median $\\Sigma_{{\\rm {comp}}}(R)$ at gas_min — strong vs weak "
                 f"(16–84% band; dotted = fraction with Σ>0)", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

for z in RESULTS:
    if RESULTS[z]["BH"] is None:
        print(f"[z={z}] no BH -> no strong/weak split; skip median profiles."); continue
    for comp in ("H2", "dust"):
        median_profile_gasmin(z, comp=comp, fname=f"sigma_geomprof_{comp}_z{_ztag(z)}.png")

### 6f — Long after quenching: composition-immune ΔX (ref0), strong vs weak

`ref0=True` subtracts each galaxy's value at the trigger, so the curve is the *change* relative to
QT — immune to the changing-cohort offset. The weak-coupling prediction: more persistent residual
dust (shallower decline) and slower compaction at large `t − t_QT`.

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling figure skipped (set BUILD_BH=True)."); continue
    print(f"\n=== z = {z} : Δ(after QT), strong vs weak ===")
    event_stack([p for p in ["dust/M*", "Rdust/Rstar", "R20g/R80g"] + SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=["t_QT", "post_quench"], split=_wb_split(SPLIT_XS), ref0=True,
                xwin=4.0, nb=12, fname=f"postqt_delta_z{_ztag(z)}.png")

### 6g — Is gas surface density a better tracer than quenching time? (falsifiable head-to-head)

Pooled: Spearman |r| (vs continuous coupling `xstr_post`) and AUC (separating fast/slow) for
`τ_q` vs `Σ_H2` measured at QT / post_quench. Plus event-aligned Σ tracks by coupling. Σ beats τ_q
only if it shows larger |r| *and* larger |AUC−0.5|.

In [ ]:
def _spearman(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 8:
        return np.nan, np.nan, int(m.sum())
    r, p = spearmanr(a[m], b[m]); return r, p, int(m.sum())

def _auc(feature, pos, neg):
    a = feature[pos.astype(bool)]; a = a[np.isfinite(a)]
    b = feature[neg.astype(bool)]; b = b[np.isfinite(b)]
    if len(a) < 5 or len(b) < 5:
        return np.nan
    U, _ = mannwhitneyu(a, b, alternative="two-sided"); return U / (len(a) * len(b))

feat = {"log tau_q/t_H":      np.log10(np.where(POOL["tau_q_over_tH"] > 0, POOL["tau_q_over_tH"], np.nan)),
        "log f_gas @QT":      np.log10(np.where(POOL["fgas_qt"]    > 0, POOL["fgas_qt"],    np.nan)),
        "log f_gas @postQ":   np.log10(np.where(POOL["fgas_postq"] > 0, POOL["fgas_postq"], np.nan)),
        "log Sigma_H2 @QT":   np.log10(np.where(POOL["SigH2_qt"]   > 0, POOL["SigH2_qt"],   np.nan)),
        "log Sigma_H2 @postQ": np.log10(np.where(POOL["SigH2_postq"] > 0, POOL["SigH2_postq"], np.nan))}
print(f"{'feature':>22s} {'|r| vs coupling':>16s} {'AUC fast/slow':>14s} {'|AUC-0.5|':>10s}")
for name, f in feat.items():
    r, p, n = _spearman(f, POOL["xstr_post"]); auc = _auc(f, POOL["is_fast"], POOL["is_slow"])
    print(f"{name:>22s} {abs(r):16.3f} {auc if np.isfinite(auc) else float('nan'):14.3f} "
          f"{abs(auc - 0.5) if np.isfinite(auc) else float('nan'):10.3f}   (r p={p:.1g}, n={n})")
print("Geometry-over-budget claim holds if Sigma_H2 shows larger |r| AND larger |AUC-0.5| than")
print("BOTH tau_q (timescale) and f_gas (budget). f_gas is the foil the introduction names.")
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None or not RP_OK:
        continue
    event_stack([p for p in SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=["t_QT", "post_quench"], split=_wb_split(SPLIT_XS), xwin=3.0, nb=10,
                fname=f"sigma_xstrength_z{_ztag(z)}.png")

### 6g·matched — Geometry at *fixed budget* (the keystone of the reframe)

The head-to-head above lets Σ and f_gas compete freely. This cell runs the harder test the
introduction actually promises: **hold the gas budget fixed and ask whether the geometry still
tells the fate.** Three complementary cuts, pooled across anchors, with Σ and f_gas measured at QT:

1. **Partial Spearman** — corr(Σ_H2, coupling | f_gas) vs corr(f_gas, coupling | Σ_H2). If the
   first survives and the second collapses toward 0, the budget's apparent link to *mode* is
   *mediated* by geometry.
2. **Budget-matched AUC** — restrict to the f_gas window populated by *both* fast and slow, then
   recompute the fast/slow AUC for Σ_H2 and for f_gas. Σ should keep its separating power inside
   the window; f_gas should fall toward 0.5 in its own matched range.
3. **Matched dusty/non-dusty pairs** — for each dusty-at-gas_min galaxy, its nearest non-dusty
   neighbour in log f_gas (|Δ| < tol); a paired test on Σ_H2 asks whether, *at the same budget*,
   the dustier fate is the higher-Σ one. This is "identical gas fraction, opposite future" made literal.

In [ ]:
# ── 6g·matched — does geometry (Sigma_H2) carry the fate when the budget (f_gas) is held fixed? ──
from scipy.stats import rankdata, wilcoxon

def _partial_spearman(x, y, ctrl):
    """Spearman(x, y) controlling linearly for `ctrl`, computed on ranks (scipy-only)."""
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(ctrl)
    if m.sum() < 10:
        return np.nan, np.nan, int(m.sum())
    rx, ry, rc = rankdata(x[m]), rankdata(y[m]), rankdata(ctrl[m])
    C = np.column_stack([rc, np.ones(int(m.sum()))])
    ex = rx - C @ np.linalg.lstsq(C, rx, rcond=None)[0]
    ey = ry - C @ np.linalg.lstsq(C, ry, rcond=None)[0]
    r, p = spearmanr(ex, ey)
    return r, p, int(m.sum())

STAGE_TAG = "QT"
# geometry axis: Sigma_H2 inside the corrected 100 kpc aperture (reduced particles) at QT;
# falls back to the catalog ISM-only (gal.glist) Sigma if the reduced files are absent. Tunable.
SIG_APER_KPC, SIG_GEOM, SIG_REGION = 10.0, "cyl", "all"
_have_reduced = (os.path.isdir(REDUCED_DIR)
                 and any(f.endswith(".h5") for _d, _s, _fs in os.walk(REDUCED_DIR) for f in _fs))
if _have_reduced:
    _Tq, _nq = sigma_table_reduced("qt", r_aper_kpc=SIG_APER_KPC, geom=SIG_GEOM, region=SIG_REGION)
    SigH2_qt = _Tq["Sigma_H2_eff"]; SIG_SRC = f"reduced {SIG_APER_KPC:g}kpc {SIG_REGION}"
else:
    SigH2_qt = POOL["SigH2_qt"]; SIG_SRC = "catalog ISM (gal.glist)"
    print(f"[6g.matched] reduced files absent -> Sigma from {SIG_SRC}.")
print(f"[6g.matched] geometry axis = Sigma_H2 @QT from {SIG_SRC}\n")
lSig = np.log10(np.where(SigH2_qt > 0, SigH2_qt, np.nan))
lfg  = np.log10(np.where(POOL["fgas_qt"]  > 0, POOL["fgas_qt"],  np.nan))
coup = POOL["xstr_post"]
fast = POOL["is_fast"].astype(bool); slow = POOL["is_slow"].astype(bool)

# 1 — partial correlations vs continuous coupling
rs, ps, ns = _partial_spearman(lSig, coup, lfg)    # geometry, budget held fixed
rf, pf, nf = _partial_spearman(lfg,  coup, lSig)   # budget,  geometry held fixed
print(f"[1] partial Spearman vs coupling @ {STAGE_TAG} (n={ns}):")
print(f"      r(Sigma_H2, coupling | f_gas) = {rs:+.3f} (p={ps:.1g})   <- geometry, budget fixed")
print(f"      r(f_gas,    coupling | Sigma) = {rf:+.3f} (p={pf:.1g})   <- budget, geometry fixed")
print(f"      -> reframe supported if |{rs:+.2f}| stays large while |{rf:+.2f}| collapses toward 0.\n")

# 2 — fast/slow AUC inside the f_gas window common to both classes
def _auc_fs(feature, mask):
    a = feature[fast & mask]; a = a[np.isfinite(a)]
    b = feature[slow & mask]; b = b[np.isfinite(b)]
    if len(a) < 5 or len(b) < 5:
        return np.nan, len(a), len(b)
    U, _ = mannwhitneyu(a, b, alternative="two-sided"); return U / (len(a) * len(b)), len(a), len(b)

fin = np.isfinite(lfg)
all_mask = np.ones(len(lfg), bool)
lo = max(np.nanpercentile(lfg[fast & fin], 10), np.nanpercentile(lfg[slow & fin], 10))
hi = min(np.nanpercentile(lfg[fast & fin], 90), np.nanpercentile(lfg[slow & fin], 90))
band = fin & (lfg >= lo) & (lfg <= hi)
aS_all, _, _ = _auc_fs(lSig, all_mask); aS_m, na, nb = _auc_fs(lSig, band)
aF_all, _, _ = _auc_fs(lfg,  all_mask); aF_m, _, _   = _auc_fs(lfg,  band)
print(f"[2] fast/slow AUC, full vs budget-matched log f_gas in [{lo:.2f},{hi:.2f}] (n_fast={na}, n_slow={nb}):")
print(f"      Sigma_H2 : AUC {aS_all:.3f} -> {aS_m:.3f}   |AUC-0.5|: {abs(aS_all-0.5):.3f} -> {abs(aS_m-0.5):.3f}")
print(f"      f_gas    : AUC {aF_all:.3f} -> {aF_m:.3f}   |AUC-0.5|: {abs(aF_all-0.5):.3f} -> {abs(aF_m-0.5):.3f}")
print(f"      -> reframe supported if Sigma_H2 keeps |AUC-0.5| inside the matched window.\n")

# 3 — matched dusty/non-dusty pairs at fixed f_gas: paired Sigma_H2 comparison
TOL = 0.15   # max |Delta log f_gas| to call two galaxies budget-matched [dex]
dusty = POOL["dusty"]; idx = np.arange(len(lfg))
src = idx[dusty & np.isfinite(lfg) & np.isfinite(lSig)]
pool_nd = idx[(~dusty) & np.isfinite(lfg) & np.isfinite(lSig)]
dS, used = [], 0
for i in src:
    cand = pool_nd[np.abs(lfg[pool_nd] - lfg[i]) <= TOL]
    if len(cand) == 0:
        continue
    j = cand[np.argmin(np.abs(lfg[cand] - lfg[i]))]
    dS.append(lSig[i] - lSig[j]); used += 1
dS = np.asarray(dS)
print(f"[3] budget-matched dusty/non-dusty pairs (|Delta log f_gas|<={TOL} dex): "
      f"{used} of {len(src)} dusty galaxies matched")
if used >= 6:
    med = float(np.median(dS))
    try:
        _, pw = wilcoxon(dS)
    except ValueError:
        pw = float("nan")
    print(f"      median Delta log Sigma_H2 (dusty - matched non-dusty) = {med:+.2f} dex (Wilcoxon p={pw:.1g})")
    print(f"      -> 'same budget, opposite fate' holds if dusty sit at HIGHER Sigma (Delta>0, p small).")
else:
    print("      too few matched pairs for a paired test (widen TOL or check DUST_THR).")

# scatter: budget (x) vs geometry (y), fate by colour, matched window shaded
fig, ax = plt.subplots(figsize=(6.2, 5.2))
ax.axvspan(lo, hi, color="0.9", zorder=0, label="budget-matched window")
n_slow = int(np.isfinite(lfg[slow] + lSig[slow]).sum())
n_fast = int(np.isfinite(lfg[fast] + lSig[fast]).sum())
ax.scatter(lfg[slow], lSig[slow], s=20, c="C0", alpha=0.7, edgecolor="none", label=f"slow (n={n_slow})")
ax.scatter(lfg[fast], lSig[fast], s=20, c="C3", alpha=0.7, edgecolor="none", label=f"fast (n={n_fast})")
ax.set_xlabel(r"$\log_{10} f_{\rm gas}$ @ QT  (budget)")
ax.set_ylabel(r"$\log_{10}\,\Sigma_{\rm H_2}$ @ QT  (geometry)")
ax.set_title(f"6g.matched - geometry ({SIG_SRC}) separates fate at fixed budget")
ax.legend(fontsize=8, loc="best")
plt.tight_layout(); plt.savefig(figpath("sigma_vs_fgas_matched.png"), dpi=130, bbox_inches="tight"); plt.show()

### 6h — Full SFH per sample + cross-anchor lead-lag synthesis

Full star-formation histories on the critical-point clock by coupling, and the pooled causal
ordering `t_QT − t_AGN` (does AGN ignition precede quenching, more so for strong coupling?).

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        continue
    sfh_shape(anchors=("sf_peak", "qt", "post_quench"), kind="ssfr",
              split=_wb_split(SPLIT_XS), fname=f"sfh_ssfr_xstrength_z{_ztag(z)}.png")

fig, ax = plt.subplots(figsize=(8, 5))
for msk, c, lab in [(POOL["strong"], "C3", "strong"), (POOL["weak"], "C0", "weak"),
                    (POOL["no_agn"], "0.5", "no-AGN")]:
    v = POOL["lag_qt_agn"][msk.astype(bool)]; v = v[np.isfinite(v)]
    if len(v) < 3:
        continue
    ax.hist(v, bins=20, density=True, histtype="step", lw=2, color=c,
            label=f"{lab} (med {np.median(v):+.2f}, n={len(v)})")
ax.axvline(0, ls=":", c="k"); ax.set_xlabel(r"$t_{\rm QT}-t_{\rm AGN}$ [Gyr]  (+ = AGN leads quench)")
ax.set_ylabel("pdf"); ax.legend(); ax.set_title("6h — causal ordering, pooled across anchors")
s = POOL["lag_qt_agn"][POOL["strong"].astype(bool)]; w = POOL["lag_qt_agn"][POOL["weak"].astype(bool)]
s = s[np.isfinite(s)]; w = w[np.isfinite(w)]
if len(s) > 2 and len(w) > 2:
    print(f"lead-lag strong vs weak: MW-p = {mannwhitneyu(s, w).pvalue:.2g} "
          f"(median strong {np.median(s):+.2f}, weak {np.median(w):+.2f} Gyr)")
plt.tight_layout(); plt.savefig(figpath("synthesis_leadlag.png"), dpi=130, bbox_inches="tight"); plt.show()

### 6h·gasmin — cumulative sSFR after gas_min: dusty / non-dusty × coupling

Galaxies that are **dusty at `gas_min`** (a surviving central reservoir) should resume star
formation afterward, while non-dusty ones stay flat. We (1) stack SFR on the `gas_min` clock split
by dustiness, and (2) measure each galaxy's **cumulative sSFR** — ∫ sSFR dt over `CUM_DT_GYR` after
`gas_min` (a mass-fraction-formed proxy, easier to characterise than a slope) — and compare it
across dustiness and coupling. Each redshift gets one bar plot showing the **marginal** classes
(just dusty / just non-dusty / just strong / just weak) **and** their four combinations. (The
environment control is 6e·sat.)

In [ ]:
# ── 6h·gasmin·a — SFR stacked on the gas_min clock, split by dust-at-gas_min ──
for z in RESULTS:
    B = activate_anchor(z)
    sfh_shape(anchors=("gas_min",), kind="sfr",
              split=_wb_split(dust_split(thresh=DUST_THR, stage="gas_min")),
              xwin=(-2.0, 4.0), nx=61, min_n=5,
              fname=f"sfh_sfr_gasmin_dusty_z{_ztag(z)}.png")

In [ ]:
# ── 6h·gasmin·b — cumulative sSFR after gas_min; pooled CSSFR (for 6e·sat) + per-z category bars ──
CUM_DT_GYR = 1.0      # integrate sSFR over [t_gas_min, t_gas_min + CUM_DT_GYR]

def cumulative_ssfr(B, i, dt_gyr=CUM_DT_GYR):
    """∫ sSFR(t) dt over [t_gas_min, t_gas_min+dt] (dimensionless mass-fraction-formed proxy).
    Requires the window to be fully covered by the history; else NaN (fair across galaxies)."""
    r = B["records"][i]; t0 = r.get("t_gas_min", np.nan)
    if not np.isfinite(t0):
        return np.nan
    col = r["col"]; t = np.asarray(B["t_cosmic_yr"], float)
    sfr = B["P"]["sfr"][:, col].astype(float); mst = B["P"]["masses.stellar"][:, col].astype(float)
    with np.errstate(all="ignore"):
        ssfr = np.where(mst > 0, sfr / mst, np.nan)
    good = np.isfinite(ssfr) & np.isfinite(t)
    if good.sum() < 2:
        return np.nan
    ts = t[good]; ss = ssfr[good]; o = np.argsort(ts); ts, ss = ts[o], ss[o]
    t1 = t0 + dt_gyr * 1e9
    if ts[0] > t0 or ts[-1] < t1:            # window not fully covered -> unfair to include
        return np.nan
    grid = np.linspace(t0, t1, 60); vals = np.interp(grid, ts, ss)
    return float(np.trapz(vals, grid))

CSSFR = []
for z in RESULTS:
    B = RESULTS[z]; wb = B["well_behaved"]
    s = np.array([cumulative_ssfr(B, i) for i in range(len(B["records"]))])
    CSSFR.append(s[wb])
CSSFR = np.concatenate(CSSFR) if CSSFR else np.array([])
_expected = int(sum(int(np.asarray(RESULTS[z]["well_behaved"]).sum()) for z in RESULTS))
assert len(CSSFR) == _expected, f"CSSFR ({len(CSSFR)}) != well-behaved records ({_expected})."
if len(POOL["z"]) != _expected:
    print(f"[6h·gasmin][warn] POOL ({len(POOL['z'])}) is STALE vs RESULTS ({_expected}); re-run the POOL cell.")
print(f"[6h·gasmin] cumulative sSFR over {CUM_DT_GYR:g} Gyr: {int(np.isfinite(CSSFR).sum())} finite of {len(CSSFR)}.")

def _med_iqr(v):
    v = v[np.isfinite(v)]
    if not len(v):
        return np.nan, 0.0, 0.0, 0
    md = np.median(v)
    return md, md - np.percentile(v, 25), np.percentile(v, 75) - md, len(v)

for z in RESULTS:
    zm = POOL["z"] == z
    cs = CSSFR[zm]
    dusty = POOL["dusty"][zm].astype(bool)
    S = POOL["strong"][zm].astype(bool); W = POOL["weak"][zm].astype(bool)
    cats = [("dusty", dusty, "C3"), ("non-dusty", ~dusty, "0.5"),
            ("strong", S, "C2"), ("weak", W, "C9"),
            ("strong\ndusty", S & dusty, "C3"), ("strong\nnon-dusty", S & ~dusty, "C1"),
            ("weak\ndusty", W & dusty, "C0"), ("weak\nnon-dusty", W & ~dusty, "0.5")]
    meds, los, his, ns, cols, labs = [], [], [], [], [], []
    for lab, mask, c in cats:
        md, lo, hi, nn = _med_iqr(cs[mask])
        meds.append(md); los.append(lo); his.append(hi); ns.append(nn); cols.append(c); labs.append(lab)
    fig, ax = plt.subplots(figsize=(11, 4.6))
    x = np.arange(len(cats))
    ax.bar(x, meds, yerr=[los, his], color=cols, alpha=0.85, capsize=4)
    ax.axvline(3.5, ls=":", c="0.5", lw=1)     # marginals (left) | combinations (right)
    for xi, md, nn in zip(x, meds, ns):
        ax.text(xi, (md if np.isfinite(md) else 0.0), f"n={nn}", ha="center", va="bottom", fontsize=7)
    ax.axhline(0, ls=":", c="k")
    ax.set_xticks(x); ax.set_xticklabels(labs, fontsize=8)
    ax.set_ylabel(fr"median $\int$sSFR$\,$dt over {CUM_DT_GYR:g} Gyr after gas_min")
    ax.set_title(f"6h·gasmin (z={z}) — cumulative sSFR after gas_min by dustiness & coupling")
    plt.tight_layout()
    plt.savefig(figpath(f"sfh_postgasmin_cumssfr_z{_ztag(z)}.png"), dpi=130, bbox_inches="tight"); plt.show()

### 6e·sat — Number of satellites at the critical point vs cumulative sSFR

Simple environmental check: count the **galaxies in the vicinity** of each target at the chosen
critical point (`SAT_STAGE`, default `gas_min`) — every *other* catalogue galaxy that is **in the
same parent halo OR within 100 kpc (physical)** — and plot that count against the **cumulative sSFR
after gas_min** (∫ sSFR dt over `CUM_DT_GYR`, from 6h). If the cumulative sSFR does not grow with the
number of satellites, the late star formation is internal, not driven by neighbours/minor mergers.
The per-bin view is split by the marginal classes (all / dusty / non-dusty / strong / weak).

Gated behind `BUILD_SAT_CENSUS` (Part 0): catalogue I/O, run once on the cluster; the build prints a
median-count diagnostic, then the plot cell reads the cache.

In [ ]:
# ── 6e·sat·a — count neighbour galaxies at the critical point (same halo OR <100 kpc); gated ──
SAT_STAGE = "gas_min"        # critical point at which to count satellites
SAT_R_KPC = 100.0            # physical aperture [kpc]
SAT_HUB   = 0.68             # SIMBA HubbleParam (catalog pos are comoving Mpc/h)
SAT_KEYS  = ["n_sat", "n_samehalo", "n_near", "nearest_kpc"]
_SAT_BOXMAP = {"cis100": 100.0, "cis50": 50.0, "cis50nox": 50.0, "cis25": 25.0}
SAT_BOX = _SAT_BOXMAP.get(SIM_NAME, float(getattr(sim, "box", BOX) or BOX))   # comoving Mpc/h, min-image

def sat_path(A):
    tag = A["tag"] if A.get("tag") else "z0"
    return os.path.join(TABLEDIR, f"sat_count_{SAT_STAGE}_{tag}.hdf5")

def _read_cat(snap):
    """Galaxy positions (comoving Mpc/h) + parent-halo index from the Caesar HDF5 (same read as
    build_prog_index). parent_halo_index absent -> -1 (same-halo count then 0)."""
    with h5py.File(sim.get_caesar_file(snap), "r") as f:
        pos = f["galaxy_data/pos"][:]
        ph = f.get("galaxy_data/parent_halo_index")
        phalo = ph[:].astype(np.int64) if ph is not None else np.full(len(pos), -1, np.int64)
    return pos, phalo

def _count_sats(pos, phalo, gi, a, box, r_kpc=SAT_R_KPC, hub=SAT_HUB):
    n = len(pos)
    if gi < 0 or gi >= n:
        return None
    d = pos - pos[gi]; d -= box * np.round(d / box)                 # min-image (comoving Mpc/h)
    dist_kpc = np.sqrt((d ** 2).sum(1)) * 1000.0 * a / hub          # -> physical kpc
    others = np.arange(n) != gi
    samehalo = others & (phalo >= 0) & (phalo == phalo[gi])
    near = others & (dist_kpc < r_kpc)
    sel = samehalo | near
    return dict(n_sat=int(sel.sum()), n_samehalo=int(samehalo.sum()), n_near=int(near.sum()),
                nearest_kpc=float(dist_kpc[others].min()) if others.any() else np.nan)

if BUILD_SAT_CENSUS:
    rowkey = f"row_{SAT_STAGE}"
    for z in RESULTS:
        B = RESULTS[z]; A = B["A"]; recs = B["records"]; snaps = B["snaps_arr"]; zrow = B["redshift"]
        PIDX = _pidx(z); n = len(recs)
        OUT_ = {k: np.full(n, np.nan) for k in SAT_KEYS}
        bysnap = {}
        for i, r in enumerate(recs):
            row = r.get(rowkey, -1)
            if row is None or row < 0:
                continue
            g = PIDX[row, r["col"]]
            if not np.isfinite(g):
                continue
            s = int(snaps[row])
            bysnap.setdefault(s, {"a": 1.0 / (1.0 + float(zrow[row])), "items": []})["items"].append((i, int(g)))
        no_phalo = 0
        for snap, info in sorted(bysnap.items()):
            try:
                pos, phalo = _read_cat(snap)
            except (OSError, KeyError) as e:
                print(f"  [{A['tag']}] snap {snap}: catalog read failed ({e}); skipping {len(info['items'])}"); continue
            if np.all(phalo < 0):
                no_phalo += 1
            for i, g in info["items"]:
                res = _count_sats(pos, phalo, g, info["a"], SAT_BOX)
                if res:
                    for k in SAT_KEYS:
                        OUT_[k][i] = res[k]
        with h5py.File(sat_path(A), "w") as o:
            o.attrs.update(dict(z=float(z), sim_name=SIM_NAME, stage=SAT_STAGE,
                                r_kpc=SAT_R_KPC, box=SAT_BOX, hub=SAT_HUB))
            o.create_dataset("gid", data=np.array([r["gid"] for r in recs], np.int64))
            for k in SAT_KEYS:
                o.create_dataset(k, data=OUT_[k].astype("f8"))
        ns = OUT_["n_sat"]; fin = np.isfinite(ns)
        print(f"[{A['tag']}] sat count @ {SAT_STAGE} -> {os.path.basename(sat_path(A))}: "
              f"{int(fin.sum())} galaxies, median n_sat={np.nanmedian(ns):.0f}, "
              f"max={np.nanmax(ns) if fin.any() else 0:.0f}"
              + (f"  [WARN: {no_phalo} snaps had no parent_halo_index]" if no_phalo else ""))
else:
    _missing = [os.path.basename(sat_path(RESULTS[z]["A"])) for z in RESULTS
                if not os.path.exists(sat_path(RESULTS[z]["A"]))]
    print("BUILD_SAT_CENSUS=False.",
          ("all caches present." if not _missing else f"missing: {_missing} (set the flag, run on cluster)."))

In [ ]:
# ── 6e·sat·b — number of satellites at the critical point vs cumulative sSFR after gas_min ──
def load_sat(B):
    p = sat_path(B["A"]); recs = B["records"]; n = len(recs)
    if not os.path.exists(p):
        return {k: np.full(n, np.nan) for k in SAT_KEYS}
    with h5py.File(p, "r") as f:
        arrs = {k: f[k][:] for k in SAT_KEYS}; cg = f["gid"][:]
    cur = np.array([r["gid"] for r in recs])
    arrs, _ = _align_records_by_gid(arrs, cg, cur, name="sat count")
    return arrs

SAT = {k: [] for k in SAT_KEYS}
_have_sat = all(os.path.exists(sat_path(RESULTS[z]["A"])) for z in RESULTS)
for z in RESULTS:
    B = RESULTS[z]; wb = B["well_behaved"]; s = load_sat(B)
    for k in SAT_KEYS:
        SAT[k].append(np.asarray(s[k])[wb])
SAT = {k: (np.concatenate(v) if v else np.array([])) for k, v in SAT.items()}

if not _have_sat or not np.isfinite(SAT["n_sat"]).any():
    print("[6e·sat] no count cache yet — set BUILD_SAT_CENSUS=True (Part 0) and run on the cluster.")
else:
    _expected = int(sum(int(np.asarray(RESULTS[z]["well_behaved"]).sum()) for z in RESULTS))
    assert len(SAT["n_sat"]) == _expected, f"SAT ({len(SAT['n_sat'])}) != well-behaved records ({_expected})."
    if len(POOL["z"]) != _expected:
        print(f"[6e·sat][warn] POOL ({len(POOL['z'])}) is STALE vs RESULTS ({_expected}); re-run the POOL cell.")
    nsat = SAT["n_sat"]
    print(f"[6e·sat] {int(np.isfinite(nsat).sum())} galaxies with a satellite count "
          f"(median n_sat={np.nanmedian(nsat):.0f}: same-halo {np.nanmedian(SAT['n_samehalo']):.0f}, "
          f"<100kpc {np.nanmedian(SAT['n_near']):.0f}).")
    cssfr = CSSFR if ("CSSFR" in globals() and len(CSSFR) == len(POOL["z"])) else np.full(len(nsat), np.nan)
    _dt = CUM_DT_GYR if "CUM_DT_GYR" in globals() else 1.0
    dusty = POOL["dusty"].astype(bool); S = POOL["strong"].astype(bool); W = POOL["weak"].astype(bool)
    m = np.isfinite(nsat) & np.isfinite(cssfr)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
    # (1) scatter: number of satellites vs cumulative sSFR
    ax = axs[0]
    ax.scatter(nsat[m], cssfr[m], s=12, c="C0", alpha=0.5)
    ax.set_xlabel("number of satellites @ %s  (same halo or <100 kpc)" % SAT_STAGE)
    ax.set_ylabel(fr"$\int$sSFR$\,$dt over {_dt:g} Gyr after gas_min")
    ax.set_title("cumulative sSFR vs satellite count")
    # (2) median cumulative sSFR vs satellite-count bin, by marginal class
    ax = axs[1]
    edges = [-0.5, 0.5, 1.5, 2.5, 4.5, np.inf]; binlab = ["0", "1", "2", "3-4", "≥5"]; xc = np.arange(len(binlab))
    groups = [("all", np.ones(len(nsat), bool), "k"), ("dusty", dusty, "C3"), ("non-dusty", ~dusty, "0.5"),
              ("strong", S, "C2"), ("weak", W, "C9")]
    for lab, base, c in groups:
        med = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            sel = base & (nsat >= lo) & (nsat < hi) & np.isfinite(cssfr)
            med.append(np.median(cssfr[sel]) if sel.sum() >= 3 else np.nan)
        ax.plot(xc, med, "-o", color=c, label=lab, alpha=0.85)
    ax.axhline(0, ls=":", c="k"); ax.set_xticks(xc); ax.set_xticklabels(binlab)
    ax.set_xlabel("number of satellites @ %s" % SAT_STAGE)
    ax.set_ylabel(fr"median $\int$sSFR$\,$dt ({_dt:g} Gyr)")
    ax.set_title("by marginal class"); ax.legend(fontsize=8)
    fig.suptitle("6e·sat — satellite count vs cumulative sSFR after gas_min (pooled)", y=1.02)
    plt.tight_layout(); plt.savefig(figpath("sigma_satcensus_gasmin.png"), dpi=130, bbox_inches="tight"); plt.show()

### 6e·sat·halo — Halo-membership satellites & gas reservoirs vs cumulative sSFR

Instead of a fixed 100 kpc aperture (6e·sat), define **satellites as the other galaxies sharing the
target's parent halo** (`galaxy_data/parent_halo_index`; §7a-style catalogue census, after the
Resolving_feedbacks environment part). At the critical point we record, per target: the number of
halo satellites, the **molecular gas of the target** and **of its satellites** (Σ member H2 −
target), and the halo's **cold** (HI+H2) and **warm/hot** (gas − cold) reservoirs (§7b-style, from
`halo_data/dicts/masses.*`). We then plot the **number of halo satellites** ($N_{\rm sat}$, common y-axis) against each property
(post-gas_min cumulative sSFR, target **H₂ fraction** $M_{\rm H_2}/M_\star$, satellite H₂, and the
halo cold/warm-hot gas), **one figure per halo-mass bin** — asking whether richer environments track
the target's own molecular reservoir, the halo gas phases, or the late star formation.

Gated behind `BUILD_SAT_CENSUS` (Part 0); catalogue-only (one Caesar file per snapshot), run once on
the cluster.

In [ ]:
# ── 6e·sat·halo·a — halo-membership satellites + halo gas phases at the critical point (gated) ──
# Satellites = OTHER galaxies sharing the target's parent halo (galaxy_data/parent_halo_index), NOT a
# fixed aperture. Also records the target's molecular gas, the satellites' molecular gas, and the
# halo's cold (HI+H2) and warm/hot (gas-cold) reservoirs — all catalog-only (§7a/§7b style).
SATH_KEYS = ["n_sat", "is_central", "targ_H2", "sat_H2", "targ_gas", "sat_gas",
             "halo_gas", "halo_cold", "halo_warmhot"]

def sath_path(A):
    tag = A["tag"] if A.get("tag") else "z0"
    return os.path.join(TABLEDIR, f"sat_halo_{SAT_STAGE}_{tag}.hdf5")

_H5HALO = dict(gH2="galaxy_data/dicts/masses.H2", ggas="galaxy_data/dicts/masses.gas",
               gstar="galaxy_data/dicts/masses.stellar",
               hgas="halo_data/dicts/masses.gas", hHI="halo_data/dicts/masses.HI",
               hH2="halo_data/dicts/masses.H2")

def _read_halo_cat(snap):
    """Catalog arrays for the halo-membership census (galaxy + halo masses). Missing keys -> NaN."""
    with h5py.File(sim.get_caesar_file(snap), "r") as f:
        parent = np.asarray(f["galaxy_data/parent_halo_index"][:], np.int64)
        n = len(parent)
        nh = int(parent.max()) + 1 if (parent >= 0).any() else 0
        def gg(key, m):
            d = f.get(_H5HALO[key]); return np.asarray(d[:], float) if d is not None else np.full(m, np.nan)
        return dict(parent=parent, gH2=gg("gH2", n), ggas=gg("ggas", n), gstar=gg("gstar", n),
                    hgas=gg("hgas", nh), hHI=gg("hHI", nh), hH2=gg("hH2", nh))

def _halo_one(cat, gx):
    parent = cat["parent"]; n = len(parent)
    if gx < 0 or gx >= n:
        return None
    h = int(parent[gx])
    tH2 = float(cat["gH2"][gx]); tgas = float(cat["ggas"][gx])
    if h < 0:                                            # ungrouped galaxy -> its own "halo"
        return dict(n_sat=0, is_central=1.0, targ_H2=tH2, sat_H2=0.0, targ_gas=tgas, sat_gas=0.0,
                    halo_gas=np.nan, halo_cold=np.nan, halo_warmhot=np.nan)
    memb = np.where(parent == h)[0]
    cen = (memb[np.nanargmax(cat["gstar"][memb])]
           if np.isfinite(cat["gstar"][memb]).any() else gx)          # central = most massive stellar
    with np.errstate(all="ignore"):
        sumH2 = float(np.nansum(cat["gH2"][memb])); sumgas = float(np.nansum(cat["ggas"][memb]))
        hg = float(cat["hgas"][h]) if h < len(cat["hgas"]) else np.nan
        hc = (float(cat["hHI"][h]) + float(cat["hH2"][h])) if h < len(cat["hHI"]) else np.nan
    return dict(n_sat=int(len(memb) - 1), is_central=float(gx == cen),
                targ_H2=tH2, sat_H2=max(sumH2 - tH2, 0.0),
                targ_gas=tgas, sat_gas=max(sumgas - tgas, 0.0),
                halo_gas=hg, halo_cold=hc,
                halo_warmhot=(max(hg - hc, 0.0) if (np.isfinite(hg) and np.isfinite(hc)) else np.nan))

if BUILD_SAT_CENSUS:
    rowkey = f"row_{SAT_STAGE}"
    for z in RESULTS:
        B = RESULTS[z]; A = B["A"]; recs = B["records"]; snaps = B["snaps_arr"]; PIDX = _pidx(z); n = len(recs)
        OUT_ = {k: np.full(n, np.nan) for k in SATH_KEYS}
        bysnap = {}
        for i, r in enumerate(recs):
            row = r.get(rowkey, -1)
            if row is None or row < 0:
                continue
            g = PIDX[row, r["col"]]
            if not np.isfinite(g):
                continue
            bysnap.setdefault(int(snaps[row]), []).append((i, int(g)))
        for snap, items in sorted(bysnap.items()):
            try:
                cat = _read_halo_cat(snap)
            except (OSError, KeyError) as e:
                print(f"  [{A['tag']}] snap {snap}: catalog read failed ({e}); skip {len(items)}"); continue
            for i, gx in items:
                res = _halo_one(cat, gx)
                if res:
                    for k in SATH_KEYS:
                        OUT_[k][i] = res[k]
            del cat; gc.collect()
        with h5py.File(sath_path(A), "w") as o:
            o.attrs.update(dict(z=float(z), sim_name=SIM_NAME, stage=SAT_STAGE))
            o.create_dataset("gid", data=np.array([r["gid"] for r in recs], np.int64))
            for k in SATH_KEYS:
                o.create_dataset(k, data=OUT_[k].astype("f8"))
        ns = OUT_["n_sat"]; hc_ok = int(np.isfinite(OUT_["halo_cold"]).sum())
        print(f"[{A['tag']}] halo sat -> {os.path.basename(sath_path(A))}: {int(np.isfinite(ns).sum())} gals, "
              f"median N_sat={np.nanmedian(ns):.0f}, halo-gas present for {hc_ok}"
              + ("" if hc_ok else "  [WARN: no halo_data/masses.gas -> cold/warm-hot NaN]"))
else:
    _missing = [os.path.basename(sath_path(RESULTS[z]["A"])) for z in RESULTS
                if not os.path.exists(sath_path(RESULTS[z]["A"]))]
    print("BUILD_SAT_CENSUS=False (halo).",
          ("all halo caches present." if not _missing else f"missing: {_missing} (set flag, run on cluster)."))

In [ ]:
# ── 6e·sat·halo·b — N_sat (y, common) vs environment/gas properties (x), by halo-mass bin ──
def load_sath(B):
    p = sath_path(B["A"]); recs = B["records"]; n = len(recs)
    if not os.path.exists(p):
        return {k: np.full(n, np.nan) for k in SATH_KEYS}
    with h5py.File(p, "r") as f:
        arrs = {k: f[k][:] for k in SATH_KEYS}; cg = f["gid"][:]
    arrs, _ = _align_records_by_gid(arrs, cg, np.array([r["gid"] for r in recs]), name="halo sat")
    return arrs

def _pool_wb(fn):
    return (np.concatenate([np.asarray(fn(RESULTS[z]), float)[RESULTS[z]["well_behaved"]] for z in RESULTS])
            if RESULTS else np.array([]))

SATH = {k: [] for k in SATH_KEYS}
_have_sath = all(os.path.exists(sath_path(RESULTS[z]["A"])) for z in RESULTS)
for z in RESULTS:
    B = RESULTS[z]; wb = B["well_behaved"]; s = load_sath(B)
    for k in SATH_KEYS:
        SATH[k].append(np.asarray(s[k])[wb])
SATH = {k: (np.concatenate(v) if v else np.array([])) for k, v in SATH.items()}
HMBIN = _pool_wb(lambda B: B["hmbin"])
MSTAR = _pool_wb(lambda B: _diag_at(B, "Mstar", SAT_STAGE))

if not _have_sath or not np.isfinite(SATH["n_sat"]).any():
    print("[6e·sat·halo] no halo cache yet — set BUILD_SAT_CENSUS=True (Part 0) and run on the cluster.")
else:
    N = len(SATH["n_sat"])
    _dt = CUM_DT_GYR if "CUM_DT_GYR" in globals() else 1.0
    cssfr = CSSFR if ("CSSFR" in globals() and len(CSSFR) == N) else np.full(N, np.nan)
    dusty = POOL["dusty"].astype(bool) if len(POOL.get("dusty", [])) == N else np.zeros(N, bool)
    with np.errstate(all="ignore"):
        h2frac = SATH["targ_H2"] / np.where(MSTAR > 0, MSTAR, np.nan)
    nsat = SATH["n_sat"].astype(float)
    props = [("cssfr", cssfr, fr"$\int$sSFR$\,$dt ({_dt:g} Gyr)", False),
             ("H2frac", h2frac, r"target H$_2$ fraction $M_{\rm H_2}/M_\star$", True),
             ("sat_H2", SATH["sat_H2"], r"satellites $M_{\rm H_2}$", True),
             ("halo_cold", SATH["halo_cold"], r"halo cold gas (HI+H2)", True),
             ("halo_warmhot", SATH["halo_warmhot"], r"halo warm/hot gas", True)]
    _bins = [(None, "all halo masses")] + [(b, HMASS_BIN_LABELS[b]) for b in range(len(HMASS_BIN_LABELS))]
    for b, blab in _bins:
        binm = np.ones(N, bool) if b is None else (HMBIN == b)
        if int((binm & np.isfinite(nsat)).sum()) < 5:
            print(f"[6e·sat·halo] halo mass {blab}: too few galaxies; skipped."); continue
        fig, axs = plt.subplots(1, len(props), figsize=(4.0 * len(props), 4.2), sharey=True)
        for ax, (k, xv, lab, logx) in zip(axs, props):
            x = np.log10(np.where(np.asarray(xv, float) > 0, xv, np.nan)) if logx else np.asarray(xv, float)
            m = binm & np.isfinite(x) & np.isfinite(nsat)
            ax.scatter(x[m & ~dusty], nsat[m & ~dusty], s=9, c="0.6", alpha=0.5, label="non-dusty")
            ax.scatter(x[m & dusty],  nsat[m & dusty],  s=11, c="C3", alpha=0.7, label="dusty")
            if m.sum() >= 8:                                   # binned-median N_sat vs x
                edges = np.unique(np.percentile(x[m], np.linspace(0, 100, 6)))
                if len(edges) >= 3:
                    cen = 0.5 * (edges[:-1] + edges[1:]); med = []
                    for lo, hi in zip(edges[:-1], edges[1:]):
                        sel = m & (x >= lo) & (x <= hi)
                        med.append(np.median(nsat[sel]) if sel.sum() >= 3 else np.nan)
                    ax.plot(cen, med, "-o", color="C0", lw=2, zorder=3)
            r, p, nn = _spr(x[binm], nsat[binm])
            ax.set_xlabel(("log10 " if logx else "") + lab)
            ax.set_title(f"ρ={r:+.2f} (p={p:.1g}, n={nn})", fontsize=9)
        axs[0].set_ylabel(r"$N_{\rm sat}$ (same halo)"); axs[0].legend(fontsize=7)
        fig.suptitle(fr"6e·sat·halo — $N_{{\rm sat}}$ vs properties @ {SAT_STAGE};  halo mass: {blab}  (pooled)", y=1.02)
        plt.tight_layout()
        _bt = "all" if b is None else f"hm{b}"
        plt.savefig(figpath(f"sigma_satcensus_halo_Nsat_{_bt}.png"), dpi=130, bbox_inches="tight"); plt.show()

### 6i — Does strong coupling switch on at different times for different masses?

`x_coup` (jet strength gated by gas-poorness) stacked on each critical-point clock, now **split by
stellar-mass bin** within the **strong-coupled** tercile. If massive galaxies couple *earlier*
(relative to AGN ignition / SFT / QT) than low-mass ones, the activation clock is mass-dependent.
The companion scalar is the per-galaxy coupling-activation lag `t_coup − t_QT` — the first rise of
`x_coup` to half its post-ignition peak — tabulated and plotted by mass bin for **strong vs weak**.


In [ ]:
# ── 6i·a — coupling-activation clock, resolved by stellar-mass bin (strong tercile) ──
MASS_COLORS = ["#2c7bb6", "#fdae61", "#d7191c"]   # one per stellar-mass bin (NBINS)

def mass_split(base=None):
    '''[(mask,color,label),...] splitting a base population into stellar-mass bins (mbin globals).'''
    base = np.ones(len(records), bool) if base is None else np.asarray(base, bool)
    return [(base & (mbin == b), MASS_COLORS[b % len(MASS_COLORS)], f"logM* {MASS_BIN_LABELS[b]}")
            for b in range(NBINS)]

def coupling_activation_time(B, frac=0.5, floor=0.05):
    '''First time x_coup rises to frac*(post-ignition peak), at/after t_AGN (per record, yr).
    Mirrors agn_activation_time but tracks the *coupling* (x_coup), which can lag BHAR ignition
    because coupling additionally needs the disc to become gas-poor.'''
    CO = B["CO"]; recs = B["records"]; t_agn = np.asarray(B["t_agn"], float)
    xc = CO.get("xcoup_hist"); t_coup = np.full(len(recs), np.nan)
    if xc is None:
        return t_coup
    _ord = B["REG"]["_ord"]; t_inc = B["REG"]["t_inc"]
    for i, r in enumerate(recs):
        if not np.isfinite(t_agn[i]):
            continue
        cs = xc[_ord, r["col"]].astype(float); post = t_inc >= t_agn[i]
        m = post & np.isfinite(cs) & (cs > 0)
        if m.sum() < 1:
            continue
        thr = max(frac * np.nanmax(cs[m]), floor)
        cr = np.where(post & np.isfinite(cs) & (cs >= thr))[0]
        if len(cr):
            t_coup[i] = t_inc[cr[0]]
    return t_coup

_COUP_CLASSES = [("strong", "strong"), ("weak", "weak"), ("intermediate", "inter")]
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling unavailable (set BUILD_BH=True)."); continue
    for _clab, _ckey in _COUP_CLASSES:   # no-AGN omitted: it has no coupling history to clock
        print(f"\n=== z={z} : x_coup activation by stellar-mass bin ({_clab}-coupled tercile) ===")
        event_stack([p for p in ["xcoup"] + SIGMA_PROPS if p in PROP_REGISTRY],
                    triggers=COUP_TRIG, split=_wb_split(mass_split(B["CO"][_ckey])),
                    xwin=2.0, nb=10, fname=f"coupling_bymass_{_clab}_z{_ztag(z)}.png")

# ── 6i·b — pooled coupling-activation lag (t_coup − t_QT) by mass bin, strong vs weak ──
lag_coup = []
for z in RESULTS:
    B = RESULTS[z]; wb = B["well_behaved"]
    tc = coupling_activation_time(B)
    tq = np.array([r["t_qt"] for r in B["records"]], float)
    lag_coup.append(((tc - tq) / GYR)[wb])
lag_coup = np.concatenate(lag_coup) if lag_coup else np.array([])   # aligned with POOL ordering

_LAGGRP = [("strong", POOL["strong"], "C3", -0.12), ("intermediate", POOL["inter"], "C2", 0.0),
           ("weak", POOL["weak"], "C0", 0.12)]   # no-AGN omitted: no coupling-activation time
_hdr = "  ".join(f"{lab + ' med':>14s} {'n':>4s}" for lab, _, _, _ in _LAGGRP)
print(f"\n{'mass bin':>12s}  {_hdr}   [t_coup-t_QT, Gyr]")
for b in range(NBINS):
    line = f"{MASS_BIN_LABELS[b]:>12s}  "
    for lab, grp, c, dx in _LAGGRP:
        v = lag_coup[grp.astype(bool) & (POOL["mbin"] == b)]; v = v[np.isfinite(v)]
        line += f"{(f'{np.median(v):.2f}' if len(v) else '--'):>14s} {len(v):>4d}  "
    print(line)

fig, ax = plt.subplots(figsize=(7, 4.5))
for lab, grp, c, dx in _LAGGRP:
    xs, meds, los, his = [], [], [], []
    for b in range(NBINS):
        v = lag_coup[grp.astype(bool) & (POOL["mbin"] == b)]; v = v[np.isfinite(v)]
        if len(v) < 3:
            continue
        xs.append(b + dx); meds.append(np.median(v))
        los.append(np.percentile(v, 25)); his.append(np.percentile(v, 75))
    if xs:
        meds = np.array(meds); los = np.array(los); his = np.array(his)
        ax.errorbar(xs, meds, yerr=[meds - los, his - meds], fmt="o-", color=c, capsize=4, label=lab)
ax.axhline(0, ls=":", c="k"); ax.set_xticks(range(NBINS)); ax.set_xticklabels(MASS_BIN_LABELS)
ax.set_xlabel(r"stellar-mass bin  $\log M_\star$")
ax.set_ylabel(r"$t_{\rm coup}-t_{\rm QT}$ [Gyr]  (median $\pm$ IQR)")
ax.set_title("6i - coupling-activation lag vs mass (pooled across anchors)"); ax.legend()
plt.tight_layout(); plt.savefig(figpath("coupling_lag_bymass.png"), dpi=130, bbox_inches="tight"); plt.show()


### 6j — What resolves the weak-coupling post-quench dust scatter?

The weak-coupling tercile shows the widest spread in residual dust after quenching. Is that scatter
random, or set by a second parameter? Pooled across anchors, the response is `log10(M_dust/M*)` at
`post_quench` (**weak sample only**); the candidate predictors, measured at the same stage, are the
**quenching timescale** `τ_q/t_H`, **stellar mass** `logM*`, **gas fraction** `M_gas/M*`, and
**gas surface density** `Σ_gas`. The strong tercile is shown as a control. Spearman |r| ranks the
single best predictor; a standardized multiple fit (lstsq, statsmodels-free) reports the combined
explained variance in the weak sample.


In [ ]:
# ── pooled predictors for the weak-coupling residual-dust scatter (well-behaved, POOL-aligned) ──
def _pool(fn):
    return (np.concatenate([np.asarray(fn(B), float)[B["well_behaved"]] for B in RESULTS.values()])
            if RESULTS else np.array([]))

mg   = _pool(lambda B: _diag_at(B, "Mgas",  "post_quench"))
ms   = _pool(lambda B: _diag_at(B, "Mstar", "post_quench"))
sgas = _pool(lambda B: _sig_at(B, "Sigma_gas_eff", "post_quench"))
with np.errstate(all="ignore"):
    y_dust = np.log10(np.where(POOL["dust_postq"] > 0, POOL["dust_postq"], np.nan))   # log Md/M* @ post_quench
    fgas   = np.where(ms > 0, mg / ms, np.nan)
    X = {
        r"$\log\,\tau_q/t_H$":          np.log10(np.where(POOL["tau_q_over_tH"] > 0, POOL["tau_q_over_tH"], np.nan)),
        r"$\log M_\star$":              _pool(lambda B: B["logm_rec"]),
        r"$\log\,M_{\rm gas}/M_\star$": np.log10(np.where(fgas > 0, fgas, np.nan)),
        r"$\log\,\Sigma_{\rm gas}$":    np.log10(np.where(sgas > 0, sgas, np.nan)),
    }

weak = POOL["weak"].astype(bool); strong = POOL["strong"].astype(bool)
inter = POOL["inter"].astype(bool); no_fb = POOL["no_fb"].astype(bool)
print(f"weak-sample residual-dust scatter: n={int(np.isfinite(y_dust[weak]).sum())}, "
      f"std(log Md/M*)={np.nanstd(y_dust[weak]):.2f} dex  "
      f"(strong: {np.nanstd(y_dust[strong]):.2f}, inter: {np.nanstd(y_dust[inter]):.2f}, "
      f"no-AGN: {np.nanstd(y_dust[no_fb]):.2f} dex)\n")
print(f"{'predictor':>26s}  {'weak |r|':>9s} {'p':>8s} {'n':>4s}   {'strong':>8s} {'inter':>8s} {'noAGN':>8s}")
fig, axs = plt.subplots(2, 2, figsize=(11, 9)); axs = axs.ravel()
_GRPS = [(strong, "0.7", "strong (control)", 0.30), (inter, "C2", "intermediate", 0.30),
         (no_fb, "0.5", "no-AGN", 0.30), (weak, "C0", "weak", 0.65)]
for ax, (name, xv) in zip(axs, X.items()):
    for msk, c, lab, al in _GRPS:
        m = msk & np.isfinite(xv) & np.isfinite(y_dust)
        ax.scatter(xv[m], y_dust[m], s=18, c=c, alpha=al, edgecolor="none", label=f"{lab} (n={int(m.sum())})")
    rw, pw, nw = _spearman(xv[weak], y_dust[weak]); rs, ps, ns = _spearman(xv[strong], y_dust[strong])
    ri_, pi_, ni_ = _spearman(xv[inter], y_dust[inter]); rn_, pn_, nn_ = _spearman(xv[no_fb], y_dust[no_fb])
    print(f"{name:>26s}  {abs(rw):9.3f} {pw:8.1g} {nw:4d}   {abs(rs):8.3f} {abs(ri_):8.3f} {abs(rn_):8.3f}")
    ax.set_xlabel(name); ax.set_ylabel(r"$\log_{10}(M_{\rm dust}/M_\star)$ @ post-quench")
    ax.set_title(f"weak |r|={abs(rw):.2f} (p={pw:.1g})"); ax.legend(fontsize=7)
fig.suptitle("6j - resolving the weak-coupling residual-dust scatter (pooled anchors)", y=1.0)
plt.tight_layout(); plt.savefig(figpath("weakdust_predictors.png"), dpi=130, bbox_inches="tight"); plt.show()

# ── standardized multiple fit (lstsq): combined explained variance in the weak sample ──
def _zsc(a):
    return (a - np.nanmean(a)) / (np.nanstd(a) + 1e-12)
keys = list(X)
Xmat = np.column_stack([_zsc(X[k]) for k in keys]); yv = _zsc(y_dust)
m = weak & np.isfinite(yv) & np.all(np.isfinite(Xmat), axis=1)
if m.sum() > len(keys) + 2:
    A = np.column_stack([Xmat[m], np.ones(int(m.sum()))])
    beta, *_ = np.linalg.lstsq(A, yv[m], rcond=None)
    pred = A @ beta
    r2 = 1.0 - np.nansum((yv[m] - pred) ** 2) / np.nansum((yv[m] - np.mean(yv[m])) ** 2)
    print(f"\nstandardized multiple fit (weak, complete rows n={int(m.sum())}): R^2={r2:.2f}")
    for k, bcoef in zip(keys, beta[:-1]):
        print(f"   beta[{k}] = {bcoef:+.2f}")
    print("(|beta| ranks each predictor's unique contribution; R^2 is the variance jointly explained.)")
else:
    print(f"\n[multiple fit] too few complete weak-sample rows (n={int(m.sum())}).")


### 6k — Σ distributions vs stellar age & stellar mass (contours; with missing-Σ fraction)

Pooled across anchors, the four ISM surface densities (`Σ_H2`, `Σ_HI`, `Σ_gas`, `Σ_dust`) at
`post_quench` as **2-D KDE contours** vs **mass-weighted stellar age** and **stellar mass**, split by
quench *time* (fast/slow) and by AGN–ISM *coupling* (strong/weak). Many quenched galaxies have **zero
gas / no profile**, which the log-contour silently drops — so each panel legend reports the **missing
fraction** (Σ ≤ 0 or NaN) per population. A contour built on few survivors with a large missing
fraction is not representative; read the two together.


In [ ]:
# ── pooled ISM surface densities vs stellar age & mass (catalog Σ; all anchors, well-behaved) ──
# SIGMA_INFO now lives in the Part 2 reduced-particle machinery cell.

def _pool(fn):
    return (np.concatenate([np.asarray(fn(B), float)[B["well_behaved"]] for B in RESULTS.values()])
            if RESULTS else np.array([]))

def _sig_safe(B, key, stage):
    '''_sig_at, but NaN if the Σ key is absent.'''
    RP = B["RP"]; DPS = RP["STAGES"]
    if not RP["OK"] or key not in RP or stage not in DPS:
        return np.full(len(B["records"]), np.nan)
    return RP[key][:, DPS.index(stage)]

def sigma_table_catalog(stage="post_quench"):
    '''Pooled per-record table: each Σ (raw, may be 0/NaN), age, logM*, and group masks.'''
    T = {key: _pool(lambda B, k=key: _sig_safe(B, k, stage)) for key, _ in SIGMA_INFO}
    with np.errstate(all="ignore"):
        T["age"]  = _pool(lambda B: _diag_at(B, "age", stage))
        msv       = _pool(lambda B: _diag_at(B, "Mstar", stage))
        T["logM"] = np.log10(np.where(msv > 0, msv, np.nan))
    T["is_fast"] = _pool(lambda B: B["is_fast"]); T["is_slow"] = _pool(lambda B: B["is_slow"])
    T["strong"]  = _pool(lambda B: B["CO"]["strong"]); T["weak"] = _pool(lambda B: B["CO"]["weak"])
    T["inter"]   = _pool(lambda B: B["CO"]["inter"]);  T["no_fb"] = _pool(lambda B: B["CO"]["no_fb"])
    return T

def _kde_or_scatter(ax, x, y, color, label):
    '''2-D KDE contour of (x,y) in `color`; fall back to scatter when too few finite points.'''
    m = np.isfinite(x) & np.isfinite(y)
    if int(m.sum()) >= 12:
        try:
            k = gaussian_kde(np.vstack([x[m], y[m]]))
            xi = np.linspace(np.nanpercentile(x[m], 1), np.nanpercentile(x[m], 99), 60)
            yi = np.linspace(np.nanpercentile(y[m], 1), np.nanpercentile(y[m], 99), 60)
            XX, YY = np.meshgrid(xi, yi)
            ZZ = k(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            ax.contour(XX, YY, ZZ, levels=4, colors=color, linewidths=1.4, alpha=0.9)
            ax.plot([], [], color=color, lw=1.4, label=label)        # legend proxy
            return
        except Exception:
            pass
    ax.scatter(x[m], y[m], s=12, c=color, alpha=0.5, edgecolor="none", label=label)

def sigma_distributions(T, split="fastslow", stage="post_quench", fname=None):
    '''Contour grid of each Σ vs age and logM*, per population, annotating the missing (0/NaN) fraction.'''
    if split == "fastslow":
        groups = [(T["is_fast"].astype(bool), "C3", "fast"), (T["is_slow"].astype(bool), "C0", "slow")]
    else:
        groups = [(T["strong"].astype(bool), "C3", "strong"), (T["weak"].astype(bool), "C0", "weak"),
                  (T["inter"].astype(bool), "C2", "intermediate"), (T["no_fb"].astype(bool), "0.5", "no AGN")]
    xspecs = [(np.asarray(T["age"], float), "mass-weighted stellar age [Gyr]"),
              (np.asarray(T["logM"], float), r"$\log_{10} M_\star$")]
    fig, axs = plt.subplots(len(SIGMA_INFO), 2, figsize=(11, 3.2 * len(SIGMA_INFO)), sharey="row")
    for ri, (key, lab) in enumerate(SIGMA_INFO):
        with np.errstate(all="ignore"):
            y = np.log10(np.where(np.asarray(T[key], float) > 0, T[key], np.nan))
        for ci, (xv, xlab) in enumerate(xspecs):
            ax = axs[ri][ci]
            for msk, c, glab in groups:
                present = msk & np.isfinite(xv)
                np_ok = int((present & np.isfinite(y)).sum()); npr = int(present.sum())
                miss = (1.0 - np_ok / npr) if npr else np.nan
                _kde_or_scatter(ax, xv[msk], y[msk], c, f"{glab} (n={np_ok}, miss {miss:.0%})")
            if ri == len(SIGMA_INFO) - 1:
                ax.set_xlabel(xlab)
            if ci == 0:
                ax.set_ylabel(f"log {lab}")
            ax.legend(fontsize=8)
    fig.suptitle(f"Sigma @ {stage} vs age & mass — contours, {split} "
                 f"(pooled; 'miss' = zero/NaN Σ)", y=1.0)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

_T_cat = sigma_table_catalog("post_quench")
for _split in ("fastslow", "weakstrong"):
    sigma_distributions(_T_cat, split=_split, stage="post_quench", fname=f"sigma_dist_{_split}_postq.png")


### 6l — Reduced-aperture Σ (flexible binning, ISM+CGM, 100 kpc)

The catalog Σ above are fixed-binning, galaxy-member-list (`gal.glist`, ISM-only) products. This
section instead reads the **lean reduced particle files** written by `build_reduced_particles_job.py`
(run `submit_reduced_particles.sh` per anchor) — all gas+star particles within a **100 kpc physical
aperture** (ISM *and* CGM, parent-halo candidate set, not just the listed galaxy particles), storing
only `pos / mass / m_dust / m_HI / m_H2 / sfr`. From these we can recompute Σ at **any aperture,
binning, geometry (face-on cylindrical or 3-D) and region (all / ISM `sfr>0` / CGM `sfr==0`)** without
re-touching the snapshots, and feed the same contour view as 6k. The cell is gated on the files
existing; until the cluster job has run it just reports how many are present.


In [ ]:
# ── 6l render — recompute Σ from the reduced 100 kpc particle files (loaders are in Part 2) ──
# render only if the reduced files exist; otherwise point at the build step
if os.path.isdir(REDUCED_DIR) and any(f.endswith('.h5') for _d, _s, _fs in os.walk(REDUCED_DIR) for f in _fs):
    APER_KPC, GEOM, REGION = 10.0, "cyl", "all"        # tune the aperture/geometry/region freely
    _T_red, _n = sigma_table_reduced("post_quench", r_aper_kpc=APER_KPC, geom=GEOM, region=REGION)
    if _n:
        for _split in ("fastslow", "weakstrong"):
            sigma_distributions(_T_red, split=_split, stage=f"post_quench (reduced {APER_KPC:g}kpc {REGION})",
                                fname=f"sigma_dist_reduced_{_split}_postq.png")
else:
    print(f"[reduced Σ] no files under {REDUCED_DIR}.\n"
          f"  Build them on the cluster (per anchor): \n"
          f"    DUST_PLAN=<anchor plan> sbatch --array=0-15 submit_reduced_particles.sh\n"
          f"  (reuses each anchor's dust_profile_plan_<tag>.hdf5; writes reduced_particles/snap_NNN/*.h5)")

### 6m — Where do the missing Σ points come from? (data vs physics)

The 6l contours show a large "miss" fraction, worst for weak coupling at `post_quench` (~90%).
`sigma_distributions` lumps two very different causes into one number:

- **NaN** — no reduced file extracted, *or* the galaxy has no such critical point (`row<0`);
- **Σ = 0** — the aperture is genuinely **gas-empty**, i.e. H₂ has been depleted.

This cell separates them per critical point and per class. If the weak-coupling loss at
`post_quench` is mostly **H2-empty(=0)**, it is the physical signature of slow/maintenance-mode
quenching finishing the job — not a data gap — which is exactly why the feedback is better read at
the *earlier* stages (`agn`, `sft`, `qt`), while gas is still present (6n).

In [ ]:
# ── 6m·miss — separate NO-FILE (data) from H2-DEPLETED (physics) per critical point and class ──
STATUS = {0: "usable(>0)", 1: "H2-empty(=0)", 2: "no-file", 3: "no-point", 4: "no-H2-data"}

def reduced_status(stage, comp="H2", r_aper_kpc=10.0, geom="cyl", region="all"):
    """Pooled per-record status code for `comp` Σ at `stage` (aligned 1:1 with POOL)."""
    out = []
    for z in RESULTS:
        B = RESULTS[z]; wb = B["well_behaved"]; recs = B["records"]; snaps = B["snaps_arr"]; PIDX = _pidx(z)
        st = np.full(len(recs), 3, np.int8)            # default: galaxy has no such critical point
        for i, r in enumerate(recs):
            row = r.get(f"row_{stage}", -1); col = r["col"]
            if row < 0:
                continue
            gx = PIDX[row, col]
            if not np.isfinite(gx):
                continue
            red = load_reduced(int(snaps[row]), int(gx))
            if red is None:
                st[i] = 2; continue                    # reduced file not extracted
            R, m, reg = _reduced_sel(red, comp, geom, region)
            if R is None:
                st[i] = 4; continue                    # file present but component dataset absent
            sel = reg & (R <= r_aper_kpc)
            if np.nansum(m[sel]) > 0:
                st[i] = 0                              # usable
            elif np.any(reg):
                st[i] = 1                              # aperture genuinely gas-empty -> depleted
            else:
                st[i] = 4
        out.append(st[wb])
    return np.concatenate(out) if out else np.array([], np.int8)

STAGE_SEQ = ["agn", "sft", "qt", "post_quench"]        # time order after AGN activation
GROUPS = [("strong", POOL["strong"].astype(bool)), ("weak", POOL["weak"].astype(bool)),
          ("fast", POOL["is_fast"].astype(bool)),     ("slow", POOL["is_slow"].astype(bool))]
print("reduced Sigma_H2 status (aperture 10 kpc, all) — % of each class per critical point")
print(f"{'stage':>12s} {'class':>7s} {'N':>5s} " + " ".join(f"{STATUS[c]:>13s}" for c in range(5)))
for stg in STAGE_SEQ:
    code = reduced_status(stg)
    if code.size == 0:
        print(f"{stg:>12s}  (no records)"); continue
    for lab, msk in GROUPS:
        n = int(msk.sum())
        if n == 0:
            continue
        fr = [100.0 * np.mean(code[msk] == c) for c in range(5)]
        print(f"{stg:>12s} {lab:>7s} {n:>5d} " + " ".join(f"{x:12.0f}%" for x in fr))
    print()

### 6n — The feedback effect over time: Σ contours along agn → sft → qt → post_quench

The same contour view as 6l, but evaluated at each critical point *after* AGN activation, so the
surface densities can be read as the galaxy moves through the feedback episode — before
`post_quench` depletion empties the aperture. Strong vs weak coupling is the organizing split;
the fast/slow version is rendered alongside for comparison. Σ is the corrected 100 kpc
reduced-particle aperture (tune `APER_KPC` / `REGION`).

The aperture is now **size-relative** (per-galaxy R50★ from the reduced star particles), so the comparison isn't biased by galaxies having different physical sizes.

In [ ]:
# ── 6n·feedback — reduced-Σ contours at each post-activation critical point (agn, sft, qt, postq) ──
if os.path.isdir(REDUCED_DIR) and any(f.endswith('.h5') for _d, _s, _fs in os.walk(REDUCED_DIR) for f in _fs):
    GEOM, REGION, APER_FACTOR = "cyl", "all", 1.0      # size-relative aperture = APER_FACTOR x R50*
    for stg in ["agn", "sft", "qt", "post_quench"]:
        T_s, n_s = sigma_table_reduced(stg, aper="R50star", aper_factor=APER_FACTOR, geom=GEOM, region=REGION)
        if not n_s:
            print(f"  [{stg}] no reduced files loaded -> skipped"); continue
        for split in ("weakstrong", "fastslow"):
            sigma_distributions(T_s, split=split,
                                stage=f"{stg} (reduced, within {APER_FACTOR:g}xR50* {REGION})",
                                fname=f"sigma_dist_reduced_{split}_{stg}.png")
else:
    print(f"[6n·feedback] no reduced files under {REDUCED_DIR}; build them first (see 6l).")

### 6o — Is the stacked gap real, or am I only seeing the un-censored galaxies?

The event-aligned log-medians (6b/6c/6d) drop zeros (they become NaN), so the track is the median
*given* value > 0. A "strong has more H₂ than weak at QT" gap can therefore be a selection effect:
strong may simply keep more galaxies above zero while most weak galaxies are depleted and never
enter the median. Passing `show_detect=True` to `event_stack` overlays, per bin and group, the
**detected fraction** (values > 0 / galaxies present) — marker size ∝ fraction, a dotted line on the
right 0–1 axis, and shading where the median rests on < `frac_floor` of the present galaxies.

Read the two together: a separation that holds where **both** detected fractions are high is a real
amplitude difference; one that lives in the shaded / low-fraction bins is survivorship.

In [ ]:
# ── 6o·detect — censoring-aware molecular-tracer stacks (strong vs weak) ──
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling split unavailable."); continue
    print(f"\n=== z = {z} : H2 tracers with detected-fraction overlay (strong vs weak) ===")
    event_stack([p for p in ["M_H2", "H2/M*", "Sigma_H2"] if p in PROP_REGISTRY],
                triggers=["t_AGN", "t_SFT", "t_QT", "t_PostQuench"], split=_wb_split(SPLIT_XS),
                xwin=2.0, nb=10, show_detect=True,
                fname=f"censoring_H2_xstrength_z{_ztag(z)}.png")

### 6q — Example galaxies from the reduced files (particles + bins + radial Σ)

A qualitative check, and a look at the gas *phases*. **Left** = face-on particles: stars (grey),
gas split into **SF/ISM** (sfr>0, cold, blue) vs **non-SF/diffuse** (sfr=0, warm-hot, orange), with
the radial bin edges overlaid. **Right** = radial surface density of total gas, HI, H2, and the
**ionized/hot residual** (Σ_gas − Σ_HI − Σ_H2; also carries He+metals). Bins are **log-spaced**
(dense ISM core, coarse CGM) on a log x-axis. No temperature in the reduced files, so "hot" is
inferred from non-neutral / non-SF gas, not a T cut.

The picked galaxies (class, record index, gid, snap, gx) are saved to `EXAMPLE_GALAXIES` and
`tables/example_galaxies.json`, so **6r** can replot the *same* galaxies at other critical points.
Knobs: `EX_STAGE`, `classes` (fast/slow), `r_view`, `nbins`, `r_min`, `bin_mode`.

Galaxies are now **mass-matched**: per class the picks are the ones nearest a common target log M* at the stage (set `mass_target`/`mass_tol`), so you compare similar galaxies rather than random ones.

In [ ]:
# ── 6q·examples — face-on particles (ISM/diffuse) + log bins + radial Σ; saves the picked galaxies ──
EXAMPLE_GALAXIES = {}                                   # {z: [{lab,rec,gid,stage,snap,gx}, ...]} — 6q's picks
EXAMPLE_FILE = os.path.join(TABLEDIR, "example_galaxies.json")
_LABCOL = {"strong": "C3", "weak": "C0", "intermediate": "C2", "no AGN": "0.5", "fast": "C3", "slow": "C0"}

def save_example_galaxies():
    with open(EXAMPLE_FILE, "w") as f:
        json.dump({str(z): v for z, v in EXAMPLE_GALAXIES.items()}, f, indent=1)
    print(f"  saved {sum(len(v) for v in EXAMPLE_GALAXIES.values())} example galaxies -> {EXAMPLE_FILE}")

def load_example_galaxies():
    return ({float(z): v for z, v in json.load(open(EXAMPLE_FILE)).items()}
            if os.path.exists(EXAMPLE_FILE) else {})

def plot_reduced_galaxies(z, stage="qt", n_ex=2, r_view=30.0, nbins=18, bin_mode="log", r_min=0.3,
                          classes=None, selection=None, save_selection=True,
                          mass_target=None, mass_tol=0.3, fname=None):
    """Face-on particles + radial Σ for example galaxies. selection=[(lab,rec),...] replots the SAME
    galaxies (by record index) at `stage`; otherwise picks n_ex per class and saves the selection."""
    from matplotlib.patches import Circle
    B = RESULTS.get(z)
    if B is None:
        print(f"[z={z}] not built"); return []
    recs = B["records"]; snaps = B["snaps_arr"]; wb = B["well_behaved"]; PIDX = _pidx(z); CO = B["CO"]
    with np.errstate(all="ignore"):                    # log10 M* at this stage, for mass-matched picks
        _ms = _diag_at(B, "Mstar", stage); lmass = np.log10(np.where(_ms > 0, _ms, np.nan))

    def _resolve(lab, i):                               # (lab, rec) -> (lab, rec, gid, snap, gx, red) at `stage`
        row = recs[i].get(f"row_{stage}", -1)
        if row is None or row < 0:
            return None
        gx = PIDX[row, recs[i]["col"]]
        if not np.isfinite(gx):
            return None
        red = load_reduced(int(snaps[row]), int(gx))
        if red is None or "gas" not in red or "pos" not in red.get("gas", {}):
            return None
        return (lab, int(i), int(recs[i]["gid"]), int(snaps[row]), int(gx), red)

    picks = []
    if selection is not None:                           # reuse a saved selection (record indices)
        for lab, i in selection:
            r = _resolve(lab, int(i))
            if r is None:
                print(f"  [{lab}] rec {i}: no reduced file at stage '{stage}'"); continue
            picks.append(r)
    else:
        if classes is None:
            classes = [("strong", np.asarray(CO["strong"], bool) & wb),
                       ("weak",   np.asarray(CO["weak"],   bool) & wb),
                       ("intermediate", np.asarray(CO["inter"], bool) & wb),
                       ("no AGN", np.asarray(CO["no_fb"], bool) & wb)]
        if mass_target is None:                         # common target M* so the classes are compared like-for-like
            _allm = np.concatenate([lmass[m] for _, m in classes]) if classes else np.array([])
            mass_target = float(np.nanmedian(_allm)) if np.isfinite(_allm).any() else np.nan
        print(f"  [mass-matched] target logM* = {mass_target:.2f} (tol {mass_tol}) at stage '{stage}'")
        for lab, mask in classes:
            cand = np.where(mask & np.isfinite(lmass))[0]
            if np.isfinite(mass_target) and len(cand):
                cand = cand[np.argsort(np.abs(lmass[cand] - mass_target))]   # nearest in stellar mass first
            got = 0
            for i in cand:
                r = _resolve(lab, int(i))
                if r is None:
                    continue
                if mass_tol is not None and np.isfinite(mass_target) and abs(lmass[i] - mass_target) > mass_tol:
                    print(f"    [{lab}] nearest with a file is logM*={lmass[i]:.2f} (>{mass_tol} from target)")
                picks.append(r); got += 1
                if got >= n_ex:
                    break
            if got == 0:
                print(f"  [{lab}] no reduced files at stage '{stage}' (z={z})")
    if not picks:
        print(f"  no example galaxies at z={z}, stage '{stage}'."); return []

    edges = (np.concatenate([[0.0], np.logspace(np.log10(r_min), np.log10(r_view), nbins)])
             if bin_mode == "log" else np.linspace(0.0, r_view, nbins + 1))
    rmid = 0.5 * (edges[:-1] + edges[1:])
    fig, axs = plt.subplots(len(picks), 2, figsize=(11, 4.2 * len(picks)), squeeze=False)
    for ri, (lab, i, gid, snp, gx, red) in enumerate(picks):
        col = _LABCOL.get(lab, "C2"); axS, axP = axs[ri]
        ev = np.asarray(red["evecs"], float)
        def _fo(p):
            pr = np.asarray(p, float) @ ev.T
            return pr[:, 0], pr[:, 1]
        if "star" in red and "pos" in red["star"]:
            xs, ys = _fo(red["star"]["pos"]); ms = (np.abs(xs) < r_view) & (np.abs(ys) < r_view)
            axS.scatter(xs[ms], ys[ms], s=2, c="0.7", alpha=0.3, edgecolor="none", rasterized=True,
                        label=f"stars ({int(ms.sum())})")
        g = red["gas"]; xg, yg = _fo(g["pos"]); inv = (np.abs(xg) < r_view) & (np.abs(yg) < r_view)
        sfr = np.asarray(g.get("sfr", np.zeros(len(xg))), float)
        cgm = inv & (sfr <= 0); ism = inv & (sfr > 0)
        axS.scatter(xg[cgm], yg[cgm], s=3, c="C1", alpha=0.35, edgecolor="none", rasterized=True,
                    label=f"gas non-SF / diffuse ({int(cgm.sum())})")
        axS.scatter(xg[ism], yg[ism], s=6, c="C0", alpha=0.75, edgecolor="none", rasterized=True,
                    label=f"gas SF / ISM ({int(ism.sum())})")
        for e in edges[1:]:
            axS.add_patch(Circle((0, 0), e, fill=False, ec="k", lw=0.3, alpha=0.3))
        axS.set_aspect("equal"); axS.set_xlim(-r_view, r_view); axS.set_ylim(-r_view, r_view)
        axS.set_xlabel("x' [kpc]"); axS.set_ylabel("y' [kpc]")
        axS.set_title(f"{lab}: z={z} gal {gx} (rec {i}, logM*={lmass[i]:.2f}, @{stage})")
        axS.legend(fontsize=7, loc="upper right")
        _, sg  = reduced_sigma_profile(red, "gas", edges=edges, geom="cyl", region="all")
        _, shi = reduced_sigma_profile(red, "HI",  edges=edges, geom="cyl", region="all")
        _, sh2 = reduced_sigma_profile(red, "H2",  edges=edges, geom="cyl", region="all")
        s_oth = np.clip(sg - shi - sh2, 0.0, None)
        for sig, cc, ls, labp in ((sg,   "0.2", "-",  r"$\Sigma_{\rm gas}$ (total)"),
                                  (shi,  "C0",  ":",  r"$\Sigma_{\rm HI}$"),
                                  (sh2,  "C3",  "--", r"$\Sigma_{\rm H_2}$"),
                                  (s_oth, "C1", "-.", r"$\Sigma_{\rm ion/hot}$ (gas$-$HI$-$H2)")):
            with np.errstate(divide="ignore", invalid="ignore"):
                axP.plot(rmid, np.log10(np.where(sig > 0, sig, np.nan)), ls, color=cc, label=labp)
        if bin_mode == "log":
            axP.set_xscale("log"); axP.set_xlim(r_min * 0.6, r_view)
        axP.set_xlabel("R [kpc]"); axP.set_ylabel(r"$\log_{10}\,\Sigma\ [M_\odot\,{\rm kpc}^{-2}]$")
        axP.set_title("radial surface density"); axP.legend(fontsize=7)
    fig.suptitle(f"reduced-file example galaxies (z={z}, stage {stage})", y=1.0)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

    meta = [{"lab": lab, "rec": i, "gid": gid, "stage": stage, "snap": snp, "gx": gx}
            for (lab, i, gid, snp, gx, red) in picks]
    if selection is None and save_selection:            # remember the picks for reuse at other stages (6r)
        EXAMPLE_GALAXIES[z] = meta
        save_example_galaxies()
    return meta

EX_STAGE = "qt"        # critical point to sample; e.g. "agn" / "sft" / "qt" / "post_quench"
for z in RESULTS:
    plot_reduced_galaxies(z, stage=EX_STAGE, n_ex=2, r_view=30.0, nbins=18, bin_mode="log", r_min=0.3,
                          fname=f"reduced_examples_z{_ztag(z)}_{EX_STAGE}.png")

### 6r — The same galaxies across critical points

Replots exactly the galaxies 6q picked (by record index, from `EXAMPLE_GALAXIES` / the saved JSON)
at each stage in `COMPARE_STAGES`, so you can follow one galaxy's ISM/CGM and Σ profile through
`agn → sft → qt → post_quench`. A galaxy with no reduced file at a given stage is skipped with a note.

In [ ]:
# ── 6r·same-galaxies — replot the saved 6q selection at other critical points ──
COMPARE_STAGES = ["agn", "sft", "qt", "post_quench"]
_saved = EXAMPLE_GALAXIES or load_example_galaxies()
if not _saved:
    print("no saved example selection — run 6q first.")
for z in RESULTS:
    sel_meta = _saved.get(z) or _saved.get(float(z))
    if not sel_meta:
        print(f"[z={z}] no saved selection — run 6q at this anchor first."); continue
    selection = [(d["lab"], d["rec"]) for d in sel_meta]
    print(f"\n=== z={z}: {len(selection)} saved galaxies across {COMPARE_STAGES} ===")
    for stage in COMPARE_STAGES:
        plot_reduced_galaxies(z, stage=stage, selection=selection, save_selection=False,
                              fname=f"reduced_samegals_z{_ztag(z)}_{stage}.png")

### 6s — (moved to Part 2)

The reduced-file Σ-profile cache (`PROFILE_CACHE`) and the `RP` scalar layer that supersedes the old
dust-profile products are now built in **Part 2**, right after the reduced-particle machinery, so the
driver and every downstream analysis can read them. Set `BUILD_PROFILE_CACHE=True` (first time) or
`REBUILD_PROFILE_CACHE=True` (after adding star/temp) there.

### 6t — Average Σ profile at the critical points (fast/slow, weak/strong)

For each critical point in `STAGE_SEQ` (`agn → sft → qt → post_quench`) and each class, take every
galaxy's cached profile at that critical point and plot the **median** Σ(R) with a shaded percentile
band. Two panels per figure (the split's two classes); the critical point is colour-coded; one
figure per component (gas/H2/HI/dust) and per split. This uses only the snapshots that already have
reduced files.

`avg_profiles_vs_tagn(...)` (sampling at fixed Δt after `t_AGN`) is kept for the time-resolved plot —
populate it by running the **6u** windowed extraction first, then re-running 6s.

**Zero gas is counted** (not dropped): empty annuli (Σ=0) are mapped to a display floor, so a central hole shows as the median dipping to the floor; a dotted twin-axis line gives the fraction of galaxies with Σ>0 at each R, revealing partial holes the median alone would miss.

In [ ]:
# ── 6t·avgprof — mean Σ(R) by critical point (default) or at fixed Δt after AGN (uses the 6s cache) ──
DT_MYR     = [-2000, -1000, -500, -200, 200, 500, 1000, 2000]  # rel. to t_AGN [Myr]; neg=before (needs 6u)
DT_TOL_MYR = 300                                  # accept a reduced snapshot within this of the target
STAGE_SEQ  = ["agn", "sft", "qt", "post_quench"]  # critical points to average at (data available today)
AVG_BAND   = (16, 84)                             # shaded percentile band
SHOW_FRAC  = True                                 # twin-axis dotted line: fraction with Sigma>0 (reveals holes)

def _split_groups(B, split):
    CO = B["CO"]; wb = B["well_behaved"]
    return ([("fast", CO["fast"] & wb), ("slow", CO["slow"] & wb)] if split == "fastslow"
            else [("strong", CO["strong"] & wb), ("weak", CO["weak"] & wb),
                  ("intermediate", CO["inter"] & wb), ("no AGN", CO["no_fb"] & wb)])

def _med_band_floor(A, floor):
    """Median + percentile band of log10 Σ INCLUDING empty annuli: Σ=0 -> `floor` (so a hole shows
    as the curve dipping to the floor, not as a dropped point). NaN = no data (galaxy absent at R).
    Also returns the detected fraction (Σ>0 among galaxies present at each R)."""
    out = np.full(A.shape, np.nan, float)
    out[A > 0] = np.log10(A[A > 0]); out[A == 0] = floor       # zeros counted at the floor
    with np.errstate(all="ignore"):
        med = np.nanmedian(out, axis=0)
        lo = np.nanpercentile(out, AVG_BAND[0], axis=0); hi = np.nanpercentile(out, AVG_BAND[1], axis=0)
    present = np.isfinite(A).sum(0); ndet = (A > 0).sum(0)
    frac = np.where(present > 0, ndet / np.maximum(present, 1), np.nan)
    return med, lo, hi, frac

def _stack2d(stack):
    """Stack same-length profiles into (n, nbins); drop any whose length != current bin count."""
    nb = len(PROF_RMID)
    rows = [np.asarray(p, float) for p in stack if np.ndim(p) == 1 and len(p) == nb]
    return np.vstack(rows) if rows else np.empty((0, nb))

def _draw(ax, series, comp):
    """series: list of (label, color, stack-of-profiles). Zeros are kept (floored); a dotted
    twin-axis line shows the fraction with Σ>0 so partial central holes are visible too."""
    series = [(lab, c, _stack2d(stack)) for lab, c, stack in series]
    arrs = [A for _, _, A in series if A.shape[0] >= 3]
    posv = np.concatenate([a[a > 0] for a in arrs]) if arrs else np.array([])
    floor = float(np.floor(np.log10(posv.min())) - 0.5) if posv.size else 0.0
    axd = ax.twinx() if SHOW_FRAC else None
    for lab, c, A in series:
        if A.shape[0] < 3:
            continue
        med, lo, hi, frac = _med_band_floor(A, floor)
        ax.plot(PROF_RMID, med, "-", color=c, label=f"{lab} (n={A.shape[0]})")
        ax.fill_between(PROF_RMID, lo, hi, color=c, alpha=0.12, lw=0)
        if axd is not None:
            axd.plot(PROF_RMID, frac, ":", color=c, alpha=0.55, lw=1.0)
    if posv.size:
        ax.axhline(floor, ls=":", c="0.6", lw=0.8)             # empty-annulus (Σ=0) floor
    if axd is not None:
        axd.set_ylim(0, 1.05); axd.set_ylabel("frac Σ>0", fontsize=7); axd.tick_params(labelsize=6)
    ax.set_xscale("log"); ax.set_xlim(PROF_RMIN * 0.6, PROF_RVIEW); ax.set_xlabel("R [kpc]")

def avg_profiles_by_stage(z, split="fastslow", comp="gas", stages=STAGE_SEQ, fname=None):
    B = RESULTS.get(z)
    if B is None or not PROFILE_CACHE:
        print("  need RESULTS + a built PROFILE_CACHE (run 6s)."); return
    PIDX = _pidx(z); recs = B["records"]; snaps = B["snaps_arr"]
    cmap = plt.cm.viridis(np.linspace(0.0, 0.9, len(stages)))
    groups = _split_groups(B, split)
    fig, axs = plt.subplots(1, len(groups), figsize=(6.0 * len(groups), 4.8), sharey=True, squeeze=False)
    for ax, (lab, mask) in zip(axs[0], groups):
        idx = np.where(mask)[0]; series = []
        for si, st in enumerate(stages):
            stack = []
            for i in idx:
                row = recs[i].get(f"row_{st}", -1)
                if row is None or row < 0:
                    continue
                g = PIDX[row, recs[i]["col"]]
                if not np.isfinite(g):
                    continue
                prof = PROFILE_CACHE.get((int(snaps[row]), int(g)), {}).get(comp)
                if prof is not None:
                    stack.append(prof)
            series.append((st, cmap[si], stack))
        _draw(ax, series, comp); ax.set_title(lab); ax.legend(fontsize=7, title="critical point")
    axs[0][0].set_ylabel(fr"$\log_{{10}}\,\Sigma_{{\rm {comp}}}\ [M_\odot\,{{\rm kpc}}^{{-2}}]$")
    fig.suptitle(f"z={z}: mean $\\Sigma_{{\\rm {comp}}}(R)$ by critical point — {split} "
                 f"(median, {AVG_BAND[0]}–{AVG_BAND[1]}% band)", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

def _galaxy_reduced_times(B, i, PIDX):
    """(times[yr], snaps, gxs) of this galaxy's cached reduced files (its critical-point rows)."""
    r = B["records"][i]; col = r["col"]; t_ = B["t_cosmic_yr"]; snaps = B["snaps_arr"]
    rows = sorted({int(v) for k, v in r.items()
                   if k.startswith("row_") and isinstance(v, (int, np.integer)) and v >= 0})
    ts, sn, gx = [], [], []
    for row in rows:
        g = PIDX[row, col]
        if not np.isfinite(g):
            continue
        key = (int(snaps[row]), int(g))
        if key in PROFILE_CACHE:
            ts.append(float(t_[row])); sn.append(key[0]); gx.append(key[1])
    return np.asarray(ts), np.asarray(sn), np.asarray(gx)

def avg_profiles_vs_tagn(z, split="fastslow", comp="gas", dt_myr=DT_MYR, tol_myr=DT_TOL_MYR, fname=None):
    """Time-resolved version: needs reduced files near t_AGN+Δt (run 6u extraction first)."""
    B = RESULTS.get(z)
    if B is None or not PROFILE_CACHE:
        print("  need RESULTS + a built PROFILE_CACHE (run 6s)."); return
    PIDX = _pidx(z); tagn = np.asarray(B["t_agn"], float)
    cmap = plt.cm.viridis(np.linspace(0.0, 0.9, len(dt_myr)))
    groups = _split_groups(B, split)
    fig, axs = plt.subplots(1, len(groups), figsize=(6.0 * len(groups), 4.8), sharey=True, squeeze=False)
    for ax, (lab, mask) in zip(axs[0], groups):
        idx = np.where(mask & np.isfinite(tagn))[0]
        gtimes = {i: _galaxy_reduced_times(B, i, PIDX) for i in idx}
        series = []
        for di, dt in enumerate(dt_myr):
            stack = []
            for i in idx:
                ts, sn, gx = gtimes[i]
                if len(ts) == 0:
                    continue
                j = int(np.argmin(np.abs(ts - (tagn[i] + dt * 1e6))))
                if abs(ts[j] - (tagn[i] + dt * 1e6)) > tol_myr * 1e6:
                    continue
                prof = PROFILE_CACHE[(int(sn[j]), int(gx[j]))].get(comp)
                if prof is not None:
                    stack.append(prof)
            series.append((f"+{dt} Myr", cmap[di], stack))
        _draw(ax, series, comp); ax.set_title(lab); ax.legend(fontsize=7, title="since $t_{\\rm AGN}$")
    axs[0][0].set_ylabel(fr"$\log_{{10}}\,\Sigma_{{\rm {comp}}}\ [M_\odot\,{{\rm kpc}}^{{-2}}]$")
    fig.suptitle(f"z={z}: mean $\\Sigma_{{\\rm {comp}}}(R)$ after AGN activation — {split} "
                 f"(median, {AVG_BAND[0]}–{AVG_BAND[1]}% band)", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

for z in RESULTS:
    if RESULTS[z]["BH"] is None:
        print(f"[z={z}] no BH history -> no t_AGN; skip."); continue
    for split in ("fastslow", "weakstrong"):
        for comp in PROF_COMPS:
            avg_profiles_by_stage(z, split=split, comp=comp,
                                  fname=f"avgprof_bystage_{comp}_{split}_z{_ztag(z)}.png")

### 6u — Build the extraction job for snapshots at fixed times after AGN activation

To populate the time-resolved 6t plot, the reduced particles must exist at `t_AGN + Δt`. For every
galaxy (with a measured `t_AGN`) this finds the snapshot nearest `t_AGN + Δt` for Δt in
`AGN_DT_GYR = [0.1, 0.2, 0.5, 1.0, 1.5, 2.0]` Gyr — **negative = before AGN** — (within `AGN_DT_TOL_GYR`), de-duplicates against
what is already on disk, writes a minimal plan `reduced_agnwindow_plan.hdf5`, and emits
`run_reduced_agnwindow.sh`. Launch it (the SLURM job's own dedup skips anything present); when it
finishes, re-run **6s** to cache the new profiles, then `avg_profiles_vs_tagn(...)` in 6t.

In [ ]:
# ── 6u·window — reduced-particle plan for snapshots at t_AGN + [0.1..2.0] Gyr (job to launch) ──
AGN_DT_GYR     = [-2.0, -1.5, -1.0, -0.5, -0.2, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0]  # rel. to t_AGN [Gyr] (neg=before)
AGN_DT_TOL_GYR = 0.3                               # skip if the nearest snapshot is farther than this
WINDOW_PLAN = os.path.join(SFHDIR, "reduced_agnwindow_plan.hdf5")
LAUNCH = os.path.join(os.getcwd(), "run_reduced_agnwindow.sh")

want = set()                                       # unique (snap, gx) to extract
for z in RESULTS:
    B = RESULTS[z]
    if B["BH"] is None:
        continue
    PIDX = _pidx(z); t_ = B["t_cosmic_yr"]; snaps = B["snaps_arr"]; recs = B["records"]
    tagn = np.asarray(B["t_agn"], float)
    for i, r in enumerate(recs):
        if not np.isfinite(tagn[i]):
            continue
        col = r["col"]
        for dt in AGN_DT_GYR:
            target = tagn[i] + dt * 1e9
            row = int(np.argmin(np.abs(t_ - target)))
            if abs(t_[row] - target) > AGN_DT_TOL_GYR * 1e9:
                continue
            g = PIDX[row, col]
            if np.isfinite(g):
                want.add((int(snaps[row]), int(g)))

missing = [(s, g) for (s, g) in sorted(want) if not os.path.exists(reduced_path(s, g))]
print(f"AGN-window targets: {len(want)} unique (snap,gx); {len(missing)} missing on disk")
if missing:
    uniq_snaps = sorted({s for s, g in missing})
    with h5py.File(WINDOW_PLAN, "w") as o:
        o.attrs["sim_name"] = SIM_NAME
        o.attrs["note"] = "reduced particles at t_AGN + [0.1,0.2,0.5,1.0,1.5,2.0] Gyr"
        o.create_dataset("entry_gx", data=np.asarray([g for s, g in missing], np.int64))
        o.create_dataset("entry_snap", data=np.asarray([s for s, g in missing], np.int32))
    n_arr = min(16, len(uniq_snaps)); rel = os.path.relpath(WINDOW_PLAN, os.getcwd())
    cmd = f"DUST_PLAN={rel} sbatch --array=0-{n_arr - 1} submit_reduced_particles.sh"
    with open(LAUNCH, "w") as sh:
        sh.write("#!/bin/bash\n# generated by quench_mode_vs_sigma_gas.ipynb (6u): reduced particles "
                 "at t_AGN + [0.1,0.2,0.5,1.0,1.5,2.0] Gyr\n" + cmd + "\n")
    print(f"  wrote {WINDOW_PLAN} ({len(missing)} files over {len(uniq_snaps)} snapshots)")
    print(f"  wrote {LAUNCH}\n  launch: bash {os.path.basename(LAUNCH)}\n    ( = {cmd} )")
    print("  when done: re-run 6s (build_profile_cache), then avg_profiles_vs_tagn(...) in 6t.")
else:
    print("  all AGN-window (snap,gx) already extracted — re-run 6s, then avg_profiles_vs_tagn(...).")

#### 6t·dt — time-resolved average profiles (run after the 6u extraction)

Once `run_reduced_agnwindow.sh` has produced the windowed files, **re-run 6s** (it rebuilds the
config-tagged cache with the new snapshots) and then this cell plots `avg_profiles_vs_tagn` for each
component and split — the median Σ(R) at each `DT_MYR` offset (now including times *before* AGN).
Until then the bins are sparse (the `n=` in the legend tells you).

In [ ]:
# ── 6t·dt driver — average Σ(R) at fixed times around AGN (needs 6u files + a re-run of 6s) ──
for z in RESULTS:
    if RESULTS[z]["BH"] is None:
        print(f"[z={z}] no BH history -> no t_AGN; skip."); continue
    for split in ("fastslow", "weakstrong"):
        for comp in PROF_COMPS:
            avg_profiles_vs_tagn(z, split=split, comp=comp,
                                 fname=f"avgprof_dt_{comp}_{split}_z{_ztag(z)}.png")

### 6v — Accretion & H2 content around AGN activation (catalog tracks, every snapshot)

The cheap, immediately-available view of the **pre/post-AGN** period: `event_stack` on `t_AGN` of
`BHAR` (accretion), `M_H2` and `H2/M*` (content), and the **dense** `Sigma_H2` within R20/R50/R80 (catalog, every snapshot) — these are catalog/BH-history
tracks sampled at *every* snapshot, so no particle extraction is needed and the window already
covers times before activation. `show_detect=True` flags where the log-median rests on few galaxies
(censoring). Strong/weak and fast/slow.

In [ ]:
# ── 6v — BHAR + H2 content around t_AGN (uses the existing dense tracks; before & after) ──
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history; skip."); continue
    print(f"\n=== z={z}: accretion + H2 content around t_AGN ===")
    props = [p for p in ["BHAR", "M_H2", "Sigma_H2_R20", "Sigma_H2_R50", "Sigma_H2_R80"] if p in PROP_REGISTRY]
    event_stack(props, triggers=["t_AGN"], split=_wb_split(SPLIT_XS), xwin=2.0, nb=12, show_detect=True,
                fname=f"agnclock_accr_h2_xstrength_z{_ztag(z)}.png")
    event_stack(props, triggers=["t_AGN"], xwin=2.0, nb=12, show_detect=True,
                fname=f"agnclock_accr_h2_fastslow_z{_ztag(z)}.png")

### 6w — Σ at R20/R50/R80 vs time since AGN (3-point version of 6t·dt)

The same view as 6t·dt — Σ(R) coloured by time offset from `t_AGN`, two panels for the split's two
classes — but the profile is reduced to **three characteristic points**: per galaxy, R20/R50/R80
are the radii enclosing 20/50/80% of the component's mass, and the value is the **local Σ at each**.
At every Δt the points are averaged across galaxies (median R, median Σ + 25–75% band). Cheap, and
it makes "less extended H2" read directly as smaller R50/R80. Sampling matches `avg_profiles_vs_tagn`
(nearest cached snapshot within `DT_TOL_MYR`); coverage before AGN fills in after the 6u extraction.

In [ ]:
# ── 6w — 3-point (R20/R50/R80) Σ profile vs time since AGN; same layout as 6t·dt ──
def _three_point(prof_lin):
    """Return (R[3], Σ_local[3]) at the R20/R50/R80 mass-enclosed radii of a cached Σ(R)."""
    area = np.pi * (PROF_EDGES[1:] ** 2 - PROF_EDGES[:-1] ** 2)
    M = np.nancumsum(np.nan_to_num(np.asarray(prof_lin, float) * area)); Mtot = float(M[-1])
    if Mtot <= 0:
        return None
    Rs = np.array([float(np.interp(f * Mtot, M, PROF_EDGES[1:])) for f in (0.2, 0.5, 0.8)])
    Sig = np.array([float(np.interp(R, PROF_RMID, np.asarray(prof_lin, float))) for R in Rs])  # local Σ at R
    return Rs, Sig

def avg_3pt_vs_tagn(z, split="weakstrong", comp="H2", dt_myr=DT_MYR, tol_myr=DT_TOL_MYR, fname=None):
    B = RESULTS.get(z)
    if B is None or not PROFILE_CACHE:
        print("  need RESULTS + PROFILE_CACHE (run 6s)."); return
    PIDX = _pidx(z); tagn = np.asarray(B["t_agn"], float)
    cmap = plt.cm.viridis(np.linspace(0.0, 0.9, len(dt_myr)))
    groups = _split_groups(B, split)
    fig, axs = plt.subplots(1, len(groups), figsize=(6.0 * len(groups), 4.8), sharey=True, squeeze=False)
    for ax, (lab, mask) in zip(axs[0], groups):
        idx = np.where(mask & np.isfinite(tagn))[0]
        gtimes = {i: _galaxy_reduced_times(B, i, PIDX) for i in idx}
        for di, dt in enumerate(dt_myr):
            Rstk, Sstk = [], []
            for i in idx:
                ts, sn, gx = gtimes[i]
                if len(ts) == 0:
                    continue
                j = int(np.argmin(np.abs(ts - (tagn[i] + dt * 1e6))))
                if abs(ts[j] - (tagn[i] + dt * 1e6)) > tol_myr * 1e6:
                    continue
                prof = PROFILE_CACHE[(int(sn[j]), int(gx[j]))].get(comp)
                if prof is None:
                    continue
                tp = _three_point(prof)
                if tp is not None:
                    Rstk.append(tp[0]); Sstk.append(tp[1])
            if len(Rstk) < 3:
                continue
            R = np.asarray(Rstk); S = np.asarray(Sstk)
            with np.errstate(divide="ignore", invalid="ignore"):
                Slog = np.log10(np.where(S > 0, S, np.nan))
            medR = np.nanmedian(R, axis=0); medS = np.nanmedian(Slog, axis=0)
            loS = np.nanpercentile(Slog, AVG_BAND[0], axis=0); hiS = np.nanpercentile(Slog, AVG_BAND[1], axis=0)
            ax.plot(medR, medS, "-o", color=cmap[di], ms=5,
                    label=f"{'+' if dt >= 0 else ''}{dt} Myr (n={len(Rstk)})")
            ax.fill_between(medR, loS, hiS, color=cmap[di], alpha=0.12, lw=0)
        ax.set_xscale("log"); ax.set_xlim(PROF_RMIN * 0.6, PROF_RVIEW)
        ax.set_xlabel("R20 / R50 / R80  [kpc]"); ax.set_title(lab)
        ax.legend(fontsize=6, title="since $t_{\\rm AGN}$", ncol=2)
    axs[0][0].set_ylabel(fr"$\log_{{10}}\,\Sigma_{{\rm {comp}}}\ [M_\odot\,{{\rm kpc}}^{{-2}}]$")
    fig.suptitle(f"z={z}: $\\Sigma_{{\\rm {comp}}}$ at R20/R50/R80 vs time since AGN — {split} "
                 f"(median, {AVG_BAND[0]}–{AVG_BAND[1]}% band)", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

for z in RESULTS:
    if RESULTS[z]["BH"] is None:
        print(f"[z={z}] no BH -> no t_AGN; skip."); continue
    for split in ("fastslow", "weakstrong"):
        for comp in PROF_COMPS:
            avg_3pt_vs_tagn(z, split=split, comp=comp, fname=f"clock3pt_{comp}_{split}_z{_ztag(z)}.png")

### 6x — Warm/cold gas density and temperature profiles at the critical points

Same by-stage average as 6t (median + 25–75% band, two class panels, critical point colour-coded),
but for the **temperature‑split gas**: cold (<10⁴ K), warm (10⁴–10⁵ K), hot (>10⁵ K) surface density,
and the **mass‑weighted T(R)**. These need the per‑particle temperature, so they appear only after the
reduced files are re‑extracted with the updated `build_reduced_particles_job.py` (it fills `temp` into
existing files) and the 6s cache is rebuilt (`build_profile_cache(rebuild=True)`).

**Works now (no temperature):** neutral gas Σ (Σ_HI+Σ_H2) and the ionized/hot residual (Σ_gas−Σ_HI−Σ_H2) are derived from the existing cache and always plotted. The cold/warm/hot and T(R) panels are a true *temperature* split and appear only after re-extraction adds `temp`.

In [ ]:
# ── 6x — cold/warm/hot Σ and T(R) by critical point (needs 'temp' in the reduced files + cache) ──
def avg_phase_by_stage(z, split="weakstrong", phase="cold", stages=STAGE_SEQ, fname=None):
    B = RESULTS.get(z)
    if B is None or not PROFILE_CACHE:
        print("  need RESULTS + PROFILE_CACHE (run 6s)."); return
    PIDX = _pidx(z); recs = B["records"]; snaps = B["snaps_arr"]
    cmap = plt.cm.viridis(np.linspace(0.0, 0.9, len(stages))); groups = _split_groups(B, split)
    fig, axs = plt.subplots(1, len(groups), figsize=(6.0 * len(groups), 4.8), sharey=True, squeeze=False)
    for ax, (lab, mask) in zip(axs[0], groups):
        series = []
        for si, st in enumerate(stages):
            stack = []
            for i in np.where(mask)[0]:
                row = recs[i].get(f"row_{st}", -1)
                if row is None or row < 0:
                    continue
                g = PIDX[row, recs[i]["col"]]
                if not np.isfinite(g):
                    continue
                p = PROFILE_CACHE.get((int(snaps[row]), int(g)), {}).get(f"phase_{phase}")
                if p is not None:
                    stack.append(p)
            series.append((st, cmap[si], stack))
        _draw(ax, series, phase); ax.set_title(lab); ax.legend(fontsize=7, title="critical point")
    axs[0][0].set_ylabel(fr"$\log_{{10}}\,\Sigma_{{\rm {phase}}}\ [M_\odot\,{{\rm kpc}}^{{-2}}]$")
    fig.suptitle(f"z={z}: mean {phase}-gas Σ(R) by critical point — {split} "
                 f"(median, {AVG_BAND[0]}–{AVG_BAND[1]}% band)", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

def avg_temp_by_stage(z, split="weakstrong", stages=STAGE_SEQ, fname=None):
    B = RESULTS.get(z)
    if B is None or not PROFILE_CACHE:
        print("  need RESULTS + PROFILE_CACHE (run 6s)."); return
    PIDX = _pidx(z); recs = B["records"]; snaps = B["snaps_arr"]
    cmap = plt.cm.inferno(np.linspace(0.0, 0.85, len(stages))); groups = _split_groups(B, split)
    fig, axs = plt.subplots(1, len(groups), figsize=(6.0 * len(groups), 4.8), sharey=True, squeeze=False)
    for ax, (lab, mask) in zip(axs[0], groups):
        for si, st in enumerate(stages):
            stack = []
            for i in np.where(mask)[0]:
                row = recs[i].get(f"row_{st}", -1)
                if row is None or row < 0:
                    continue
                g = PIDX[row, recs[i]["col"]]
                if not np.isfinite(g):
                    continue
                p = PROFILE_CACHE.get((int(snaps[row]), int(g)), {}).get("Tprof")
                if p is not None:
                    stack.append(p)
            A = _stack2d(stack)
            if A.shape[0] < 3:
                continue
            with np.errstate(divide="ignore", invalid="ignore"):
                L = np.log10(np.where(A > 0, A, np.nan))
                med = np.nanmedian(L, axis=0)
                lo = np.nanpercentile(L, AVG_BAND[0], axis=0); hi = np.nanpercentile(L, AVG_BAND[1], axis=0)
            ax.plot(PROF_RMID, med, "-", color=cmap[si], label=f"{st} (n={A.shape[0]})")
            ax.fill_between(PROF_RMID, lo, hi, color=cmap[si], alpha=0.12, lw=0)
        ax.axhline(np.log10(T_COLD), ls=":", c="b", lw=0.7); ax.axhline(np.log10(T_HOT), ls=":", c="r", lw=0.7)
        ax.set_xscale("log"); ax.set_xlim(PROF_RMIN * 0.6, PROF_RVIEW)
        ax.set_xlabel("R [kpc]"); ax.set_title(lab); ax.legend(fontsize=7, title="critical point")
    axs[0][0].set_ylabel(r"$\log_{10}\,T_{\rm mw}$ [K]")
    fig.suptitle(f"z={z}: mass-weighted gas temperature T(R) by critical point — {split} "
                 f"(median, {AVG_BAND[0]}–{AVG_BAND[1]}% band; dotted = cold/hot cuts)", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

_DERIVED = {
    "neutral": lambda d: (d["HI"] + d["H2"]) if ("HI" in d and "H2" in d) else None,
    "ionized": lambda d: np.clip(d["gas"] - d["HI"] - d["H2"], 0.0, None) if all(k in d for k in ("gas", "HI", "H2")) else None,
}

def avg_derived_by_stage(z, split="weakstrong", what="neutral", stages=STAGE_SEQ, fname=None):
    """No-temperature phase proxy from the cached components: neutral = Σ_HI+Σ_H2,
    ionized/hot = Σ_gas-Σ_HI-Σ_H2 (incl. He/metals). Works with the existing cache."""
    B = RESULTS.get(z)
    if B is None or not PROFILE_CACHE:
        print("  need RESULTS + PROFILE_CACHE (run 6s)."); return
    PIDX = _pidx(z); recs = B["records"]; snaps = B["snaps_arr"]
    cmap = plt.cm.viridis(np.linspace(0.0, 0.9, len(stages))); groups = _split_groups(B, split)
    fig, axs = plt.subplots(1, len(groups), figsize=(6.0 * len(groups), 4.8), sharey=True, squeeze=False)
    for ax, (lab, mask) in zip(axs[0], groups):
        series = []
        for si, st in enumerate(stages):
            stack = []
            for i in np.where(mask)[0]:
                row = recs[i].get(f"row_{st}", -1)
                if row is None or row < 0:
                    continue
                g = PIDX[row, recs[i]["col"]]
                if not np.isfinite(g):
                    continue
                d = PROFILE_CACHE.get((int(snaps[row]), int(g)))
                if not d:
                    continue
                v = _DERIVED[what](d)
                if v is not None:
                    stack.append(v)
            series.append((st, cmap[si], stack))
        _draw(ax, series, what); ax.set_title(lab); ax.legend(fontsize=7, title="critical point")
    axs[0][0].set_ylabel(fr"$\log_{{10}}\,\Sigma_{{\rm {what}}}\ [M_\odot\,{{\rm kpc}}^{{-2}}]$")
    fig.suptitle(f"z={z}: mean {what}-gas Σ(R) by critical point — {split} "
                 f"(median, {AVG_BAND[0]}–{AVG_BAND[1]}% band)", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

_have_temp = any("Tprof" in v for v in PROFILE_CACHE.values())
for z in RESULTS:
    if RESULTS[z]["BH"] is None:
        print(f"[z={z}] no BH -> no critical points; skip."); continue
    for split in ("weakstrong", "fastslow"):
        # works now (no temperature): neutral (HI+H2) vs ionized/hot residual (gas-HI-H2)
        for what in ("neutral", "ionized"):
            avg_derived_by_stage(z, split=split, what=what, fname=f"stage_{what}gas_{split}_z{_ztag(z)}.png")
        # true temperature phases — only once the files carry 'temp' and the cache was rebuilt
        if _have_temp:
            for ph in ("cold", "warm", "hot"):
                avg_phase_by_stage(z, split=split, phase=ph, fname=f"stage_{ph}gas_{split}_z{_ztag(z)}.png")
            avg_temp_by_stage(z, split=split, fname=f"stage_Tprofile_{split}_z{_ztag(z)}.png")
if not _have_temp:
    print("[6x] temperature phases (cold/warm/hot, T(R)) need 'temp' in the reduced files: "
          "re-extract with the updated job, then build_profile_cache(rebuild=True).")

# Part 7 — Compaction along the quenching sequence

Two complementary views of how the $R_{20}/R_{80}$ concentration evolves as galaxies quench:

- **7·1** — compaction & morphology vs **distance from the main sequence**
  $\Delta\mathrm{MS}=\log_{10}(\mathrm{sSFR}\cdot t_H(z))$ ($\Delta$MS$=0$ is the MS,
  $\approx-0.7$ the quench threshold; galaxies drift right→left). **Binned by H₂ gas fraction** in
  **4 bins (<1%, 1–5%, 5–10%, 10–20%)**, three subplots (**weak · intermediate · strong** coupling),
  **median + 16–84% band** per track. Quantities: stellar & gas/H₂/HI $R_{20}/R_{80}$ (stellar from
  the catalog, gas/H₂/HI from the reduced Σ profiles), **gas diskyness** $\kappa_{\rm rot}^{\rm gas}$,
  **stellar $B/T$** and **dust fraction** $M_{\rm dust}/M_\star$. Sampled at **every snapshot**
  (catalog quantities dense; profile $R_{20}/R_{80}$ only where a reduced file exists); zero-H₂
  galaxies fall in the <1% bin. κ_rot/$B/T$ need the `rotation.*` history keys (BUILD_MULTI_Z).
- **7·2 (event-aligned)** — $R_{20}/R_{80}$ of gas and stars on the `t_AGN` / `t_SFT` / `gas_min`
  clocks, faceted by **mass bin × strong/weak**. (Unchanged.)

### 7·0 — Catalog radii vs particle-profile radii (validation)

Before trusting compactness, check that the radii from OUR particle extractions (the `RP` layer, from
the 100 kpc reduced Σ profiles) agree with the CAESAR catalog radii, at the critical-point stages
where both exist. The **R₂₀/R₈₀ concentration is dimensionless**, so it compares directly (expect
~1:1); the **half-mass R₅₀** is shown raw — catalog radii are ~comoving kpc/h while ours are physical
kpc, so the annotated `med(prof/cat)` there is the unit/definition factor, not a disagreement. Needs
`PROFILE_CACHE` (reduced files + `REBUILD_PROFILE_CACHE=True`).

In [ ]:
# 7·0 — validation: CAESAR catalog radii vs the radii from OUR particle-profile extractions (RP).
def _radii_compare_pool():
    C = {k: [] for k in ("catCg","proCg","catCs","proCs","catR50g","proR50g","catR50s","proR50s")}
    for z in RESULTS:
        B = RESULTS[z]; rp = B["RP"]
        if not rp["OK"]:
            continue
        P = B["P"]; recs = B["records"]; SL = rp["STAGES"]
        for i, r in enumerate(recs):
            col = r["col"]
            for si, st in enumerate(SL):
                row = r.get(f"row_{st}", -1)
                if row < 0:
                    continue
                def cat(key):
                    a = P.get(key); return float(a[row, col]) if a is not None else np.nan
                with np.errstate(all="ignore"):
                    g20, g80 = cat("radii.gas_r20"), cat("radii.gas_r80")
                    s20, s80 = cat("radii.stellar_r20"), cat("radii.stellar_r80")
                    C["catCg"].append(g20 / g80 if g80 > 0 else np.nan)
                    C["catCs"].append(s20 / s80 if s80 > 0 else np.nan)
                    C["catR50g"].append(cat("radii.gas_half_mass"))
                    C["catR50s"].append(cat("radii.stellar_half_mass"))
                C["proCg"].append(rp["C2080"]["gas"][i, si]);  C["proCs"].append(rp["C2080"]["star"][i, si])
                C["proR50g"].append(rp["R50_gas"][i, si]);     C["proR50s"].append(rp["R50_star"][i, si])
    return {k: np.asarray(v, float) for k, v in C.items()}

_RC = _radii_compare_pool()
if not len(_RC["proCg"]) or not np.isfinite(_RC["proCg"]).any():
    print("[7·0] no RP profile radii yet — build reduced files + PROFILE_CACHE (REBUILD_PROFILE_CACHE=True).")
else:
    def _panel(ax, x, y, lab, unit=False):
        m = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[m], y[m], s=10, c="C0", alpha=0.4, edgecolor="none")
        if m.sum() >= 3:
            both = np.concatenate([x[m], y[m]]); lh = [np.nanpercentile(both, 1), np.nanpercentile(both, 99)]
            ax.plot(lh, lh, "k--", lw=1, label="1:1")
            with np.errstate(all="ignore"):
                ratio = np.nanmedian(y[m] / np.where(x[m] > 0, x[m], np.nan))
            rr, pp, nn = _spr(x, y)
            ax.set_title(f"{lab}\nmed(prof/cat)={ratio:.2f}, ρ={rr:+.2f} (n={nn})", fontsize=9)
        ax.set_xlabel("CAESAR catalog"); ax.set_ylabel("profile (my extraction)")
        if unit:
            ax.set_xscale("log"); ax.set_yscale("log")
        ax.legend(fontsize=7)
    fig, axs = plt.subplots(1, 4, figsize=(17, 4.2))
    _panel(axs[0], _RC["catCg"], _RC["proCg"], r"gas $R_{20}/R_{80}$ (dimensionless)")
    _panel(axs[1], _RC["catCs"], _RC["proCs"], r"stellar $R_{20}/R_{80}$ (dimensionless)")
    _panel(axs[2], _RC["catR50g"], _RC["proR50g"], r"gas $R_{50}$ [raw; units differ]", unit=True)
    _panel(axs[3], _RC["catR50s"], _RC["proR50s"], r"stellar $R_{50}$ [raw; units differ]", unit=True)
    fig.suptitle("7·0 — CAESAR catalog radii vs particle-profile radii (at critical points; pooled)", y=1.03)
    plt.tight_layout(); plt.savefig(figpath("compaction_radii_catalog_vs_profile.png"), dpi=130, bbox_inches="tight"); plt.show()
    with np.errstate(all="ignore"):
        print(f"[7·0] R20/R80 median ratio prof/cat: gas "
              f"{np.nanmedian(_RC['proCg']/np.where(_RC['catCg']>0,_RC['catCg'],np.nan)):.2f}, "
              f"stellar {np.nanmedian(_RC['proCs']/np.where(_RC['catCs']>0,_RC['catCs'],np.nan)):.2f}  (1.0 = agree)")

In [ ]:
# 7·1 — compaction & morphology vs distance from the main sequence (ΔMS), binned by H2 gas fraction.
# Sampled at EVERY snapshot of each galaxy's history (dense catalog tracks); profile R20/R80 (gas/H2/
# HI) is added where a reduced Σ profile exists. ΔMS=log10(sSFR·t_H); ΔMS=0 is the MS, ΔMS≈
# log10(PASSIVE_FACTOR)≈-0.7 the quench threshold. Zero-H2 galaxies fall in the <1% bin (see full tracks).
DMS_EDGES   = np.linspace(-3.0, 0.7, 14)      # distance-from-MS bins [dex]
H2FRAC_BINS = [(0.0, 1.0, "<1%", "C0"), (1.0, 5.0, "1-5%", "C2"),
               (5.0, 10.0, "5-10%", "C1"), (10.0, 20.0, "10-20%", "C3")]   # M_H2/M* [%]
BAND_PCT    = (16, 84)                         # shaded percentile band per track
MIN_PER_BIN = 4                                # min points for a median in a ΔMS cell

# quantities tracked vs ΔMS: (key, label, logy).  kappa_gas / BoverT_star need the rotation.* history
# keys (BUILD_MULTI_Z rebuild); gas/H2/HI R20/R80 need PROFILE_CACHE; stellar R20/R80 & dustfrac are
# catalog-only (available now). Each auto-skips when its source is absent.
DMS_QUANTS = [
    ("R2080_star",  r"stellar $R_{20}/R_{80}$ (catalog)",           False),
    ("R2080_star_prof", r"stellar $R_{20}/R_{80}$ (profile, my radii)", False),
    ("R2080_gas",   r"gas $R_{20}/R_{80}$ (profile)",               False),
    ("R2080_H2",    r"H$_2$ $R_{20}/R_{80}$ (profile)",            False),
    ("R2080_HI",    r"HI $R_{20}/R_{80}$ (profile)",               False),
    ("kappa_gas",   r"gas diskyness $\kappa_{\rm rot}^{\rm gas}$",  False),
    ("BoverT_star", r"stellar $B/T$",                              False),
    ("dustfrac",    r"$M_{\rm dust}/M_\star$",                     True),
    ("h2frac",      r"H$_2$ gas fraction $M_{\rm H_2}/M_\star$",     True),   # check: binning axis, plotted vs ΔMS
    ("dtm",         r"dust-to-metal $M_{\rm dust}/M_Z$ (De Vis+19)", False),  # M_dust / (f_z·M_H2 + M_dust)
    ("dtg",         r"dust-to-gas $M_{\rm dust}/M_{\rm H_2}$", False),
]

def _prof_r2080(prof_lin):
    """R20/R80 (radii enclosing 20%/80% of the component mass) from a cached Σ(R). NaN if empty."""
    if prof_lin is None:
        return np.nan
    area = np.pi * (PROF_EDGES[1:] ** 2 - PROF_EDGES[:-1] ** 2)
    M = np.nancumsum(np.nan_to_num(np.asarray(prof_lin, float) * area)); Mtot = float(M[-1])
    if Mtot <= 0:
        return np.nan
    r20 = float(np.interp(0.2 * Mtot, M, PROF_EDGES[1:])); r80 = float(np.interp(0.8 * Mtot, M, PROF_EDGES[1:]))
    return r20 / r80 if r80 > 0 else np.nan

def _build_dms_tracks():
    keys = [q[0] for q in DMS_QUANTS]
    C = {k: [] for k in ("dms", "h2pct", "cls", "logMstar", "age", "oh", *keys)}
    used = []
    for z in RESULTS:
        B = RESULTS[z]
        if B["BH"] is None:
            continue
        P = B["P"]; snaps = B["snaps_arr"]; recs = B["records"]; wb = B["well_behaved"]; CO = B["CO"]
        PIDX = _pidx(z) if PROFILE_CACHE else None
        zsnap = np.clip(np.asarray(B["redshift"], float), 0.0, None); tH = COSMO.age(zsnap).value * 1e9
        _DMS_CANON = {"R2080_star": "R20s/R80s", "kappa_gas": "kappa_gas",
                      "BoverT_star": "BoverT_star", "dustfrac": "dust/M*", "h2frac": "H2/M*",
                      "dtm": "dtm", "age": "Age_s", "oh": "oh"}
        _CAN = {lk: CATALOG_PROPS[cn][0](P) for lk, cn in _DMS_CANON.items()}
        def _av(lk, row, col):
            a = _CAN[lk]
            return float(a[row, col]) if a is not None else np.nan
        hit = False
        for cls in ("weak", "strong", "inter"):
            for i in np.where(np.asarray(CO[cls], bool) & wb)[0]:
                col = recs[i]["col"]
                for row in range(len(snaps)):
                    ms = P["masses.stellar"][row, col]; sfr = P["sfr"][row, col]
                    if not (ms > 0 and sfr > 0):
                        continue
                    with np.errstate(all="ignore"):
                        dms = np.log10(sfr / ms) + np.log10(tH[row])
                        h2pct = 100.0 * P["masses.H2"][row, col] / ms
                        vals = {
                            "R2080_star":  _av("R2080_star", row, col),
                            "kappa_gas":   _av("kappa_gas", row, col),
                            "BoverT_star": _av("BoverT_star", row, col),
                            "dustfrac":    _av("dustfrac", row, col),
                            "h2frac":      _av("h2frac", row, col),
                            "dtm":         _av("dtm", row, col),
                            "dtg": np.log10(float(P["masses.dust"][row, col]) / float(P["masses.H2"][row, col])),
                            "R2080_gas": np.nan, "R2080_H2": np.nan, "R2080_HI": np.nan,
                            "R2080_star_prof": np.nan}
                        _logm = np.log10(ms)
                        _age = _av("age", row, col)
                        _oh = _av("oh", row, col)
                    if PIDX is not None:
                        g = PIDX[row, col]
                        if np.isfinite(g):
                            prof = PROFILE_CACHE.get((int(snaps[row]), int(g)))
                            if prof is not None:
                                vals["R2080_gas"] = _prof_r2080(prof.get("gas"))
                                vals["R2080_H2"]  = _prof_r2080(prof.get("H2"))
                                vals["R2080_HI"]  = _prof_r2080(prof.get("HI"))
                                vals["R2080_star_prof"] = _prof_r2080(prof.get("star"))
                    C["dms"].append(dms); C["h2pct"].append(h2pct); C["cls"].append(cls)
                    C["logMstar"].append(_logm); C["age"].append(_age); C["oh"].append(_oh)
                    for k in keys:
                        C[k].append(vals[k])
                    hit = True
        if hit:
            used.append(z)
    D = {k: (np.asarray(v, dtype="U8") if k == "cls" else np.asarray(v, float)) for k, v in C.items()}
    return D, used

_DMS, _DMS_Z = _build_dms_tracks()
print(f"7·1 ΔMS tracks: {len(_DMS['dms'])} (galaxy,snapshot) points from anchors {_DMS_Z}"
      + ("" if _DMS_Z else "  -> none (need BH history; profile tracks also need PROFILE_CACHE)."))

def _med_band(x, y, edges, min_n=MIN_PER_BIN, pcts=BAND_PCT):
    cen = 0.5 * (edges[:-1] + edges[1:])
    med = np.full(len(cen), np.nan); lo = np.full(len(cen), np.nan); hi = np.full(len(cen), np.nan)
    for b in range(len(cen)):
        v = y[(x >= edges[b]) & (x < edges[b + 1]) & np.isfinite(y)]
        if len(v) >= min_n:
            med[b] = np.median(v); lo[b] = np.percentile(v, pcts[0]); hi[b] = np.percentile(v, pcts[1])
    return cen, med, lo, hi

def _missing_frac(x, y, edges):
    """Per-ΔMS-bin fraction with y==0 or non-finite (over ALL galaxies in the bin, not just the
    finite ones). A vanishing median track then reads as 'all went to zero/missing' (strip high)
    vs 'no galaxies here' (strip has a gap). Same idea as the 7·1·weakdust dust=0 strip."""
    cen = 0.5 * (edges[:-1] + edges[1:]); fz = np.full(len(cen), np.nan)
    for b in range(len(cen)):
        sel = (x >= edges[b]) & (x < edges[b + 1])
        if sel.sum() > 0:
            yy = y[sel]; fz[b] = float(np.mean((yy == 0.0) | ~np.isfinite(yy)))
    return cen, fz

_CLS_ORDER = [("weak", "weak"), ("inter", "intermediate"), ("strong", "strong")]

def compaction_vs_dms(key, label, logy, fname=None):
    if not len(_DMS["dms"]) or not np.isfinite(_DMS[key]).any():
        _need = "/PROFILE_CACHE" if key in ("R2080_gas", "R2080_H2", "R2080_HI", "R2080_star_prof") else \
                ("/rotation.* history (BUILD_MULTI_Z)" if key in ("kappa_gas", "BoverT_star") else "")
        print(f"  no ΔMS track for {key} (need BH{_need})."); return
    dms = _DMS["dms"]; h2 = _DMS["h2pct"]; yq = _DMS[key]; cl = _DMS["cls"]
    ncol = len(_CLS_ORDER)
    fig, axs = plt.subplots(2, ncol, figsize=(5.4 * ncol, 5.8), gridspec_kw={"height_ratios": [4, 1]},
                            sharex="col", sharey="row", squeeze=False)
    for j, (ckey, clab) in enumerate(_CLS_ORDER):
        axm, axf = axs[0][j], axs[1][j]
        sub = cl == ckey
        for lo_, hi_, blab, bc in H2FRAC_BINS:
            base = sub & (h2 >= lo_) & (h2 < hi_)                  # all galaxies in this H2 bin (for the strip)
            mb = base & np.isfinite(yq)                            # finite-only (for the median track)
            if mb.sum() < MIN_PER_BIN:
                continue
            cen, med, lo, hi = _med_band(dms[mb], yq[mb], DMS_EDGES)
            axm.plot(cen, med, "-", color=bc, lw=1.8, label=f"{blab} (n={int(mb.sum())})")
            axm.fill_between(cen, lo, hi, color=bc, alpha=0.15, lw=0)
            cz, fz = _missing_frac(dms[base], yq[base], DMS_EDGES)  # frac =0/NaN per ΔMS bin
            axf.plot(cz, fz, "-", color=bc, lw=1.3)
        for a in (axm, axf):
            a.axvline(0.0, ls="--", c="k", lw=1); a.axvline(np.log10(PASSIVE_FACTOR), ls=":", c="0.4", lw=1)
        if logy:
            axm.set_yscale("log")
        axm.set_title(f"{clab} coupling  (n={int(sub.sum())})")
        axf.set_ylim(-0.03, 1.03)
        axf.set_xlabel(r"$\Delta$MS $=\log_{10}(\mathrm{sSFR}\cdot t_H)$")
    axs[0][0].set_ylabel(label); axs[1][0].set_ylabel("frac\n=0/NaN", fontsize=8)
    axs[0][-1].legend(title=r"H$_2$ gas frac", fontsize=8)
    fig.suptitle(f"{label} vs distance from the main sequence — by H2 gas fraction "
                 f"(weak | intermediate | strong; {BAND_PCT[0]}-{BAND_PCT[1]}% band; strip = frac =0/NaN; pooled)", y=1.02)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

for _k, _lab, _logy in DMS_QUANTS:
    compaction_vs_dms(_k, _lab, _logy, fname=f"compaction_vs_dms_{_k.lower()}.png")

### 7·1·weakdust — what drives the weak-coupling dust-fraction scatter?

The weak-coupling $M_{\rm dust}/M_\star$ track (7·1) has a very large scatter. This reuses the **exact
7·1 compaction plot** — dust fraction vs ΔMS, coloured by the four **H₂ gas-fraction** bins with the
16–84% band — but restricted to **weak coupling**, with the **subplots being explicit ranges** of a
secondary variable: one figure split by **stellar mass**, then the same by **stellar age**, gas-phase
**oxygen abundance** (12+log O/H) and **stellar B/T**. Edit the range edges in `WEAKDUST_VARS`
(edges → subplots ≤e0, e0–e1, …, >e_last). A short strip under each subplot shows the **fraction of galaxies with dust=0** per ΔMS bin, so a vanishing track reads as *all-went-to-zero* (strip high) vs *no galaxies* (strip gap). Mass / age / O/H work today; B/T needs the `rotation.*`
history (BUILD_MULTI_Z).

In [ ]:
# 7·1·weakdust — the EXACT 7·1 compaction plot (M_dust/M* vs ΔMS by H2 gas fraction, 16-84 band),
# restricted to WEAK coupling, subplots = explicit ranges of a secondary variable. A short strip below
# each subplot shows the FRACTION of galaxies with dust=0 per ΔMS bin (per gas-fraction bin, same
# colours) — so a vanishing track reads as "all went to zero" (strip high) vs "no galaxies" (strip gap).
# One figure per variable: stellar mass, then age, gas-phase oxygen abundance, stellar B/T.
WEAKDUST_VARS = [
    ("logMstar",    r"$\log_{10} M_\star$", "%.1f", [10.3, 10.7, 11.0]),
    ("age",         r"age [Gyr]",           "%.1f", [3.0, 5.0, 7.0]),
    ("oh",          r"12+log(O/H)",         "%.2f", [8.4, 8.7, 8.8, 8.9]),
    ("BoverT_star", r"$B/T$",               "%.2f", [0.2, 0.4, 0.6]),
    ("dtm", r"$DTM$",               "%.3f", [0.001, 0.01, 0.1, 0.3]),
    ("dtg", r"$DTG$",               "%.2f", [-3, -2, -1.5]),
]

def _zero_frac(x, y, edges):
    """Per-ΔMS-bin fraction with y==0 (NaN where the bin has no galaxies -> a gap, not a zero)."""
    cen = 0.5 * (edges[:-1] + edges[1:]); fz = np.full(len(cen), np.nan)
    for b in range(len(cen)):
        sel = (x >= edges[b]) & (x < edges[b + 1]) & np.isfinite(y)
        if sel.sum() > 0:
            fz[b] = float(np.mean(y[sel] == 0.0))
    return cen, fz

def weakdust_by(varkey, varlabel, fmt, edges, cls="weak", fname=None):
    if not len(_DMS["dms"]) or not np.isfinite(_DMS["dustfrac"]).any():
        print("  no dustfrac ΔMS track (need BH)."); return
    w = _DMS["cls"] == cls; sv = _DMS[varkey]
    if not np.isfinite(sv[w]).any():
        print(f"  {varkey} not available for {cls} (rebuild history?)."); return
    dms = _DMS["dms"]; h2 = _DMS["h2pct"]; dust = _DMS["dustfrac"]
    e = list(edges); bounds = [(-np.inf, e[0])] + list(zip(e[:-1], e[1:])) + [(e[-1], np.inf)]
    def _rlab(lo, hi):
        return ("≤" + fmt % hi) if lo == -np.inf else ((">" + fmt % lo) if hi == np.inf else (fmt + "–" + fmt) % (lo, hi))
    fig, axs = plt.subplots(2, len(bounds), figsize=(5.0 * len(bounds), 5.9),
                            gridspec_kw={"height_ratios": [4, 1]}, sharex="col", sharey="row", squeeze=False)
    for j, (lo, hi) in enumerate(bounds):
        axm, axf = axs[0][j], axs[1][j]
        rmask = w & (sv > lo) & (sv <= hi)
        for glo, ghi, gblab, gc in H2FRAC_BINS:
            mb = rmask & (h2 >= glo) & (h2 < ghi) & np.isfinite(dust)
            if mb.sum() < MIN_PER_BIN:
                continue
            cen, med, blo, bhi = _med_band(dms[mb], dust[mb], DMS_EDGES)
            axm.plot(cen, med, "-", color=gc, lw=1.8, label=f"{gblab} (n={int(mb.sum())})")
            axm.fill_between(cen, blo, bhi, color=gc, alpha=0.15, lw=0)
            cz, fz = _zero_frac(dms[mb], dust[mb], DMS_EDGES)
            axf.plot(cz, fz, "-", color=gc, lw=1.3)
        axm.set_yscale("log")
        for a in (axm, axf):
            a.axvline(0.0, ls="--", c="k", lw=1); a.axvline(np.log10(PASSIVE_FACTOR), ls=":", c="0.4", lw=1)
        axm.set_title(f"{varlabel}: {_rlab(lo, hi)}  (n={int(rmask.sum())})")
        axf.set_ylim(-0.03, 1.03); axf.set_xlabel(r"$\Delta$MS $=\log_{10}(\mathrm{sSFR}\cdot t_H)$")
    axs[0][0].set_ylabel(fr"$M_{{\rm dust}}/M_\star$  ({cls})")
    axs[1][0].set_ylabel("frac\ndust=0", fontsize=8)
    axs[0][-1].legend(title=r"H$_2$ gas frac", fontsize=8)
    fig.suptitle(f"7·1·weakdust — {cls}-coupling $M_{{\\rm dust}}/M_\\star$ vs ΔMS by H2 gas fraction, per "
                 f"{varlabel} range  ({BAND_PCT[0]}–{BAND_PCT[1]}% band; strip = fraction with dust=0)", y=1.01)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

for _key, _lab, _fmt, _edg in WEAKDUST_VARS:
    weakdust_by(_key, _lab, _fmt, _edg, fname=f"compaction_vs_dms_weakdust_{_key.lower()}.png")

In [ ]:
for _key, _lab, _fmt, _edg in WEAKDUST_VARS:
    weakdust_by(_key, _lab, _fmt, _edg,  cls="strong", fname=f"compaction_vs_dms_strongdust_{_key.lower()}.png")

### 7·2 — Gas & stellar $R_{20}/R_{80}$ compaction, by mass bin and coupling

Event-aligned median tracks (25–75% band) of the gas and stellar $R_{20}/R_{80}$ concentration on the
`t_AGN`, `t_SFT` and `gas_min` clocks. Rows are stellar-mass bins; the two lines are **strong vs weak**
coupling. One figure for gas, one for stars, per anchor. Needs BH (coupling).

Also tracks the **gas diskyness** ($\kappa_{\rm rot}^{\rm gas}$) and the **bulge-to-total ratio**
($(B/T)_\star$, $(B/T)_{\rm gas}$) around the same clocks (from the CAESAR `rotation.*` dicts, as in
Resolving_feedbacks §8i). These need the `rotation.*` keys tracked in the history — add them to
`PROPS` (done) and rebuild with `BUILD_MULTI_Z=True`; panels auto-skip on anchors where CAESAR did
not expose them.

$R_{20}/R_{80}$ is shown two ways: **catalog** radii (gas & stellar only) as a dense full-history
clock (labelled "catalog, dense clock" in the title), and **our profile** radii (incl. **H₂**/HI,
from the reduced Σ files) which exist only at the critical points (labelled "profile"). See **7·0**
for the catalog-vs-profile validation.

In [ ]:
# 7·2 — R20/R80 (gas, stars) on agn/sft/gas_min clocks, faceted by mass bin x strong/weak.
def compaction_by_mass(prop, comp_label, fname=None, triggers=("t_AGN", "t_SFT", "gas_min"), xwin=2.0, nb=10):
    if prop not in PROP_REGISTRY:
        print(f"  {prop} not registered at this anchor; skipped."); return
    spec = PROP_REGISTRY[prop]; ylab = ("med log " if spec["logy"] else "med ") + spec["label"]
    fig, axs = plt.subplots(NBINS, len(triggers), figsize=(4.8 * len(triggers), 3.3 * NBINS),
                            squeeze=False, sharex="col", sharey="row")
    for ri in range(NBINS):
        mb = (mbin == ri)
        split = [(CO["strong"] & well_behaved & mb, "C3", "strong"),
                 (CO["weak"] & well_behaved & mb, "C0", "weak"),
                 (CO["inter"] & well_behaved & mb, "C2", "intermediate"),
                 (CO["no_fb"] & well_behaved & mb, "0.5", "no AGN")]
        for ci, tn in enumerate(triggers):
            ax = axs[ri][ci]
            _estack(ax, TRIGGERS[tn], spec["fn"], ylab, logy=spec["logy"], xwin=xwin, nb=nb, split=split)
            if ri == 0:
                ax.set_title(tn)
            if ri == NBINS - 1:
                ax.set_xlabel(f"t − {tn}  [Gyr]")
            if ci == 0:
                ax.set_ylabel(f"logM* {MASS_BIN_LABELS[ri]}\n{ylab}")
    fig.suptitle(f"z={z}: {comp_label} around critical points — strong vs weak, by mass", y=1.005)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling figure skipped (set BUILD_BH=True)."); continue
    print(f"\n=== z = {z} : R20/R80 compaction by mass & coupling ===")
    # dense CATALOG radii (gas & stellar only) — full-history clock; noted in the title.
    compaction_by_mass("R20g/R80g", r"gas $R_{20}/R_{80}$ (catalog, dense clock)", fname=f"compaction_R2080_gas_z{_ztag(z)}.png")
    compaction_by_mass("R20s/R80s", r"stellar $R_{20}/R_{80}$ (catalog, dense clock)", fname=f"compaction_R2080_star_z{_ztag(z)}.png")
    # OUR profile radii (incl. H2/HI) — sparse (only at critical points), from the reduced Σ files.
    compaction_by_mass("C2080_H2_prof",   r"H$_2$ $R_{20}/R_{80}$ (profile)",    fname=f"compaction_R2080_H2prof_z{_ztag(z)}.png")
    compaction_by_mass("C2080_gas_prof",  r"gas $R_{20}/R_{80}$ (profile)",      fname=f"compaction_R2080_gasprof_z{_ztag(z)}.png")
    compaction_by_mass("C2080_star_prof", r"stellar $R_{20}/R_{80}$ (profile)",  fname=f"compaction_R2080_starprof_z{_ztag(z)}.png")
    compaction_by_mass("kappa_gas",   r"gas diskyness $\kappa_{\rm rot}$", fname=f"compaction_kappa_gas_z{_ztag(z)}.png")
    compaction_by_mass("BoverT_star", r"stellar $B/T$",  fname=f"compaction_BoverT_star_z{_ztag(z)}.png")
    compaction_by_mass("BoverT_gas",  r"gas $B/T$",      fname=f"compaction_BoverT_gas_z{_ztag(z)}.png")
    compaction_by_mass("dtm",  r"DTM",      fname=f"compaction_DTM_z{_ztag(z)}.png")

# Part 8 — Interpretation

*(Fill after the cluster run.)* Read against the strong/weak-coupling scenario:

- **6b vs 6c** — does **coupling** (SPLIT_XS) separate the dust / H2 / compactness tracks around the
  critical points more cleanly than **fast/slow** does? If so, the quenching *mode* (not its
  *timescale*) is the organizing variable.
- **Strong coupling** (X-ray, radiatively efficient): earlier compaction (`R20/R80` rises), a dust
  rise after `t_QT` (6b/6d), and — long after QT — a *steeper* dust decline (6f, ref0) → less
  residual dust at late times.
- **Weak coupling** (jet/maintenance): slower compaction, no post-QT sSFR rise, and more persistent
  residual dust (shallower 6f decline).
- **6e** — are the dustiest-at-gas_min galaxies preferentially strong-coupled (Fisher), and do their
  event stacks match the strong-coupling signature?
- **6g** — the falsifiable Σ-vs-τ_q test, now with Σ measured at QT / post-quench rather than gas_min.
- **6h** — does AGN ignition lead quenching (`t_QT−t_AGN`>0), more so for strong coupling?

Caveats to check first: per-anchor N and coupling tercile edges (6a), QC over-flagging (Part 4b),
and the `DUST_THR` dusty/non-dusty cut.

## Mechanism: H₂ geometry, dust, and feedback (6e·geom + 6h·gasmin + 6e·sat)

The mode-not-time story, made explicit at `gas_min`:

- **6e·geom** — strong AGN–ISM coupling carves a **central H₂ hole** (`hole_H2`↑), pushes
  H₂ outward (`Rpeak_H2`/`R80_H2`↑) and **removes the central dust** (`Sig_in_dust`↓,
  `dust_core_frac`↓, dust less concentrated). The hole/extent co-vary with the dust budget
  `M_dust/M*` — H₂ *geometry*, not just amount, tracks the dust, and the controlling variable
  is the coupling strength `xstr_post`. Weak coupling keeps a compact central H₂ and dust
  survives in the centre.
- **6h·gasmin** — the galaxies that stay **dusty at `gas_min`** (surviving central reservoir)
  show a **rising** post-`gas_min` SFR; non-dusty ones stay flat. The slope tracks dustiness
  (and the weak-coupling / compact geometry).
- **6e·sat** — the **number of satellites** at `gas_min` (other galaxies in the same halo or
  within 100 kpc) does not track the central's sSFR / post-`gas_min` SF rise, which instead
  follows the internal H₂-hole geometry (6e·geom) → the late SF is feedback-set, not
  environmentally fed.
